# So sánh DOCX OCR trước / sau tối ưu

Notebook benchmark riêng, có thể xóa sau khi đo. Không sửa notebook đọc đề chính.

1. Giữ notebook **Private**. Bật **GPU T4** (hoặc GPU CUDA phù hợp) và **Internet**.
2. Bấm **Run All**, chờ model sẵn sàng rồi đến ô chọn đề ở cuối.
3. Chọn Word/ZIP và bấm **Chạy cả hai và so sánh**. Cache kết quả luôn tắt.

**Mốc trước tối ưu:** Git `1d70099e4070ee0995c3d15cc47312152d4559d2`.
Hai bản dùng cùng một model DeepSeek-OCR-2 đã warm-up, revision cố định, ảnh 1024,
dtype phù hợp GPU. Bản cũ giữ parser và xử lý ảnh cũ; riêng loader được thay bằng
loader chung để chạy đúng trên T4. Đây không phải khôi phục nguyên môi trường cũ.
Bản mới gom nhiều đề, khử ảnh trùng, xử lý ảnh và thử GPU batch có kiểm tra.

Đo thời gian từ bóc Word đến xuất ZIP, không gồm upload/tải model. Kết quả khác nhau
hoặc thiếu công thức được báo để kiểm tra; không tự xem LaTeX thay đổi là sai.
Tỉ lệ tốc độ có khác biệt đầu ra chỉ mang tính tham khảo. Hai bản không chạy đồng thời.


In [ ]:
import os, sys, json, time, socket, subprocess
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HOME"] = "/tmp/docx-ocr-huggingface"
os.environ["DOCX_EXTRACT_NATIVE_MATH"] = "0"
ROOT = Path("/kaggle/working/docx-ocr-benchmark")
MODEL_REVISION = 'aaa02f3811945a91062062994c5c4a3f4c0af2b0'
OCR_GPU_BATCH_SIZE = 4
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Bật GPU trong Session options rồi chạy lại.")
print("GPU:", torch.cuda.get_device_name(0), "Torch:", torch.__version__)


In [ ]:
packages = ["fastapi", "uvicorn", "python-multipart", "Pillow>=10.1", "requests", "lxml",
            "transformers==4.46.3", "tokenizers==0.20.3", "accelerate", "einops", "addict", "easydict",
            "ipywidgets>=8,<9", "olefile>=0.47"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "libreoffice-core", "libreoffice-impress"], check=True)
print("Dependencies ready. Torch được giữ nguyên.")


In [ ]:
import base64, hashlib, io, zipfile
PAYLOAD = 'P)h>@6aWAK2mk;8aa}bkA{lZ5008m>001Tc003fPb7gF4Ze=fHZ){;=UvFb_Uvp)0c4cxeZ)0*_Y;R|2V=i!cU6M;{+%OQw@BS2nPByH)<QixpgiR>qu_W6EgkZe3Y^iHYMjCIjlt7_}(qm{ZA(SSRB{V%WkY2hzI^=t}pQOn9Y%U&;q?!5uW`+>5BNHA1vZ9{c2(SmN`YjU3SzV5s)AAw&lqzKHIM5j6+<~b2BN7N%(R?MyQ2CmwAyr;J@+_-YU55-5uWm&G59f9H0mAAYkeR<iaCDJs1XR_nfPOt0BMj>D9<u6s1fsf<U=Nz0dh&gGI8$Y^Y<FL_tsUN1bU%)HNFDh5{>%RoAY^s<33!%Eg%DQPA+YKyGpR?Q>hd#(gjGLs7}n(_hpj;4M)yTKlMzilix6U2{=vp!uXEsc-t-RP8N4I8NJtCF+k9qdWBXZdOu}PhPG~<TXO{n}v)S8Ui9N}hYnsL6%(ASA#z0Tem$4o5l!_pu>!49N;668Y-2wn0gzVIl@yLwAOeU+#Z$|uuNWeCLzW~|eU0r?^No(GqsGf`qgFhK(4q#pW1T;-Hbvy%RGAoLO60j)lDV?VrZQ_v@tT{*S+A;7Lh(~}=L(MCosir`P<E8S1D%+Wn2udUEhf)lw(p(Cgp5hXxFJ-#mdA+;U?(A-vx!wqvh9KxbB%lI~RDs>y+janz;kpcAz&JddKi&l}G+WUO2=+G*o}$oGoU1aDR|hat*)Mn+`C2wB<A+&fbCDP5=K-=tZ3|?9e#F&Md&)H)X;jB611UcW1}obKj0RCkwO-(?H64RFYeU!F+kPtfsBxOSGgW3qTz8&t>Itc+MyY0(vl=c+<2Nh0Oyjz;z=$idKxx4ea{BPt`Ug-;0|XQR000O80041ankUxc(+vOs1RnqZCjbBdVqtS-Y-w&~FJo_PVPaozV{%_}WpZ|9axZUVa$j?0a&~2ME^v9xSxb-O$dSJDR|HrGlW2)0Raf^83ZusMR8?2ErylKc_1FgkL@Jpg(;_oD$xM|rgBgrr>|wE|0tPUQf#GR4jDcUU4X+O_VX+4_=6}$?#E48%k1EZ2VeesekSH=TA~NF3c+vO$XQa|kU%r5g!%L``nlJ@4t8ZV=Kr1OyIFT~V863)-3?aDMhm)7D!pQT^v-<XT0_xj0A_KbkPmzMo>iMrIoDrQ}%}NFrr^gU{@9f#`@!7KwbUptKB$7_MY^oKZx&u^-2~!#>B2svHe&PkEe5lyPXv8UljFBW~RRw?j?DIx`quD7JX4-*qX2sIQpWx)^#RK>r7m2*C!j9Vv5vg}duGsfsSl|9pq;SMo`CZ1w-HXG^?p_4%XZ8Gx65ie{Zjp@?RxP0Q{3~KLjT=p)g6K?A6?xv_;x)}SE#y=~C8w-_%j2g<XU9>Iz}uU8{$s1%+nf6KtC_v=70|_N3Y6t}<oUku@uHMU1DEZUtlF!}3`->`tCn{(E6Nei+46s;7)iKDJ#VCB0V7grQgUb>H@@w_GoG^!Jioenc|4`8)Le=Vyee}^5`46=I8v-CrKnidvibIRFU~KUn;mGc+@hB+PFu1U1xZ<>h@>jcWy)#$aA<g*#FezJjcJi|fK7EQM;#E1B~|ROP;n_dkCY`0TI~bPb<PI<I&=UXqAkXju=PXF+qvUo2(%)i8c9_!Rl&6)Wyus2GGTed#fYgO3|kzPS$+G4!;om20Un9Je)hSy)B3tXSKBX<@p}Fv1B!iUSfxR$#jgd>#a~1UeLNnJ)boD>l<bi5%Myn5{HGugZ@W~HDvP`wct2bGi8LrdguE#bsBa|b`t~)4qMrX0JzUR!#^LP^ujfB5klp;>EvEXA*SB8@sLTv<W6vUrcG^Z>oR%ieB^|?Hg_7u0(s2-aUVL_ObbK10KRr8!0r&}HWrao~e0S4;z870#!2r&sU|y`{m<j$7Q|%pZ2hN#+A!h;#$w4cFMtc4%etvOw42h8x5`tF3t8vD`OgNQFK&6S&5KIz0M=&AM+i9cCk>V2u34uZyp~DVrxUphD3C&n@{wnDliYy?J(0u|q5#A0|#K1U{91<kU2?u3_=0zLL#u?r(Gr~a=ZV9x3=O#_LNS_hP&`aNE1st+Ghy7tAPe{#%QjQ@(6}8*CAhXExyo8M)PMLOA3&H~r0KoVCQ}WTQTXB)*OiKZH@*1r%a5%SW#NX~XtLH!G76h^Zf%7!YB+&Zy4KzU^!tjCY(_9Wo4o+qVRz9*%`KWmUys|EHI@w1}6p@N%%bWK6ZrHmadup^ib0YBaYAI#8bORk5G!b%*OU1MzT(BfqLm+zBR616ykebC*CTwt}3=5ZhcOb#TJ{XR2Np$})Jb+wY$FwwY!HU>aBy~#;K#&r2hWTU(j^U{P*db`O$sHIX9JMkcg_IL?U<sr0;hsg&<<m1beEF&YZSZOTVYk13@?HmBo{||{T%2{sJkLQZCf1m?>qd&PuI5ZNNc*(EhpWdKp)ipgK0rd3NLPzbje*$#B&d>gNo;ow!5U8Ooe;WRKr?|e8Bl{K1zjx2Y>i`Z)Ew-K!^^#AQXLXgk^J;*^E?d(`_ca6=wS#n;~+93HpG8u$QsKy=7Q@u?l@KOEDnhT%=gDiNp%25uxZJtW(kbh>{_Y>h)PYtgoba~Mc-QOx@dr6adi(QOEL?tHN<Q@*p8k}8dI^jA|s}msHCd4P^@ivV6USVZJTu4fw!Vrp>`QLi-=7~9)uA!2?-p5V%jLNc49gWz2|xVO3E^lB4!gNbl}@KX-YqI(WO)jqlcr8rHGs7L5PuWb-;b!zp@GK0c?g82NZ-cM&qXDvrG^dU;m+=f8hePa3x-w=WG>YK=Z|~MdS|sd42oeX1xv@Hp|80%UNf+5|i}$YN+bl|H6Xsk1$^Rj%JX_`u4Tnh@3E4d}(D2>-m4+Ig_C3+h1zP7r!tNB=TC=Ao^fEmVWxM`!L#n4<kSURXx86-9}Sh&wo!KU))llIcGv!)*7|_IYtR*gts?~H>lg61yqYKMe1<chS_!P+E7oGEaU5pYgUznvY-@cJe+A(9f1AmKuHozy$8UBh5_vLteYH7%l;6vh=d6y5<Hn$N*jVF4=V^f{5mtV11UGq$Fzkc4Py|QY`n2vJWmbL(A3_``r3vopalBhTRA=yghIkaX7lhnz23Cnf7lNloLvO9NWdhDacQYc^TqHN?_ubm8D|CbVS<#3XjBa(YV9Hjis-|51{r4t-W@#ZO&|5jsU-kP#DrAM_FcrwWf;K+HXPc!9~uI=6i`Xa4g={@v2ljRyHox7IiWsG2xKJB4dsGp28yh&k|MRt5ERzn5G2m>P+`^Lc6oprzy#|KNF)(3Ap$Z?AQ{@ZQI|Klz^=w@-3M0!a-b3l!$C6v{@{rpMsTQPX+<oRd9dzMG)if?Ib>_hYc^Y1T1-+)Uu?pZ7by%}F&@d1iGa)KJ%rnfixoV>Flw5wz`s0s&v!v-2ZlpAwdutA(B`Xxn8yBrNUg8;9`!I0(u^A{hM55(H1Cp}6RdDRWKHf)6`8>RHCjnKZDd(V14|N>k_#Q=q!=ay$^+<O(Ne;@RuaoA278Ylg;B~hu4*DEA2}ZPs+uOWF87N2LO*D9oZCtFiFK$*&5Q|1CAFCy!<05QpXHJ}Sg;XBMOk)0@ifyN7)q@rrY#h{q`@dpra`ac{&ibdI#snSR|NRy&U4EeDYyvGSl2w!StD@;i^guP{FXRl8;ac_#{gBbr1gT5We|MRrb4$w8q{SCH5;6wicDA_jqU)}4@3|Am**#qQybGuqtRItk{h;+l??)7E#p$8K|(ZH21z#62T*CXEjBhYf;N^pmv*1nc;^0KT-Z7|2v!|*6Vk9HNTl5hZu(xx4X}m38`Z;QrG3qHwrs+qD~5$JQL`hi7}ZkEf)Fr<73EruE#i{s3<e;p2(O|aN-hFjlpPrPXhM^d=wmk3pEO2X3Ua5{t?z(;J@hU6(TX{KM$xt6npuOc7_4?4DI+#3se8TnN{BAX#!~HvW{i$qjRTaF+9(!Fqf4Wko7hlEqX&<B+lnA*^f&K+x&2b4TQ}dX?trhh?_QB>gcA&46k(Tpr@XqGpBwTPlgfx#QgttFTO3wetx)KO0C)0>>-^ma4B@yj1`BL8q49v_vu=A1ZFR8%)LQx1R*M^D#y<+WBK;?<-md=pm|NwOO*YHv!yRX_%H%fCtqou{C^l_!?@_5_QR;#JF30rt<Ojlr?2K177MYxYmSAUok$-=+T@<4g2g@zqBie!Qyn1>LSC<#(CkODIYbN%N7<LtE?^xL1Xe8A&QArR&fRhvEZcp~2a^`v{EU30H7HP%_>;Wr=%q4m%UGas*f`~sfOwC|0fd2kI1UAh}Zd2F4JbovHOcJ_{>uTNJqsN=;`n|n<+by~Zg-YArWj3cqFeKO&@sHTx@qTX$=Kv+bqyFA*fA5`Mh`nBg+O`sHdC=eA-$l9IC*YSvks{_r>Nf&$`m=>`N(9os5<_W3aw;jwZSxqxVa9Q@ShJz-kQ40AK`RcKEFo`aK3Mi_VXRpP5(88gw%v5Jwo^_eZMIM0@qTZ&-PNiUZ9glkpa$QBqW^)w#vDZW;2FtrVv3r*|9tT|7T`Q*cH;B)X7P&|<clw%zZo{(-hBO9bneU~?byerse95e9mnoqRgJ*EgXw1OX_2~0*rm~xt@W{2b?Re$kKlu*6xw9CqOvj}Xd8(TeYZs^z?O_=WCl}o%EjU3HpFpTgk$Jsq+rSx$3q!{$Pm`=KfgFXX?0FHR^tKIel}b;Zr-MB#UM9QxY-8HjO2jGh~Op8RE9*~*)$e|IhncP<(O&I={`_4*W9<dGpY?U0}Q+y1RH6}St_cu;@HOX5%7wON)v%8%RcP5^2ip$zTXZBQl<lcH)AB%Su-DSf7;1oBj95kB4|_j2jI)G-vQrE53qG+iGQ#*Arw;Edis-AMky;4lCYskF?af*N2-~ifYO$C0}QJNYwr|rfX{OlL{Sv3#-K+?f3SS<(-w72IX5B8J}bjOgP{uVC$b%aSpZ!eUg8@lWO-yeA_imZJaMfZ>)k0eLp=G4xMpnGz&7y&8LrwuY*B5mfcV(WCErCjicE%3kT#0i{!cceJuDb?hB4>vyZ{R_7y!8@T!YQsYXu4)Sxeh0c-6TOUa`&BG=X!Rn``a&df2FF!t;U8ic-%k-zJH(DW+EUR>is_!Y@Gz22z3FP`P4hOEGS%a0|Y>FJuDyz(NlR*MZf(3<~h=854R?inId-OE`(`D9yLwa@XNz6H$_?4J=_H_P;>U|4*}OHCqY%Ke91?a-tdb$!nErs}^nrmOH{KjZNuFi5)nt=Re_iT6Wv>)_nq#`t~>2@?b{U>-At#&%eNTA1FrE^Piyqnm3nrP==$n)r+_FnLNe$OZN32z;BY38}s2z?G5!Iz@#NI7%pB*99`A(-#4XlTF-B2h(o#=OcuXw{PU>SgECvZb}!&o;kH5A`b)^*W8W%_;S7Tr!{WiW9b~|!H2h>Ex_-k-`};}g{S8n{0|XQR000O80041aSuhXtcmMzZmjD0&B>(^bVqtS-Y-w&~FLQEZFJy0Hcwc3BbaG*1bT40DX>MtBUtcb8c~#C$3c@fDhT;93;&WClv>rgvwJR?m5@I@mz~qN?iaow`CGH<y-q>r#*{`eQw+d2jwHtzpFK($}pYZ9)#MRB(W-F0%Ue!5V@#G)l3H%57MMR54WS`0~V!q)~i(Ci*T2I9WzKd!_JPwY=EW33d?8GB|0Z>Z=1QY-O00;m80C8Q8$h7zX0RRBW0RR9c0001DVRL0{X>MgNb8=%ZWN%}5UuAf7a$#e1FJE76VQFq(UoLQYb&$Po!!QhlcRdB6Nd_Es58$Cw*S>+k*s?3s`T<2D1`@w}p~x+gp_{1t?m(XNDG~$Hl8~|$Ger*?bGH7nT5wGsSBt?+aol>il4EOP;Z!&aG1(bBx}dtV006)Q!(W_!r=@rs__I!=62)bZHOJVE2}hRQ;nFxw1`k|4nE$j>ls(APDHfJJOm@by@*wMYIB3ari=-c0RBKjE@<`t$^)T`V!_!@=GP=GR_j09ra`lgN9Sb)wkqfM5`SIo8_ABjg&7pZ?xI?Xz^BHcd4Qk{~*L1d_+u;sZq~OBxA|GGfE;}LI48{cX))*KDkjB)$jghVJR=)TFP)h>@6aWAK2mk;8aa{$}DL`Zc007hl001KZ003fPb7gF4Ze=fXa$_%KZ)130Wq5RQVPkYJVQzDGWpXZXd397lYurW{z3W#zW_ys<wnE$6f(v$QY}0ttCJq#Kw~KZ(D+x<8Vl>*=Jq2<Kp_GLlYI<=J+>)fh4fZ7x!3UZAkLfQdGb=j|#wSTLpWgT8z4uL$<w7b*<~ZY>wfo<1+{Y>;^Qp-4BxSB7KfA-C>$!uqmG4Fa-y3=M`wQYZPROQ6im}L5BKSB9<Q|i*V*9FxLP-xR-(Vs{+H(LvGF3`$7u>d$p9PA>_74aNLOjoLI#~NfsK6j3IVK_vu@~$5w*+Z0VJQk-&q13PC0z2Yd3@O;%UHaA<r5FH=BO=gK7b1Dbl{S}Bfd{Uk9Y{1_n>i8@dy|5kN@iWC539va52}_+YIib-o%5Ff=37i-N%2Ae@B1o<4J!LGhMymh!S0&SY}QKG!`KJxH-+R?HeLLZ0YK+KDK?|ehxNwN0wi#>t8v3@`;bFAc15N+ec{n|C!)o-n=+~mBZ;^_g?e3k9A%D0j}%A1YyY|6fDXZS8(^@=8Q+6v96yYmu#Bsn^XUTiLTzY8?Sf-73=yPh0@je;yg|V(<EicboHjR%vG$b<AYYNv~12e(&p%@)x})bhZL!<-sW(r2vSy1hL%92>%$x>ZeC|j=Q>mC&cd@<b0k2U_dLQ>$ShEp*#p~-p4GNRvxje`BzIaFiSLYme7Nyocl%y{e0Obq<L-@k4vVhwepoCG!=+)-9r(S`u;`kAjvqW+-`?HWx&6;Uuo8}jUR!?|=sb`?B!fI27b=iyY{}jlY=u>Pitx>>f6o6h^PVaHoZ!_~%S19BGI?XJmNCtg<xnWOpwPjNEdp&$t?f@R(e<;|p)2FO)``7X=<20Oh}6}IO$rkzf)ZR;=Vdz`Ei#Rbr6vi_c#Ke2&v^uQ7J{t%1WO<wEv%hYL#s^>S@%IaADd=gS~MP&2hD4KlfuQkY|iW;oZ3_@((T(@!1Yk*>H~PG>$x)VJywRvaS7Tyr!lCm{^o556(_nnS5VT<7QG(KJv$0!->70^XViK{@0fNcms>^-&$oCPR&ncXWh;^?xKPLzPF!=}6xy|M%7SniR7kcAC4(fzDl$KkqRb0-fl?Z)(}Xju8hZy{kFE#FbxGG~Z6G|;@l!cV3)ZeLEx?`s0Z>Z=1QY-O00;m80C8Qz!y^5`1pojZ3;+Nn0001DVRL0{X>MgNb8=%ZWN%}5UuAf7a$#e1FJW_YWo=??WiD`erB=_2+%^=x`>%L-E;h`1X?iIGQwX6=Lg_Y?+?saK$QnyoTXJQ2f3zty<WNdl(m)`D5VD&k327SIbPEOhkb{@}AL?IHo@CF?W}8yzTt=3j-uu3%?>$KrrP2njR!RoKk5x5{Gz%756_7wbA*+5OS0rV6n8vSufhuX(ju}?Y`Okh)*RmQ`G?OXQvyy0%X;Sju#E^N;-ZXNPK@g;D0mnOL=s7B5SbHBTqbC6XfM%xF62dUKs)US8;?O7!VF(MQ!GMrZCc-cs$tw9W3b(^CgeHvRI0%9f9Q|b5>tMJ5!(abq+Xpg(!nP0L=(cU1)-botOFwnbK*gvLO0IE6QXw-)ZL=>k2rpf*&65qdm_U*w=zoKEhea@gi}(#(+2&6QM%(r!1tV<ps)X=d_o#+<CT>dtmo-MePhs#ayS?4<5ZAW1VIgt`g>9bKK)b_W1hm_yaCBQC2nY*iNJ<O|LE847fVpk&L9~@ZxVE;oarqL$I>!~vF}(3%W7y=~A2NfT+BT03aADi;WH17wmCT?dlI3vrD`g1$_3a(lxbXVdjq^S(b^8pr?K8S=n*+%~+U8}^Z#{wS=yrDqTef+QL!@~?gHi9-bT1@_t!__w#F+M11(Z3VjH?_~XH>T*f!g*lg{VIdih*S|fedXV9@DR8I5XQm3`S58!_98VX77%{AWXz@0HG1x&vgiAd{!w}IN!(&&YU21g3g&G7t{vI#z(?KiW9huJoW_`+w3dg-5ydBj`yS|p4s+CajFW|R~3&~f)RXd+aE5j<j%G?PNc>h-H=(2c?^=j{;ki)_h@zt((ZSefhMrYZF@^VLG)%_I-<t5_XTWn5sV<S&7UO@Nx9O!WYXHRZ(b}@iOG@>d4aDxS+ApT<0Kfl>ggWsLf-B5LUrvxnzcEpSPGogT2w}$S`Hn=r-=In<%skqWWwR$j%bO01#F24X8u(jjNqdS-JOl|UicCf;t#{?!KGxb(p|^%{*p@Mkzn4gySPI^ZF1Bkx6Pp(2cw}V`vO=ojK^Te5PQ%ol)QUZLiea}33`gcu+0H%VU#|@+O~IPhLsYG9GwF(6xIsll~g-DdIeww0QGe=@AlQWKlt3XcZI9V8`25$CKyh3VXONk82MU8O=66QDp?s#gftNLwB|&)EI$CMNyT9u^1Ay&4zN^h%Gx;cGUG?jNEj!<?Ci$53!i*C`}6~tf@TR-MJaL?>F|qQ2iKeqt~niCnTK)Ts%g%MoTZBHL?tnNf?YcHZ8}vHrtaQwOR<}zW(TIjl@V!{SJPC{1P5avRmF^-@I$|c<oPTYl2tH;&p!84?%Zc0MQ<BQGB=COpK_6>cmbdnl!^y~Ke6zui6I$Gr?7H2lwl8g#dalzs9X>rGKJ$*B?U2*M{nL%E=HVGMvF3v-|%}1e9mOFB<T4LKd%y&x+6K+Lwt}irt{>UrK^OLC6j41I5{?B`M)H>%=0%>aT<o}T>l5c2(~A9uU^uBefz|BurF&8yfyL1fT-C?wuzQPW=sDkqK8t!sm?X@iG`|V>aI0b2xG8AEb2UWEhSvKImS7tZb`|lu)JD9bjs$lWBe1Z*1lIvA-p2{N6X4__s<z`<nr0q*0vUg&3wI9kzaQO=8Pso6^*f^XQ|M>`0%N8oHJD=1=(b&&{fo%je+fm%FNW}^s=s*2T_TX!4%E0Rz1e0eTb`+=rBIz6IcGx;kGawRLNxY7N4F^Vs=azkHM*rNvg0LPs7?Q*4_(aAQdc*Q@u!BO8Qj9%bH)vG2k-@%iak@3uBnX7hq9ED{_7dsLGL8NQYy7D(q@#4D+7}O0i&-N$}zQpE;s&lGa5TeI5EH7)~6i!5#YP3!q;aKED~mVtlIG05DnAn$1W>g_wRw@`{Z?$dpMlJrf6i15ir?1QY-O00;m80C8QlxO@r(1^@sw5dZ)r0001DVRL0{X>MgNb8=%ZWN%}5UuAf7a$#e1FJo_RbY*ySE^v9pSIukOMijsMuXvFIUXiujv^f;PhJfR=p>|x7Knlhzv(iYKW;COWX6-fGg+dOcr(zl?C5Jkg5YkIp($nft=%)QY)W4*>(dXK0<D?DUgLd`i&HMblk7aQpm4-y7Y?KfkEd@%(;1f+ePSTX7zzU9<)P;ai?iY-F6!VIMYDH;5b}E`kPY`0FBTh6OEuc$JEiEnGs5LE_C#*}M=}5av004^e)CHe;+PtEPx=rxpP-v=LU_wI=J}|_6#k8iPV1r)aK*y&68FJkZh^M7G=;5J_?63?0<5ZBC!s>Mx9B7&vIWZ9|ps99&v-E61cxV@ji4shN>4h(t(7c76lKE)jCb>_)fX?>T)+Stm?e)7Rp86o6g;?pSejr?sgD<I(d`Aju1gsOz$$-<H9ph#fQmuB)6TUKByX80h{q+;jZ1!^uUL--&R4*CXJ`JER=ty@`%7ZQ>iUw@tq6iypOSLLehaY5d${fL!9$c+Y6xBl|Ah5=TfJuMcW}a9rZ7pR)M`x%i1=h%R(u6YvQU~AcovXVJPqku6$8s#&;c}m<jtx?QHDIEpPR21W)lp94)0!{|y~4F35Ee#1=2j7W`~-@G&?`#B(GvBFu!G{dK!qoLCc>UI)WPb<ty@N93ErK+8cXI+oKeiJT?B_BVa;i91HGoa9JIQxW#7CRO^sX@lowcPa@)AlXf2WJkR+kP@08kIK#Z+kF94^beEM9&AfG;CT?kp8y@G@jk4BRFRJA;zJYA&4Sg|9Sj}8@WVeY&txXOw-31f@-mW$Evf`wSOT&N=71uK<9<rx|ioC&&$5y1na$8rmO6Z<p*cP_>5W#jJU-8SqLO(ox>B^DI(?Zzd<R0RU;QBFv82U*%dH%w*B5oqRJNDWLBH_LW5=VoaZ-szg>DTEUpb=&uMkh0qt&SDZ)R{M(VGZ9_{#4HY5860qyK(D&*1k8ZKSz8;6)OpldAuC;2L8cXZ(z>RMSfrbX7+y<Z8fZMAzGd4O7$cmeCKe_*h?jY13omgt)Mi%sI2mTc^bXKgw|y8Z=1<^ZH2Y12i0J7tgExnN{gP+LUdydB5xNkR7|A(A#w<;lFhpheQR{it5(upLvI7*2v0t~ciM7DVs%r==EsV&2WIJkHSTrmS7gi0<o@yGEAe78EO}ogaTTIw4;E=C-`F!#$%fhLaPme_eTSQ0q4-yL7n;X!1uyt!~W9ya;DG^~g2@%}${l%$8ZRXhz4D#$J>|4(@Jj}D#Fu)8y^+3#yB;GDD(uL1Hojtkp33!U6QT>)uU33q}vYObN92vF6xJ4+=y`IE(x$`)tKI>jx%hT#$9t2gB4@3v!W&+j%&B&vPh7I$@qyUm4?X7zI`rRhD9OLA|n2x-5K{rW1HBR2H!Jjb_E@oU=pjzeA<09vl-J00ELfU@6Sb3^`S1vP+-etI#P!Ilian1SA1ux38Zw;(A9u0|!VyF7$+`PAS{~D~-N6tq!#Ci6#X%H=`o@BshN8lB9MtSzqtIc;{-z`@$@6VSa1aJ1*@H{payd>bwo_W9}4pua$CHP^4N1XwdLTIwhK}54Df>qCs7;u(n-wfe_RDOLhV0rd<2yYJa?0d&?Y_rUXD9>JrW`|fV%!D_Gv)2{UlgM<yIWP8RM^*eQ$05w8uS9$H*9&M%d}q;3Tq%c;=Gh^5dG><9AkTh8OQ*l%ra2U$-PEq+6cY-)I<cn*k;Fl?y}7Y%&gZ95qGV_WQYpyHB<ph}!`nA4Gl?n<=MN(mnA1`mu8b8ZtZJSYXBPnY9Ea&6R*jB9o_z<8hQx)!#@@IPdr8$Zs?1Ouv-9I%QR&Wmvv_|s1E9IQ42lX&Y}s!?<kMec(?|DQ)4|Nxy`fy@H*u=z=VfdUGgh-M+8^q}lS>4vHi$;k|Gq|GD#}>Bw~eAs(7#(N=m`G|o>(W&Sv=@Szw5FwzXup^77Vd7qj0K9E*_R5<@MU%%q0UR$$WcqM!}r%kKi9rO9KQH000080001SU3Icda<u{g0D=Vo03!eZ0AgWtWo&6~WiNAbV=rWHV|ZU>cyw}MV{|WUZ((F*axQRrbyUxb(?Ary=dXCAmjr5x*Gj~LD=hxHcv57UW+s`@$xNJ??zUYP6c3&kyeuNP6cJfr3wmk}9xDAm#=k@+KiX#3&0*TS?|U!beDBS$ypYPkj1?*4#AEeqAvK#6$WSj;G6#$dN;uM*XlNLlP|e_zoVO4$ijX=SwK1VR&%3uxs#dr!lW|Uj8F~N!BZ;TOIy*Yp<HR8qpa|DPz$`Jad3vDsG2FTf>6mjTA1b*HayreGgkG2D5=T=*oGMIG;1c1$0YlJ6g%0j&J7R{mCPmL#@qlqsC4_)9%xaQ*uuKk|7ctBtMGy}BBdPEJH`Ss^&i!!1?!jYQo*9=l0cy*60o;}!K+yF<Wbm4<eu)f{)laa?xuAomkM;(NaAcSinz91K>Wi#!Lv8tfr|wQ_LGALJfJ(=K^RlBRauF~}imbBd@7s@U@}JZ7!j|)7r);8-rjRjPULr`SEiXg{hpQi8D6`AoW4H|kQl_mivdc>Yqp>Z|l4w)LsOi?Qw!!~%?FO|w29ila3Ih+xbWbU%ZY=vm3bMiK0f88fJ9IH@zb>{Pa8D`~2w(zckw7`&Oq-y~U@3@58>EbOlhVk~31(>Ehv4$B!z3ROjG4&vU}r}sUOn?t8i<?I;LegZiis>}<}2Ks7>Fxwlrn(>p`$l}f8_VUcbFfBO*Pl!7>-Y>H<oU#VH$)ql|Nha+=dn5Zp+sMw3*=*uNihq2#+#i0)HRxY^mxwQW-J3lxJ1M@6viuhrz5D>+@;`ZF$Pz(3YPV7`k3SZp*JpJ<V_j5<0fa?*gb?ojcaMW@<l+46t2Z2shN-Zn}#g-d=H$kSce7f^F!PWTNhp-*2RcU7^l~7mDGR5LXo&o6T5>&D9aO9>i|xSNzy5{bu~)`X}`M0Z>Z=1QY-O00;m80C8Q{p+}^v0002T0000e0001DVRL0{X>MgNb8=%ZWN%}5UuAf7a$#e1FKuCTXkTG-b7gdME^v7@%`pnXFcb#hy{Gu0qfiu_Tm?7JAR#`RLYu_owdkgJ0>J}x69sW_GdSfW@g#!c{=t`{5v&%%dtzVME#A8mY%L@yQ!mM30Cw>v6}|*Mg`+OP&gc?TqnS2bWI5($+13Od`z^{Ug*!N2HCT~>w<OWU9x^__Dun~2O5q})sJ7(5s!h+c2DkBrE`>e3Xp3y+OoKxH5>$;fj{hf+Lz}gjU^X6(vJX&80|XQR000O80041a%sNw>mID9)T?+sJCIA2cVqtS-Y-w&~FLQEZFJy0Hcwc3BbaG*1bT4gTbZBg0bY*xhaCwzhO^e({5WVlODB2f^QIz{&?Irl;goWe~h;7kol|B-wTW+<M8H~X<FXRvm2|fmL$-x-=X4t{_up$3L`%6N%dS|4OdhF;TrmO3{s;4*Aok>wc2xZ*}serd<XDMb7E3c85m74$*&HeQm0Dwf*Ne;XXpH8ASg%xF$O5|}x6>dDNu=e4rb7+;E<P#G9EK2UhT%_nRA2K0<LCL|SSTwXS4VB1M00uqQbQ4TM$V2)A{s^q!{Y`-h4)^Qze)Bnamnqb1-qiurYxLpv#qS{d-D{o$@2@{~aM@qK<9SF3F@Po&A{cnXtmLU@Iwmz4SK*IaI_};4^r6fwT-gE_3ceE@0jm~)5Zk8FU@n!SIYdT}P|@5hmD#Ww>5pTIA(GfHV^TKiF6F3bqKGYbRI;ZwDpa^pAS4iCiq#NHr8?-<nBKw;CGTNfso6rRB#$yl*)syG!{^s{0TABR#D^7-=ZoPKxeYrR+wZ$(f%F&&2b?F1G_8jCEKTy|G_ivw`4XDthP?kzwTHNSH1%f{@SU!t>A0>GbC)ku%*W15Xp0anX*C9m2csYV<x<Rzhc%tc!?s`xawu_X!uF|DKk4T0$W}le8#PRUhwm~^TiUgt&_tgE^8k?RnL|7s^$+N2Bc0k^$gy_1twxSJpSG7kq&Li963o9TCP?~0$8O~5!tOGI#LaAbgb0!_M>U4Kq!sZ4-S!L+PXi8mp$pI-0hXMjR&jsq9M9^x|M$=2WHM^ds+tHw{P}2=<WYs0GG4$OR<?wd4#+YuOayuAvInPfJ9fxDP&aQ`0Zno97ynxIyFYlohF@+zLQ(d+mmT!GKS3v5$a7Xm@aq2h1A~8F^}Anr0k>Cg4kfZe#x-pqf@kFR>J?cV>vo3s_UgCccOykQEW8;OO0LNBJx05&wVpb<U*m444wW65RquCxVFr=jFhgRC-eae&GE~bPR%y5O0nivCwh4n!CavApM>|EQXV8BCL7Gng(N0g<w$t)2PS&wPZI?`&(VVO3Zjx)J=_oQe&2r8#%{B8h$Hq^MTx)94DVJX<W6Q^RL5GRxIFf7~_aLDHJn=16MF8e?K_v|U12Fft6nPkiJ`}?ETGO-Q-}e+Y{q8MPHy`!YDeboX?xoT<z*9G~(G^qFbrmZy4%o|=+|Cd1!$r-6aKo@s@W8xE!Dk8DjA#D=P)h>@6aWAK2mk;8aa{^=&~EMn002b?001Na003fPb7gF4Ze=fXa$_%KZ)130Wq5RQVPkYJZDDk1Z)0*UaCwbXU2EJ%6n)pPxXhClv~p<PmaM^DuN^nGcUjBSrIck>nq5s-no(v(%Whpm`xHtkB@ief6zrNn3k^wWpj3LugG&C#^p}*;$8KC4TD?eU=AL`bnR^Bybc9vttr7x*!#>y_AMI`RkM>&7-iAZ$ku!LhVi@#BKv<A)$`r5}FD@zAJs#o7)+l8|9&EX;+j1={;Qq2rmP%k>!8|Npvl%Fz{|0`jv#S-vJRngC$8`P@gB=%FD_H65P3koJ_lKt~`1`9TrY8ReW;%aCL1))2wEPM%)cG&I!F@&{*7;MlA$Q;Nh7fRIv$wBw{v(B;__N^<a;d;x1i-i=6VB&QN&vD<xFnVl%4UMkxBQ5w!8F2x1P+8;XMgxm=>$o{f{2Jn*>a)ZbS>c{4hyRDM?UzREr?K7ZKm;3V5x|JMRD0!u}esDLdd*5=#APRxR%w^+4B^_;#U@yrcbEO-Y|%Z*A$c}<*-@DYQyfJcQ)>yd83`4Hy-px<NopPaBy(!b>MV#xbecC=<KC}NviXwz5^;WfT>|K(fRky!8&lv8EM^yyE8l(Kj?qb0iGzz8IGL#H0H(6F@yvMM8Hm}LN4eR*lg**hXe~E;NG09wr|7q4vzYRQ||^QEq+>o9hW};^;qZMvJmdxX<61j#k@Z54h#n;9>i&+Xo7{>nsO1h1FWzEQ%a%$(3D05K^JVZlt$RSN0w^afyqh{X_!CuTdp-CGT}@T_~>MC40E0T4hx-s$7UTkM-f(Ji8_Gp`C+Q_A7~YVZFJ#U2qBBc2^R{uw3<&r9;QSpX<6gl{@%{%q1Q-+F5JCiSyn)%Kw~^3W5vg11KZH<RJ<G@OVt6XM436M1bkVZNXG&IKnU$`0syxJ;FfGUz(Sq9U`E*2l>)a!Gq}LYDTqpS6P-O*l?f9wVb6SE#U%%Ojp&DOUt#b!v*|#rvnw{M5FAi6COlP6*+4Yr)>#g9ew~<|kJdq2BRAKB(AjlGvWoqK`^A&veeeaAp=(Sx!0W0maOo1Zpn@}ZMwE@-8wb=XOX<`EevMH1T9H+$V5KSLziOZ*bm9D>{DBiXo|D!8=~+i*)3UDJbgR)nMK=P2K7%eyQJtFey6U15U0Z8*L>w#oKUvh1haC9#Ft#dUO1#6!92Ruf6YM5jDto>9uIts|Hjl}f?4DP7>wrB)P2kSSQe7Yi%zU>lZdTR$yY>6#Vy(1$W0=fS(Qeqf0#ixGQ!d6*VU<dIv!38p#Bopxl8O7d<g8vLOP?eP`sK@9QtzFBGM&(Euqv0UJE4NA3#xrwRF(e#P)h>@6aWAK2mk;8ab0k;1(q2B003YD0018V003fPb7gF4Ze=fXa$_%KZ)130Wq5RQVPkYJZgVbhd8LrCYQr!PhIc*1af?%`*r7uQK?;VhA;Cj3lv0Eu+oG0Ek(4+P{O-k0I<?S1r#tw6zWcZxz!LD%rsPFYMAd35V@_VXv+ZgMR;HL!Xo{Y~-Um`TM&laW6;F%Bb77@%1G3e<6sl{Z^ed^O)h>NAQad0k?`7<R?0+g$|4|6KlgW^at?_X{w!OlR!HFAbZjA~QsZMi6R-9}%AY1=cqv_Ud49GUsoswRvPub|-?byev+`tkp*rR|Q2ec?)#Q`M>C^=w20Tl;4p@1z1BowgbfPM@%)xg+Z*KnBmplfp$5L4igWHW6KFn<}RU;qG!c$j5<#xtKyNp{Kee0sXi5OFF6osPl5==FAGzQf}PWM5EA0|XQR000O80041ah>TNh8wLOX_zwU8ApigXVqtS-Y-w&~FLQEZFJy0Hcwc3BbaG*1bT4vsZgVbhd97E^YurW@zU!}e(fS}!S&q{jipv(<(Du|4NN5kPSw_;VG+Ajz8I8Q_mO@KTC4^uogpfn*m|#*8^2<=VBIF>G|3m#t%8W+)QyV9xdswZd_rCYu_rCWuCP~V;0FI45X9>7K>Kim$xPfRk#N0594s8F#1Okp}fH@4e1oFVE#xH3Iew0_mJpw^Bew>3}9{CZR9aP0F@Xu~h@T=m!fRI$hbEKX^8ohh*?udp^9#Plmz&9)oFoixDiJW8bBL*U>#*g<E%zfZhai50rUh&cgQH|e%N2>8#3L+xqpYnnk2;{A2f~Cs$nJ_wFedfUCB?mS?b>OlCmoCnwDDtX!4MF*ehVaj?f52Y(R2P|)r;tWfaaRVNL6y<{syIPL2Yz5Ya9|LXr!<tt<6{Desx+!u$*ST&S|6w)Zqvwi;rsF_2%HHRL{;%1)xpmW%40?1_z8uGl}8fj4lz0tW(N_g#>X-nUX7pA5aPOQNkR&HRq-2a&edj(aH1k*8}hpsDu&&3e`~8p8Bc7-=s-xyBZ*kt20tpF(FjJeJF3FDDrkAQ-;^rzh#-3RqADKBgsS2Jff{~2LBfxKBSF~bMh64}KdOp56fVyt7ng_0Z1AuepHRudn1&El#c>L8`9{fr<s!`mm?A1qO|TNg2YKp>k+6*p3?kBGHOJl}TMmc_?W_I{$~Q9Qm$I!|Ct%4|PWo(gWWUBIwf0VATCa%A>MohnMW0czo?yX=uhc_BRvr?igrfxZ2ndu76mzy~T;Kk(ckSx-m9MTr7fb*E(=Y<u2cFYjpYa3<i>KV)GUPMT2cB{u8GOemuJ6eY$0Fwxtglg=VF&a@+UP%QW$LjwfUbma`-BD-&D@)oIW!$GJ=3-&E@eXb)!_t+iNC^ww0Z2I71n#0i|v?%)U@kd<PGS;9&$r0<}6XHkoyvB1|M*o;c28?Zkw1;ydq%UmR`e?B*M|<IVu%r3TW327;=<a0*{0P{Y2;nG|SGbio0+#N6iFF70b1ykT+`J4T)<jv8vsc;V{8eDCPgIzMmisx~R(yDMG0ouPW~NAW#TZ@lY!6(Z2Q<)%b`3N5CcafK<h;9L${y-qsqd?V4Kt8s9HpgW!yYdLCW4jz>an&wFyPST+932f@li3d4v9oTbRe5a;D7g;8FO9}D1U=r$UZoJwYZadmYw!AWY_(Cvb0F3D1wO!T)%doPEsB)Gw@b<qVg%TvrVpOaK9L-x&`3t66St^F*)e_7i#Tl~HEdwVV3YC;eVbds%}-UYJ4v)#PK_&w-L*s~-mVfN}bZI9>F;tu#?<TTpKv`qVnoUPdvvNRw3DmQICi+#qyk4PK<p}?nUN!@kpC$2!Xm~;i|Q5R+ag{gx#pC%_4r75O?Wtp-DQ;n0;plTJF&03Hj9JdER-p7TQ?$-0cJ$^nmN7A(3s^*WAP4*ttfTJOyp=m2k`C>G?kv=AzFGem&P>3zXn|*$0=H0-)_>jo8oUnW|$S^E!g3MXTS5ITO66*Xy8Vp-++1?Pv11m)wQ?WwJv*N4`|D!^eT0A>R3S%*YWs(U6Jw=IP`CB9+({^C--c`hrd;|RteP>A$e{kEQ{Zzfh21~UnqT8#iJK-}F<IKm_LIV=aL#2+L-AO|Rga*wZGb-e3jnE0=G&1QcQ?-!QXevT9;`Hav`sVJ=#obmj^=wl;>nAg<pQa`)pbPWEKjVC97d~myoFy<@XhrQZX~c4kntr{JYx9~soDi|d8_UI%w1!;CV++$0Vo5M^pe?lI7t=h!90_bq;Ous;rH1B<W4UN5YoUVlCw-L6*Z<tPB<rzM5Jr3Qc#ZkQb0xHP)N?iORiC34qLN@F6TVmvN^1%!;wa=Oje40tT=Zh(;dn(V9k^0H%Hg}Mjr!_tAlr4#z&7;(sv4i<aCX4taShhL(8sw^ea1;+L?{I1-|B5%A9p^CzeOp{SgUj790jsdO{wZoYnpSyGEMgeBh+HK(6g!eoG@XfYKC?3jf$pf(_6K5o#`O+iZ%WPP)h>@6aWAK2mk;8ab5P;dqjZ)001Wk001KZ003fPb7gF4Ze=fXa$_%KZ)130Wq5RQVPkYJbYWs_WpgfYd4*M7iyTE1eV<=(XkWU=&E$P?7P4j~f^3##$;&zn)!jAS<<wMls;ehijbQM>2MH1oM10xJDk3WD7l_jjK6LPZl)pskV?Htt`c%!Wd(XLb?>)CCO3p9<n^jV2h`Kj%2GxjqX;f6?RE;a5NTEno_Il&PyZ3G%-5cM3`}Wa&3=v0KR?!|JPAA%)snlA|?3ZFi=}b)2ET?zYpAVv>H$J|5d^CP=>m7^oK_sN0)<G~Hec0>uayr2{qnwX5eXL`h?IF{Tll>k5R7B|-L!52r7gyd56RA*v3FvGvVIn7-$D$rQjH5@ErHYbd<e(JQjS`3u4c?PX#Py)oidC_Ps4awNV-u|Il4XLd9&>Uo8$H$}<uukQPjKS^sg&HO`Wg<+;z|_AyFWyMm(NZ6G4k$0z?w^BW^pM>h>|pjr)Bq>C{UGVamA2^K+|4>Gt++3K$m9mqfYSh`CmVo_9_b$x89eUAmZJ{%mx1{iU@yy^f>}xYT9QR&iZ(Q{%C-cvU?$rT1ahKes(VI$V$_GXHV096To$8Q`Sm~0u#n5W@hnyv%gHwzaP_*Fr?k3#2b(Hb;=K=%BiXkFgNX245w!C^(o#oix&Yk>n^k9<L87A{Q#^=FlVNHDw4gxJ1zgC9Ehhk_08_G3eZ!t_%m5PUCyaeET>rKgxL4ObJ*wnp1H<+>iW@sOW?YRTAy>83>>UZ`H8~No^GHvr^($#N#Y#YB>1u#$5K!~1{<UH7U4B@#x&&PP*2n@+>r7H_%i7K;oSh+r!#?9WV?=`9K$8zHoXlEw1JxPB`c%?<%5Fi&3xyCM@a%9wWC}Ke9gBr&tN!J4Gp>qvR-K_i2B9>YOOq47`x$a4SE+QsM`{wRXMbBJ0bLNYbj{k!*#B9`MB^MYsiCT4ovOD%2pdD6wg+Li1XONZqWFdz0+NY0<-P{Gt)j}j@-@U(+0}4UpNi8fu0djK$!MRhP)94itYy|_0+VV;bBAkirIqKs<T3?nJAHWzlZ`_nZ>mzpsWsKU%&*VbYh6k`apjah1(6~ZQ*(yCMcL`uL)*EO`FOKW>rQhG!u<)b*P1>d$k6xTLV&6ROE5wZDA*8uH3HLzA0>iY-2PjuQzn?Z?LmIBD{_WPgaPo3C`9ZMILGSCwNx`CkK-B{sB-+0|XQR000O80041al`&8r;tK!(%pU*%6951JVr6b)Xl-F~YcFzjZf<3AE^v9JSzmAC#u0z_r<iPTFm<MB`_cr34l3aK+@8^#eYhy;rGh}L$dR})wae@-{pSn=?MKK{(Y~}nfj+i*XkYFgQ{4A(KS?pWONo?ZCvEj&ayk2(+4(<f#@LC>u|k=nv@k)L6?3l7Ayd+7RhFoqLZww+XI3Z)naU9)UZ4gsuave1Y#|KHRbH2vFvhl~TFoJ&t!|46gm9#4X;H_J<CNE>%|&ME<?9;#o}62>Vvfy*=6x%?7u*!3m^80n86}%%WjsX1Es-<`FS#um5>wPxl#4gZtuR((X0cLFDy=d!%a!YZTGt|PAnExhTMdaO>5Vm8TU$9!fg6K1J;&?FJ@xuq000_ot)>4sV)nX1Hje#kGD90NqiUTY>xIP;GX;P0@h5B?U^6wZxW;ILQ>8HmKDDR=crK<>)F?AFFo4n6El!n&5~nr>EoOxUA>oav!u}D<RaVbYTI1v(u+CaQrk^mTMi{^in~D-y-=U8D%lNiSHey0nwN0~v%Ngcu3<HST^l{kToUR8S3?On6E=j`cf@aBM>b850q1H;X9w_ZMbXJ$*w`6P1mwF-+ZnTCM*mu^vM(Ky8!7etBVFH8ZSdv$!P_})F#nL8`D9-Atqc7g9vN0)VvV7eH;Jn3j078&Ob6!OYZ;T1o6SHy8sr??hNt>?2a&k=+Fla|_<bt<LN3Q1}$mv>ayNX*Pj#gu4obZa$QF!fa3nyJV`bMskhq%tU%?cMQ_wirbBBsr*bvXJ}8ZU&Z2{X_om8!@oqOSNYz-EtkwUS%BV$LwFv=CM|Gi|A(QP>Uh0iKGS^%opQOyuN}%mupt7SfFSa4-7mFQB)eJ7P8*8u-q~+ukPs4f3a+h}<`<w1b|@>*#`)H98|ZFZHiNdYnX9oSUde#N?tM48UiT70tCWWuq~--;3tbi`<kVomuGHgPC#z`pq)wWE5_TK<&d&*BEpXszYmk$oi(9&$$+FFw(U&ft6pRIK1)cmy|N6qAVmtjfobvXbEVL3bGF#T}!u-{t8Kd3@Jd@a>NLXY%y@K%V-c&kjgG#y1J!qLC6}H+`OD4hv5o{RhE~jVf|L^P3D*jE+efjM+<_n7foG~XbrpG1AfY;O3&+(JM2I|+8>XW>koN3n{rXs8fo{OUwK&;YyX2VT$GrnOD|c0?R?t|zbvfuRq}bTu0ef)I;}KPJCGW(b668{irTMeoGXi|nDZIBg~Hk@Z8{ZN$aq#@S_o@aW29rY5pgEt-D9vhS}iie#xB%}IEMY+#uK$YJ@5!#u4|bty3VMsO2u;%gI3DMfHJR?kT!O{Xa*Y`JO+&wTH#vF!9h&T>P(|CgGvk8v|t~?E36HIlI1nHHieRi$T@9G;1q=|P`BD&ij3NzY4^TfxIL#m*J+x>hamNcQdyiRb)I^vrd^zI0$y6Y$u%vZU09KQ4`TSpwKTNg4ookU|6G7KWa~QUBmuXvWme6FLHL4F<}r0N5j&CY?5)!pv{uNOVZl&JSs1wPIilxYqS2`G0z*A(%mRbOW1@}KqKepFz?g7@-S1T?EPapRpf}q0iKRfEqc+QQCZ?cVT|s#fv1eyTuhZk#!&iq#!}RRvReJdHB>mO#mu<HV5F>V2+d^sa#@W<W6ZUiD8Z|K3AykwXLMu7CAJz+{(Ns!0@tde|9`25`18i@hxV-{-X$uxNQVhc17-(=&wH+Mn_d1}7vqw5rI;F6xP0*=Fax4QO%{Sp=$i1G%7gjQJnUbqv?i=A%ic7BNn5T8c*1X|WElIxM4=)G`vf}-B9L(L@e@Zal|5?tSLRQ?p{q`ElySLv8`2HVoUfjL=lYoDHbNB8Kat1#2B*EZcDQbnC>+AuelEPpQqGpIoq`HiHiLrW0uMvCn<)itdeD~4wM=u_Iw#$03<7ca=mlSw4>VFb?e;2LfA;e%0{E)#~pOX1`E_6gGQCc&g(&~5mv1(;35;DiD7(%+rx#c~I8kF@MHMbb8Mr@OldMzzd@WY@HZ4ajG#^J&a9Q5};%5RfgWml|2pImBTF`ZmnG^7>2Y>9%P%f5kxV||zTV_>w~l}$RNEvVB9<59nvi4(uzaY=u<n4z&z(|83AX%IZ9vf)df#@6V9z!tA;L=;J0&npuJ5=zdd)|hgW2{HHuFAer+F;lsavjMB^boZxCLZJquV+P89+~qS(JyF!khb=vS2#2LbExAQdI!6swfyTVfkfw8sb8bZjQ=yIBGngrv?^5QT2p)1X{rt`W*t#l-(*<gSxl5|1=i0^~%_Df?hCH7`J4Lw3t`4hPGn;}FP1&dKzq@<)7YjE5a0d=<_ikEeyY0ix_VCsFzu&$4<I%R8EC_fofNh`Ex4i({pB;X_eGB&P-8TZ>e^<wFLmg<)eec%A;P0M-uSLVVw|^V{`?r4$+FqQywp8dEb5B`Ytlb;pe9JcM2m#dvvl?+3-w~r^SPF@(eTb$s`9m;OSuq!HC_#0wYF*O&<<n`@&Zj>HwM8E4Uk5y7d5T>sD~}&9Gk*;1<(I?fXQwX@htESqxpPr0=!`wo*Fm%Q<m~wL=<M0a=`UFyf|tk8ttqF6!;>$L(-)o%+h-30uhpZY!=vZNv{Ox|Yv4pMJb7_^_W6)V4xaqi@W-dmUY?zt()L=H%Wji?m%<hh1=raFcB%9^>ZDXN)=N}{GGdo(L!1LQP^#Jbu&UA2C7EJ*!ItP+Mk8}=61>8!wtP}z?5xq0kEMwN|LXca>#*HjH?sI1F{bwrCd^~Da~kE&-QA`sVKJTmURbY$fZ|_z;~28bd_Z19O1K!b+>Hoxs`;gwxo6aR&^q;J!)Iro4_7APX;4t&yJNav-Jrwus(qh{H1gc%>{S!R;BK_cfN88sP8W`&-H-S8_xr6|Hpfj}QqrK5?<!xLm#(ER7Y}h-xjJ&g!ZHcE)@+%Hx}>#SltHa7d-o;U*GkU%a6`!3VQ$<W_md~nTk|w5H^bld_!eGpTMVx&Tzm`T9epeB-u+&{+0m=^$M%p$L@wQW(x%_pc=0Nx)JuobLb>Z^ty{lcns-wnJm?HGq@a}%t>S`;nUggP>wRak*4(ui=+`yhsJ<KZ8*eXK8~b5b46W;r{^DYqWEwRm9mh__Q8@2-(St0mE>><fLVxV$CDNG5R7LF%D}=F1i;S0$m4Xub4z$*24)&oX<P0thWVNOq5GHgB2bHK?EqMy2;2LwNgp`=OG*N=<ob_7fWRqIv?xwr#|BJNbKK_^hSi=pILO-<V%@iJzJaF3_k;^=E)_#~@_(Cv3cw1vYrB==e@7GG_P;u?lvIFiSz$uGsHygnYXyhgNJnR5<c|_>b5P95hPF_9*yvoX&Mo>R3`zF}u*pe;;w5=;AYcO2JvvWQp3DzVFhNpt-pW-kd^qr%UPc7Y#_1n(4hV^yLOJT258n@}JFUZs+zE4pGa^(i0N~;TWC*)aHYo1*{1tCp^nO)o>f69mW+Uwhu2xwA9!hCf2F?j$JrOK!!RnKiAHZ+{OAiL*E=iW`rfo$cv$?yH4tD@Do88IJ<!YG+i30JjEbF}XEC2gyhlCL{gldl%t72Vf_5G6@N_^vKDH$+Wzx9{zK2K%i^GYF<8&YSbpREW9z&qk90u9rEZ4YT(Ull$@ATtNPvR;V+S)^R%6--pK#9l-8`qVGK@Qd`w4Gn8L!jZiJBdnPDPv1x#c-FPbP;K{wC)Q2!E5OQ@XDIlqHY9f&CV7ZWH5ej6kQq4>ZliE7Z^HGdBC@p3}@-j?GcomXwI8+1AT3<U;f&XV}lBw$YNz^)V+(4~p#_u8rcG?q?I3=@EO(OPq_wimxKFJ%=xvl81S;2SjiJ$rtcSpJdY_|jSLtNX9)BnS?jpKQ8ofYDO1}1lj79i*R9MkYAYPGv`r5F0$UAn6xDG9ts?JAw5kszRli`2%C0^-(x08mQ<1QY-O00;m80C8QNCz|3f1pojP4FCWd0001DWo~0=ZDDe2FLPybc4cy3VsdF@XJsyMd7W2nYuq>x{_bBf=##f~oLss4a0?4Z`gVn*38jUD5RBFy+oG~0r156AA^-i*i@n~Y>6PY#ogF>%_RMG`GsZ4a)&0Qj*E6l;0eHg=p<P2S9BC_RF2PtdR@dm9P+b99A-JAfY~YUD0UH=}gR;!B&pm>c={HEt8?+!CG$_RuEhm%*uxKQ&y|SPls7I(rYf%vxp%gYrFLN5UntCvtdIzJ!fOUL8xI@wP<O<+Ks7Po9Ts83i`pa3Z2g6BhB|<|7gJt$ngR#09>wp_{GfD(87;VV`^}<a#ml<O$iz^8r#kL$t$VsYQ)>=J;n55Vy6kK!aCx|;F42fa#0|sqd)p}HfHjB$Gcf^gD5buP<0^Vsm6!67(3cMsj9dRp>2+LZlNLGEA6z!-gp(^Sv^2TT_vn=~`b@drGA*@`x64g#vt!~(xtJnWlmv2A5{=4!JcDpJqIxTmYuQGRVrO&Gr%AuwFq5wxW54O&MX+vTV->^BawdC7M*S2zKcWBvaHS;qIfxD@$S(Y`}hO$<zkeDkzU=cW6tup`sXRwCNBhE`NtvmritCG6rzJyElpq)zkL_f--1#c_&kEr5vjmz%Fo4bWgGBd|h#x^W*k98H-^7x7-EYfJFe#)}U@f|jm)Sd9DEFCb<N?2gG8QDY3`AElNy?}}CEq5q|!Y6E>&*;(|Sd7J3+zCB8*euO%r=49*>r=sg)=m63`e}vYQl{XaOV4M1<RP2<mQ{!3Nkj>osVx@5CD4q%(qB(-NtSVA7IJP(YAuVk?v(h3bEbL2i5Gx-np%vwC2U}GvKRB2Kf^6np1_OpBFKbl(Vj@=()qtFc0zsM+7oPWp*oMA@~pMmp{0s+WmLC_7DRhQDa6KA5$k;G2Lwuo;5B9kC$*V9E|NNp63$}yznqNZg!^SWehBH>=0<g^A4heh^?zcr=tfDffqNF*v2}F&5D>Vp(Ga{1uQ30em#u%K&R78qt<{#T!)U{Oy^7`{`iA)tr#crJ`JT-Tvh|V80$9p{t*2~?lN3+>7-`}%ns}UW>dA-YJc*HHS@st<rZn17&d%qZd_L`0w#v9Ws2Y6xPr}^q`}N}ZfK518^0F*fX&)pv{Emz7oE`boom3>Ow_SQ%zABD|$30cL4US9y@WT|e1?CTh=aS$p`BP(&L6H1h_^GVVB`^(#V<LiZSXAgHXRWrwC^`G})NTuS$E7<x3hrOLnEEKb9lO1`xGi9AtM$`Hd-yR+=UYy7|CFmL$U|^$4N^F|Ir4V9B(4C_xX|xjs_pP>H%!#|dA=~e3D00=BN|u0J{%J@uY24u-L`D5v{n`FP*Ki8YmbJl!hH7E#N%jAzNd?p4!^=h=la7xlla5HJ1pYsjpcVqg?GLyg0xh4m$OgrFIfTktKTmQ`14{_0K4l2`TgM)X7$oP@}R#e3HOxeuPzFB^@rEk!%O&$6}}JThHWLUzb5_d*tY126S}-S5V{XnX)Xp+r9bTQ2r7VGe|(?x<~IG#$rRybhqU8zMBj7Fs4i)W7L=xBz=QHMzY=S~Iu{P%)2RFv^Q|9fIlD|VAaH#)8fdwY*sOuU%P{Ps#pA9FrWftjBbVemElia8>JY_`YDU`D{iD{Ox@S@-j+cunU9kNOe(tlvM;X5<(pSLp8-kq<Sk{@5tz#5rJvQ7YQ#H5~HCBF`=AjY7`i!fpbcj|}0nBU7OKBmPddU6*P)h>@6aWAK2mk;8ab0&B<X<WY004m%000jF003fTZewU|VRCCPb!jeed8JtIYaGWJ{;t2`WuRyeZ&pzp(sI62Y2+p`v18%LX|NE--0a-VRJ${qnOR9^A!zCkA%uWMT1qLUQG^@(!NtXek~@k&oa+B$`<ImYv3GmY)pp5#kZyjw?>qC(Gtc|%B80AUT}X}l83kU94tP4GMgtdyssSbx22z1x6f`Zc!iGYh3c_-%_CQEO`%>;D2%#087g8DMV@=<AyKXU)XDcfMC38@v5L7`JP^IY29IdRZq;vp^+M%AciTv+Ydj$YMmwArW2)bd)4hlVJ*NtbmPcsc&*olyl24_8uVnEx~<o99-hE<b4a~N_gAR6-NccT_+#m;d4L`YM$PgSo_cAwOH*kt9XnmiC|M9BiG@tI&?s_`iYKl?TOaWSCEd;7GSJb`TbFSxyV=e6Aoq{_<-I|rXsldr(2=@TH+e}tVqCU6TIwmmB7GIK?9X=sn<cu1X(YW$sGFsvp|IGmqU<L4qQVwp8Frsyg9SxL2V)_44Gy93mUAwmQbMth9=;hQg>R+BRWSv7t%0yhRE<5ZZXTf;mnm0#9ob3=r^c#45cpTS-=es18ivKpWHc-Drdy;PDir@}`)e^^aEhpd|Xfuoj3gpvmi558p~EO1U^j}+h{K*qz%KKnQ;X~U#8%VkRK{0Q~wKq_kE;eoN~nj6m?K)1$fGNdN5#xWcnw_MAvpmo_sC%7o6NF(&0-~0{%9|Tui!$(q3pqZvHKu4~(oxC+Tc9X2Pgo;!jaKj?h)2QXAYNH}Iw>1xD(@}>RppH=+9g}Vxxg;ES)=cW#emy&H?q2R&ulv@F+Cbj3qPolsbRk;p7)K!5j569qeW_Ba-XJo|aG`0t{(-=cbkRBjLbI&EY0Aa0i++kWRxjXV)i0B6Y_e+dCFp8$(pfXib!{sPv_BCr)#QZ3i>Kvm=wn=7?6pf?g)4wi@@vUOWV6w-?arE8cD)}y03`eFHl#gWP@0wnvWM?kX8|lcZLkrMd4UzzQuOR;$kOyuYA7)lseCY?iV8w?U@OOH2Ak5t4SmQQ@(VUSHNbc^`9i=Ro1PNr+lFzM^!pMN2BV~(`M9vXc8`SdYW%!RYTWSDg8*$-<G+3cAKs`Yzr6+9cYe`!*U8-xV6qzjll+e;*KGE_v@^e0U{2yr!G?VRhT$CcST#PA37Uh<!{JI7EXInBPJd&2sVfssZ%yN}4M!o1s~1zhy6ezx$>e)+4d#Iw&j1a1H9pIicu_PhiQ!W83aSVd#`;?y9RDQgto1iKeYJ7qPvN+;R$PFz-Gkl2rem-M3Bh(X{@XTGlfMA5O@_HWkS92STQa2?oS#h3Kvd(?0`{lh!U0zK8)Y%uGVC!+yE3`Fon_@}SWUjkp+9{_;8#+mF%VW(<HrK}wi38{6k9HiVaWKxI&JWKH@23f4efcvdEaG`Nu0*E9G!8;c3F*&^ONZ_J8$}oLw|ZIflN=AF8usqjnE;lCMRMDg5k1eaZBl92ssyQv22~6xN1RMIV5cQL@+p<o|2?-9*pA5u*OyM9pNU;Ps<+G`&)ORj}2iiz~3!7jGwmw!4)S%5~>L<Xk_`(dvUCcT!FUtv*(O^n%{RU|8vv|4B7~D4fc37d60uq*av%c0Pl96Dn8&eMKQdGGfi79M?rqGA&e=N2+@Kh(o7!AbyBb7k&@EHKt)<eE^2`kL{&q-VGZG@uYgv8-Pjr#CPk8RU1WGvTYa#7>t+n0b3VHkcRdf0dbV%%xS_dTakImCw+@)R$1{4@bw-#FT9__MQb`Wu<UTJ@40)LuUSzbptx6h0N;OfuuvfK<EZKi89M62dcJId4>|PcWn?{W5N*1X+5Rr6-y1Nsh^M}$RPp~$uL*}IjiecRe2Rpb|O@3&@)p{%rI>o3`3}L^Te1&4T+S-kM1@5l<*uL;LV!vA&BSo~ZZtH$_$!GNkxxCsMlZMS^cqB{HtzF%8&FFxqhIP^U&)zH!Q9zwj8LtslHw)g63D}qJroAlQCacD$qdE0imeCUQ^i(huL*JS7f(p<jAyn&2OIpU`;d$v|-+HUDSXr8)Yy#r8>xlfQrkSh0jd6?esD>Ar{vd{yME~fK2878_SnKU1aLW}-SAC#gjUVNHt|c~?1*hU!HGX1ZTT3morOvO@y-7f>&xtMdkq1)jPd0Y>Zuu)@uR38^3~3tq{|$9%m7Y_k_D4g@_N+8%UdLd%g9JT5yFm#z(~;C76`envo;hm%T6hApjOW!+lggZAorSdV++td_0M)hH@M*n%d-Kz}dfdCQ<+&fb?uJgV?tE6P!;qgvG3`5(8aL3jYy^ENGq)t&NLx;^E){iJTQ{=i6Vr6zIi%TR;rQd0wcx>HG9ekos^=Tg0p`ZRelEt18ZElDnlTtJ4%nwqTo_MZ?%K^ko4^N{TV$Sz8hFeS+51~}*IXgLR>Lv?)>?ftQy2+sPM^8o4<N6`kBq&|PR)!>f?^nCrDhkFUO&DX$AA@KeBUB!;e9^JxHi!OG{bR(7n2wk%h+;Ga7Ws8e`9xghI_6LC+VEUP$)T6RO@br^L~mUr&{A7ZHLr_=E8<9?<kbN$kD`-4-rdDs30<RDDdArlKwky`@SgTB~cWoz5d8h9kp6XO5GsBnsAO@v4+8F2ZOC}Hg8>Y)V!9pdv`sL5imsuU6f*j+qft)PO#0>+I?PJ^Gl15!eWmvzh_-|R=1rJoC!X-vY;#3406%E8(Xc;8s1o4lAC~9s|0}j+v-32`r%(-JX5#P3_f#ndT8YW{L=m7Qdk*756S<3UBX)xlhbd7)q#)9{IivfOb^LC>VONJWfv~x>+pvYatl_kK_ai<w}p$+6j(`kvk5Gp#-4$@$W5)QjVm#U6g`r0vKNIb(QD6ma{eKs#<VL(FE{aDJZ2WQa=l)R*~gM#Swp<8%AdR|52KyM8r@-6H$Dzk*caKx`(pFmo?YFrABE~yn7YcnOknskjOVJr#&>KL&dPF=$JXvl{X1EHgnS&TP<s}#-e)@`ezxCsUF4C?l^vIc#elqP)DAI@SN;o7O9KQH000080001ST{)Z3x3>lW0BIBe022TJ0AgisV`yz*a%(SlZ*ps8axQRrwOHM6<2Dk1&tEZMU!*Og{V48GxX?op+@bey-JsnA?qCE0BaLicBvMOKwX;V4_rvf*wrnNYEpU2CNDha?`Tb@%W5(D&tnP%v25xTu1v_mX&=hd{pHIJh{Pp(dPrrc04r^xt_o8;)BWQ(CcM_EY*$*A|sGM+8tAa6hCHp}e2f}OyVJxQcR@iMP@6+(VR;#Pl=pF{)XaVBFt#I2UWVfS}T@Z7!$0ReLX{%a~%AvWsx@xcmtHfZp+GV{k597e0*2>}D<*O?I0AVdU3ni@Y{J7NyP&xAmQYF!%7fQBhUD!gOmJO6aaLr`H!e;u6*g**j_b$sC;e^+yM2{T&@I>v-Nas)$(qV729AxVS3uA{)x{MW!!~1;w?heY5PRu#nGnP+B1=|ZT45*syKz6K1jstU#1F}^z(2wpoFsnyXPYcBcU6$r@KIuLCnB9sGfB7q0C8&xpkr}bei!JUOxk2l)oC71e?C1K!RQ?p~Ht0s9;J^YLYy}Ay7Oulq@m?I*P8cQChUO-p#i^$VuNhU=>jpW5dsq&rnn6qDIOx$0V?x@KrP1}i!o4%1c8N4~heB1f!hs;cD(tg<Tchidcx*ZJ=!~obSvVZn*14fF_<2O@?S99B8SJ&gO7>!dX|5La7OSm{LeKd*x!bGtq6O`?X#PM`4F(5c@Nz+_7EPLd^2%ik<P)L|yT?`=WJl^g>xW*rt**_H<3?_z5*?IjsYsVC25sHdBuPN@DC}sksc0#CBh1Efc=P6ANB>0#(ibMzY>T2nW7!>)a7eO&HED^%;faX#@9U2@zg^#uC)o;^Zd(@an!X6m`V5sj_`Zbq%aU`f79_0`hSFa-ODx$j0_k(+aeli%b315d5z5Jp7e2Yzlrjb7h>S(R83*PeW@k=3zfPM7aUdN}RlHHMq@cu_<-xm7_CBt>cl}h$rfnSFUD_7oHffu!w{*R3!3%RtqCs<4!gccUw0n_kM8Z>sUA>CW442c05UoRV5)ghpgg`oSe!cU6Dei@<x0mUo^7F6kjkim?Ki>Ry<6q1<@dA?cv{AY_d}|VzDXTkSE&MZj?{~CTYL!jijS~y(qOkVSY_V%Oyd2XD%(vq}Xn6cyXMcog;;VjGKH$$s%93~g&tbC^Iy0f2blW2dvh;S9$44x@9#jmKKJ9VvG^?!qg5?h%IM@JSd5&<NgRrq4yL3pHFJhiu&k`)<V}I7guE=6+PG7_G2*}n{(X1KS%<eo`G(lj+NBWM(^DjLX)fVH>2#47zR!3p%EY7bq8Fnn?PRVk1{-rw>v$(fsUN+9Io1NZs8T`qvRHmKRp&S^?FZbfU#=*hg{2!?mL5cf%^_7P(#^G13s4Q|O!+o8foZ%g!8r*Zpf<rj5bL?DBK0-D2Xvq0r<d7Z9s&q(x7GAIA)&(zeKx4E4A7^7nSuw1~;UpHvUV4uA1p~g~n3=rMRHz?VD?2JF!adiF!a3B+sp?HDae~&oS}Jqt@4OC??L-x6L22i4xHNTEX<m|oU<-@zma0lJ-!ga$EjxJZ;Cpz#diO)~w8S2xcWah9A0N)own3N7`#k!d&DXi16?(BLq8rcp3{M1l=E88<xK^8f>_n*jI#k(o>@&A%h3rP->oj^<`ixwA*=EdbeMae2)2DbbdUurg4sJ#LAT|hYi(u)Qi7<gyaIf7KtZdN!1mn<Y(ST5mkAN-|B;!Dl+Kml1#j%+78jAj*ktU<pN;zA8G9yx8N$V>8;D<*u7U0k0{;%Hg){cEI%%gwBSnfMctO&l0W{PI)2b-O=IH>3esf4YiEdMS#i#bi!x{+#AveC8IKb~};p${*5@qm>u^;Z6Y*)nsfhW{c5mYTBJ+DTp)gIy@-&Z@+Vg7d|O=cSr573XKYsctAGguaqF?L_Bmt7skv7GEh63jc9nlN;IUq-&g+;aQALbt+0+zSp`V60T!rxeGj|6eZPB*;$W%jESJd=v|0~X`X75uhW#m@my1ye_j)H;<AORqRd%U9)qgtg;Z4)QSiU_P#R9Z{S-f&jo*IjS7OB;#0t?gm5A#ZyT0~laHzLh)@aKtA<WZrlhZr#d6Wj5lZbdmTYS$L2TD&R{i^gW+f!Y}zGYLO{HsR_v!SFLmH0oT#zyyqaG}R+RGIIT13`TxQzEiqHr}u>D4G!K7Xh4ee^_y>GW{_AUgA#+x=w<1R4-?dVBS&+;lA2XuKci=O#TN@O9KQH000080001SU2Tw-NZkei00<HQ01E&B0Ah7+WNc+FYIARHy_ijp9JvjK@BS4HpMxDz5=n9QmP2w(fFS1qzhpPu(>>^Je<X|j??Ks~i47Z`xp;3XK*gs>QLo<8?=N<{mu{Cj?2mPOu6O&({&al#>+U6{0I1dgfhtg?=!q0Dq+>&BIfyjd%fD<PPWRXS-TuFoU&TmqHcM>H1EPwujE1nguSgYN7}U`}#kf0V9lm<s%%H745p|F$0yp5MI)Y+YUn9U)bNKZB+8^q2dwFe*_~oy=-(T!@UnF-Er-S+JDbBZ-I=`#)%{DxoUhU(T`~PgTa7J4R&}Yb4X58EA1W-7+t*NUI8U~%ooF*eLmsWkUFfXK!U-2aI)9^DA8V|9cMM#J%xD{1fy?Kh6GO0UfF%ZqbY}iBvqKCR9q<w?DoZ}`>@!_^UT+bHQo7>y{aeuwN{S34fF>F9<71oiQ)fx=S($-v2X#l$xWOfr;O;D-`S5))1;yJ)O+aDi+h=a5MJSCwUmvLX#wL|MxjTnrCq6v9L$CMgo)5i$W8In%}?Ra_rSWpj&#j<M2R?!^bRDFPZBX7f7G7>_Jv8LMukSSdhLv50Jdj{@usXIT^V?hPlw9wYbV|4Dzb-?VSX^t8pGHGTxGWozUQmP}-5RsWUo(9J0c&!iD2VNhZ%FO~wT}NdnRm{n`0Ch6URKuD&mh1GS4D#M>Qli>37VvPge5~h;L`=fs1zJv_%;}Yk3E@pyDbr*n2Mpt&L45!tqL)s3MtXO4eSNzX^?G^O6trv-XM<7dV@S6Vj%@|#n$X)^q>xHI$tm2JJ$wcMmHB|@F%NdF4-Z6LxGY!CxrTOdo~w6hiZY!*0j3NZitd?1IcLn7Q5phENS{YM#Y0swhx8^xMH(8ERT$y&X+D}%D<{tA1HcBPbtC92Qd~(!4}O|=d?@C`=4mrHL}7`*7;_3buQVB26Sil^>`ibJAM;7%ZfQ7p&Zkl5w~w$TtP<K}O_ZT#O=4p<34-Q;+cq)L5H(d^+GcPMw=J`Y&ycgL`Jo<gth1$Y%dXh5(3xbciLDP)M697eNnD0)Nw<s%GbSoPFZx2T>lm-^?EGfS<@nh9%h<_)s%T0dea50KF>qDJm@&Op7-M=9LFlR$=q4HItLeAlKWYE1pZEFdK|q${0PMCPq;XN7KDDbO*{~%V#5O#pudqsDSJukbl;&E1{4M3}{&VPB;)6JuAq}O&0uJ_R`cd%QVuf#d(s|05Ih?$3Q_eM`n|~Yn{ptMXu|5e?`x;GUjg=$}L8D2h;RdUxcHTC!KrPkA5?Cn2$WS6ZAEo`#J3M_vdtYeeOS^aHa!M_lI=T~)fJ&4Nb+F+g)4OJ0W08CBn{eSXkzbwB0b{NnQ`O3gXAG$^qGg*=%)-8K`RKh|o5j-3Z7k;~UfdO)!MvXBcv+|OU7attZr6DInN1jsT@VdhpJnI_Awf92R}x9HPA!Jd<{_NTIFLe?i!rSEE&GkVdUdD`esers%TK2_w{Pz^&Hs#4Ddke=vsJZDJ!9~w2HLdPh^Sn*(|oMuxmpby*v9M>MNQhb$Un=I{E+ySDdE&4Pn3vF?M5FO9aozzj2Y1<$aKuM+G?FRS8kglF#8Ca-|5^x$Cdnu_q8nnfn>pG0ZA<-qG(e0)m5N~sVqmarjxdYbI6P_dYfa>Qs!q6zwA?#MOBMgrK%YvP60O6Trq(Mx9*csonW<|gk$r!Yo*PR#oy`ECu|Us6EABMbu%W|szF;d3b@bgAjxA)od~^+*u3==k)(-Z5qf_H@Td5oAMk#j5qbjpGn~*9$fI5N?;d(JfsBBnS}8P~nq63oG}`Dx%x6a{@}8coWpn8mqcm$3lRRfFzqVg}N_K4Axo9#pAnTB8DuK}oav&f#9lR8(L|Uj6U~U#Z^gE}>&-Y(HV4dx9eQeYQzEp7`Q+hfZlxUX`X&GC$LT2&COyZChb1fa5VnVamb8`E_@NNA#xpAN9)usSen-G-<6+M_|=tl$4L^35Uv>N!3X-2N}4BA(YuT5xf4PdoO2Pvf))?zlb(Z$$tAx!<~kA_hb@68sDF`MW2EXOyz4|j+6c6jrDM~mBE*08lO=cnIrS`(a^4PYsIl=Pf4Mbv3qZybTJLa64;7rWgbA8#ifFIg<@{#xfP`ajP5SNo$Kc5&F(@p^M_=S!V;cFbLzj_>Mxy@~Vb{$_t%b>6MR>HROef4#PI<?eoeJXY>L-kIz^N!9LL_Xmr*>|XCL*VB0)cG$f=exCW)zaf8~_}lJcceVTbKmWZuoZ`(dFJJueKTt~p1QY-O00;m80C8P_=Bft>2mk<X5dZ)u0001QaCB*HX?kU3FJo_PVPaozV{%_}WpZ|9axZUVa$jt3XK7<DaCx0qUuz`C5q}rxI~0=#&*o-F(vdrY>^;!wB;`OSEjpd!a2!L=OwCMNGu@r;p4G0+K_Cwyk0yCBCd5ZLOq_=poR?@HW&AxhpCnYzpPfCO31kJDo$jjYU;XM|ujhHEGUo}{qF&v6Bf!5ZZvQ26P{nHb)~znTOu$N=6gwF+mMK1h^!DE(hoq?8e+e?tkwHB}9jZmRy|uk{dHW_QK=a#gMGilFv0DBflH2dW7WAza+>R|83s&ELD<EFoytQz(T7D1J?VAON+pi_~S59el^S5TP#YI+5rZ0B3w@!Ji@jT0Tf+P6P@BX}r0+M32{3Gy6No^syeUm_O`*lUO7ob<mKXJ&5+rKNAt(ISLIA+EkOrP#lGQ~37-ts(;+=wm?o?jkaL`R=oK8Fc>?kSOb0eGLP_JQshSBf4v&**WDvGQJTZ$%#;9bTRfl&>qdk5xw7^p|1=IJc3^{EU|<ScUswtRBJm5k2>}w*UY<&pTb+yj@UKC`i?6`B!q{OOb<5j{XyD@#A-^<zGY|bWEkIo3}O5aPzhrfn6>C0hZXNgF@#Ynbv4)EucR>)JB!u`d;V-us3=+-g^Z+126P|H@YMRSQ-S+8-=C3M(vN*?g+%xM2QqL)P_sp<J_)&ilr>qPW}4X@y^k+V~X-O0va6)hKU?BGZr=Y)3cKiSWV7-mB3ZO;fF7N{Eoqla>PXo&JQo{_=9uXK_lG5P^_WGH80c1NS8xVQl&l@s<x40ws4RGkhP6cuGbEf+}M}K>Q}?iEdL-17~~@q$kGyx>3!OB)IsO4b8&ihatHUz=FM4dfl<ng$DPIIO-PN#O5JX@r;M$ncbB6BJL9peZOXORyA#0L(N^-+Ptg)Hn-uSmV+NL#mgE|}j9N}IEh`|olE+;IPMIw>J>BpR#ccnDLH$%1%gT~jF2%%ydrr?Eo)Ky^vYeqm3Ym#&&Hd59+1BCzpRJ!{?B@gb2V^B$s_44l7L8&F`by+cys&8Y!94;{EKRw{_kjxw6S%+I6oQj!&TT2;azNko>1ntAv|>3nTi4tc&3h;n3ZKjHF&TP#zCL&~0%kyJo^!#vVspgjF@s6J6Sh3~-o?q|18=kzj9GG(YgvmFChchguUomm*$EB*45`<V=kM1IS*Z#ZC9+b|aEm_a4J0`khRIN4h^RCAGsw}#v7AH3MQT7u@fm6fiqEk$#Of7PwXqN*+<)*O00*gGBH6I9<cv(3bJs_mdZSQDF05a&Doz=weX>JMPr*lU|G{q4>~=?C3xv9zb0BL>asy{l;E-v?4Z_E)tno-|seLalnYe;X>Z&eT1EF)U#LNa@r3ygvys!bp(pp&&HL;vGLX>7}>{=?{uh`t*4V?P8&9PAc?W&b(y||g(gHOeU-<Z7SsV!On_XgH_fdYrvdXZ7<gsiQqDZK|7*=`Zq(6AZ$jZoldV0Pk7&rUiL+NRJ*5YaOtZ#Y*BmYCScwfrcx3x&SBmC#CEVgMR{Tcfd<PCQmy>2=-6^Sr6X3^fR5`3&Ll@!ms_0uCyL8t(;gJZ%;(f3Y*&+ui*T1kO+cQmkwvzi!kO`&8-xBACGN{4l9gM!=&KXKtJ@#bRD^yYP3{3Dbjw$pjjl!@ODVf){6{WOnbNM|$p5cr<}uwxhS(e2--=;PZYz054O`vtVWBz1}#^k-p(ZtM^l8dr)-i$@U(4uYrkl0Pla8@ATWK->H)q2I0y1;V04g;YIZ1@;B3?36ZqZss$i5bT%ktY$K>A(-+<#Wmw9Tjkw5Aw;AKICjj>nY#<wC%K@o@6y*YJ!A*P1;53;;f*%+tm`)R!V%nVQ2Ez`)Oqj7iz0SBxFxh*8sGcKUePEzg#%g30I7I#kGCoJspy-JKE>ct|QWUmafXu)a*g$AUfGb23!l`6bg53f&eV53rlma1@NWIcR#jX&z1zn3lj<+rG(8;<J&=xXn1`GqG`HWKE5d|?9bkc`VBAa0Y*oB=GO~Ev#jk#adc}|dyS>iC)CGI3FDG+Lr6inophA`zqVA`^(amMMEXd0ZvcVo)M&RqjQqtY_16CRfs!kL7;lrbxzlHv*%itZ;!1FtbTb{8#7xuqbqA??$-acJ^R?RMnfP2Sy&3C)b%oY=xBOg7gn&h52kYIAX+uyw2(zc6X=hAtbzD9UP)L{X2lAJb8kl+x51qv+0vxu}KZRaZQROi;ufb-?<(rPenLZ)*;WW$W2(HPSF<#)gV%gMQ~nm?Gt(-}*`|&d6FP=ywSFA&`{J7<fX?x*9_5lf5&U;-zN7P~XS?mgqEO?Jh+v&GezdrJS(Qfwh{)wME}xnFSzZ+Lvj6iO7JPt~7m;2;JleoF`|j!ja3=^-jo}-s=*;TW>OJwcF@h+&f3nhk%3)=z9~`yDQM94IQ3HwTMcYxFl|fU={j2op@0yllibQc>yGLJ|Q70YkxFq-Q7H)weC>lC{Xhxa$NZXVHbYXFxia6-8g4_9#z~JF7h6y@muw<u}k7zczN4VQ8yF}_@Uo~?*0oCNO@xW+z6wnsa6w317~{qm+#@?6&ay#Yk8bw;?qr%m*_CjFPL}ok?MY(lhhBqk7T09&h*$BeB^$9&oS|c-wwk7*PtS4zEjnP&s^zT()d`z9{5L#QS>TklsE4)8+>hV{TEP60|XQR000O80041adz9oCRT=;Q$4&qMC;$KeZ*X*JZE1RCWG`cHY++(wZ)0*_b7gXNWpXcXV{%_}WpZ|9axQRr&0PO)+{ktR-GKjxSKbfd#uTNMeU77M549!P=bpY8KHcR~RUl@`+2v%(86G)Y?Vh0^MbICL{$wCP5Clcy9)d%Y21$_ihe{~=gAVyWHh)R*W`^7$mn-Y80opZS?Q&<{ym|A!&F9Um<2X-AAs@eZ4#&GMp`iJc<`Bk<%Qth7x!_}XCd4?QuqzTW1oz|tJbUp`&%b-;?wzA}arqk#i_6b>4B_>E@G;1EarJK@91<Cy%rgqd2m9cD_wecF{^8Re$i>yKArj$igU)15Lg_&$_>|@n3c|<m;^^7kJMIA+=Ja?pVj+c?k|?1?0e|}8j|~fkxdX<_nFo_tjT>J70iNwWzX#uCJQC+c&(rfEe0=xL0Xd<cz;JQ-dp?FeO0y3soopQMzTDXM;e&W_^|K7Ve0_b1bOcw7LAbd3f+$`C!>gYX8H+GiQ{TOQ6UI$BY3`8E8BO8k{^PyFeLs!h%h!vmA1MaEe7(5*Vy+&10pazV5JH+H{@pu{<J`T&(oEzMn5f^zq=*wXTz*bTSbz9_A$a{k6nF28a*;wtBz6Jw`~{J5-KZ#sSuR3a6!jxX(`>{Nx_l7lltc`rrszqQ(u!C{^HC6rl1rMKuS1a}G?Yy6qH1-VM>MC=9t&mNh)X(?7F)bb67v*=salOlAxXxnCPphgc*+v$!Bdf^9=uOUL~{?`KRJ1^KMQGwBYW^smIxAI!zL|1r$r`sL2Ksz_QT_&m*$}d)jQLxkR(s2l++Ie56e8KJe*fO^O6UQ2U<Rv5Q*vZ;`u?<<atWQ)YE_Va&lfzO@w)nh%pPRmM)1%CIiMtG!G&<&!`8(k|j}42&MPIkjO9&!Zh+A%taQQ$4t^9BO&$f-hu8HoykCqJm8c@MW79<Py#9_u2PVSh$ecx9x_L=3<i~?UCCrZ2aau?fGUs8E$85<uJ5$$2wa&HUW`PZ(!7B4oMaizL5=9EdAhw`Wvp4excr=fnj@g=`O^=7j3kXY6=+TBKOVEi)jyPw(@#rUNRavUuQ-I)|Ha49M?VD7;_4p(ht|gU%M6B#tDgc#quva2Qp7$|{vf{o12L9}R`_{}?%;C)a&h^FLAtp5DeC^>>R%Xq`I;@Rev~4;tAA2<+lORv`2~lfoI_G7pNC^>cKy3|f^k*`i3le!SQ?OjAi{~;yL%@HNf^`h-`~9x93JoO9|T8_5BFgJPDE)|pkxW#Y)aX=dq*!K4B$v`diPEs#e{P98O@iEtYuz4JSTZdqoC9)o}khPIW0sorBRU6DML+x0f@pMQ%U*Mb@q;TKM9U^UmEsa?tlFJ!{^6Gjt5Rk7)M12h9sO&9<Ay4<nitY`$v1&!&VQ~Qz-@IwXPTJGh{a<Gq>+S%DDT;1IDG>MfmyA)BTtGN4xvMlgB5!?*|_~|GRx0rSJ55J<S)=oYI_)7>%qsn=eShB7)u{pm{D(c<6#xzm+Uyd|bc)UR@y5XxkO5YneNI{A_>8($@9^2YF&6u={f%fpK{Ct%r|1z(yB4+Ol)P3JM>Sq@?@ETPI~m%R<7C!nYni0+A~MN~~Ef(oEufzM}?v@9#O<u8dg$DE=}bAO)z9=;M@gT9k>@N(uwpecQ^*Qiepgw;nvw=6^&>7&6Ks6%1rvqPf2RG<g5`a37*lgb)$5S9~{#8I)5NikvGSE5V&c7&Ku@1ky4Y#$L{*6e0pC#+aT54VO|-&^+pBTL|r3kOo8oHr|7TaHLj{5+<==5MjH7F!bFjj_C+JiSc<B69$qn)j*c;emNd9K7L9<syxOgG=*K7B=BHpM&v@$p%4>@kV&a$&j~sJ{1?#?1Y>HvtlKlI={U{-`E0&XFg{MG6dW*C8Qw8j&~8SeN8Fsmi>seA<&=bRO4{>;v4C)K`8k*X?yHOe>5mgJBnjwIJy`k!4_58l1KT-h0;ran8&)W1VHmHzt1Wy0&NN9<$NC;$D#GCxRd72vI9$`h*Mrj;4NEC<{ju;Xe;0@_uZSEy%Srlz<}c}|2V%@*@j_2r<W1VOJ3Kfzbk+NT<xM@PW<KZ;#@av%KsgyEG{|x~Vl!m}jE@J(hj@^tXl0rkV;|$;VCyD&82x3j>{bj;*U#|mCQ2!pnKeu(A3W-Rd)osHKPgQ`J`YOC684$$U4t$E(Yw7SsZj6{8^gfJ%}<gvpnS|YHEW|A`0>cRYe~B+VvMOIQs%Di;DHl_B2AeLFnx7ArBglhQLDY1)N?A!oU8VgqIRp|UkzIEq@|KKcu*6OYWLZ2F(OxLihi^n)}+NE598%iof_L+O}odX6o=Xp@$E@vjOZYmDqF|r)!fWgqj{YXb?@YRFH~3|r4%Zx)MV{UpfR|Q?4mBqoG?x!_tqAI4!n>-PE#Ri5Q>NnPV$mkqrL-*<-u^22qOCr;T|O7JP5T$24z9Uy3Y=9l7WnIn^$}A;i&&myLn5v@4yhFhRjPohA|`PWHT6L_qUa=fBE<jc3*sCVvGBO{{4;qgJ(}XbjxEhhvVbJjR{K<kU6y!?47#fD5rEoGMbxs@Pqy~cAvx~gsEWgF*Y=N+-2{BvV{2#M97p{AG3h26;5<!4I3MPq5>^82uqA;=%qNB+a%pR<M@tuUv57Y`7S97k{ld11K(iqz<==2zu$u}X24^@8{>ak5}RfaFwSHUc$yiEnnNN0_u|SFcA(^RmeEks2qtuXF7gOSJ}y(rC4575`i3>})F_iu>J>$5c?V0a+LnD=FWjwgtwv=*gMy6EHk2>zXkv+IVB1!!SwN)Z0ZX%lq5!nt?6SMZCnx(yCno!)WQ_Zu7Oif*w;I{TT7|9Dr&E%+JwGfX;#w_Mc8G2r;_d0^iQ1HJa?!@?e{D}~Jai+;Fn${U{SC=7A5-cn<<fOju2*L0PE~T0<rH%t!@uBxY2fxSd$Upx$8k>74)Go|Ga_w8Xyuqw8n8-HL7VyG?-o}-)9EJH$W*m&RKQCKl3o9bt8@)VIa*x)Pif;vLqmXH|6=adeKC8s`$Ms~{BK0Eeh-uD--I#5VsZIKHgZ3hUjIT5GF)8!GkPKc`Qq}I5|ZnGEy0b%Iag`mJv$rx!TpW<{(~pzBvU9BSFd||u4A^i`YnOv`Z9ztVU$ZvTOz+dM^CB+@#X95&yl<T<WOAyf{%?E*Quh-;WGEL^See+1;Qif-ln|=eJkXe?Gg>_?z*`(P?uGG_wmcak6xfjZ8<%~LEnz$xMwZ1CChc1-GW5ijE{riT+(6()E^Hr65%F$2RCVI-=cI(x2eUli&?yYDJJsBJHnSGL6?ef0=jh>m!St^Rze^54cNp`1|ExAm2AXvRuTyY@vKoz6^tMx(APp9JCKqPBF1C28G5v}HS729_bZ>GBc#$~+{j?isrfPgQT&U~Ff4>Iqbc-ZiY;>@kvkB`xQPxpNFOG1h#4*6JA-$(X76rgGt~k@!l$G#BGB<H5j`J1R>4j^{j?;I2o8nN^e|A?tW9E+U<(j(5)wKvB@mM&DMQAIq>vN4mq3_NxlZhlVh1<U@`nPqk9wjz$OBAq{|AvE9HxXrj448lvNjB}D%z{=xMAvphJ_e}0@9L!q#T^Vdrr@X-CSg9jGCQ$_R6AMGEGj~ro`k$WhAOGCL;BMwC&7z$+b#aYp12s1_lqL2r@o)BO*zy%XB7pppbc|3LE&mQaJTBwnL@UA8_e9A$4~N2v>TCd*?_f)?p~*ifOnAZMV&@x7>|pIn78;BM%g;g1AXwo@wl(a8F&w;1G_7`)Hyj*gIQ#3f<N611tHwZgE-@syZ9VS6o;PEa#zPs_m;*{D+csA#+R48k%hBxW*%O{R0KwsDZf3A4@ilWisDDnPoK6{@Ie46-uj|VAW5bGa1)dy?;V+vqAFt9?NMcMLu_XfcBea_TD`m2Urcz&>X(=zb_bfWtw?V3C7R(*xv4_fkZ~RTXpxqIUhQT{;0k4_s?@CEyZr^3r=D**d(Djh;v?mjHyCR5eN0*m?v|@t~L?40+LfG$_%$qG=kw=!BsXf$}tzr{~9n`QnFFW6+#w>cGC#4EigDU;#)=&a8Rr=SQ)LxyD;^#uxV=*G=MfRK$-^{vO~cSYMI=yKUYDYV1mj%qXP@@t0=8Je+w~XY-+{^ekjP48q}^Z&)!0!DKrm6DYH_lsSISAHH}A8_6wOa%*@ybSiyK92@k2Ozx1>>qDes&Y+9QZC#i9E75r6DDGh*n+Sqn>f>S0J@ob>fgK8;OXX&7DHm2K-;>c|jIJ)=Gx5(r1)ro>X{v<d#{$T&;!zHZibtg!-q^Pm$CS62yoZeYuNlYEkFJiL&?n4(Ve|}V^S>aaV?fKe5++NR*=`3PnS}6RsUYeE{W|F2wwMZRa`_+Y~5bs-g9}SZY91-RlN(dWHheMjG)=Cr1^zfzaHli~RT)`qcBp^Iad709jNZRDMda=<;wadk>t!7FSMVW^PxL{2by<iVEVN^N7;FV(H!hc0)^1@lyU3)z3?NpF%;cM1BeXM}-$knsb-EF)^8D>)0aBKCF`Ml$)y;7%wMQ0c)+DXP5o~SC8LR~Y()uFR+%_t_knlaF_=q$8Bu~9Y27J+WYmuZHdm)u=b=yXqUpsc4fBJd!?<u}8ed8kX;$m`|Xbh2EaN;<tR6K=i2n1++eg@z=PWljU_2#r_75`ie?;Nez>0FfzcAuZn4$Z-7{-fA5ybz?FPigGk!GuP3<+Q<5#(ZIGoeZA$r&60mLs+yW#X1FbMUkMiN)HIi(3@j;LbZ2mtM_NhW5U0h&qxJ|ZWrz<+nk9^nUyz)nCYcCsO0pQwN~zq_E7Ytslg>^Zd%#J{L^A9_k^rsD*@0J%@}|y?uKD{qaT5L$T%6~}$X~CAXbYz3uJiKQ6Q_4^p#+feG0q-mV4Pv5_=G`=aTgsgdcZQLgDro{QxyP}uM~rI709ko^)?$(CI5ir<6;nD@sE#llIm>b{+0*l>RgTf{OucS9UU*F*a<6ee>0&xU{Nu6)!%;fXuI!$`n!G6^PrfpEGRM>GLlqNwD4watwo)>jVU*wL9HBat59OgjYLAZYdZC?Xn>z>o$0O>D5%pYB8`AR#6~K_OLJl&p?oZ3r>U{0ubhNPI&*dmpS~ISbnDDlDX7V)+M;umhC4gps2PJUwdNG-N|B?GlNEX+##s>uIVA<+djoI`PgPE4B6sytCa1ci(iuWG?}-e4vtnYaZ7PG)G}NQ7u(dJbvpVZC9wM8|oSct~b(#6?^XU#WSQ!(i=WjEp^_g0)flDQn<N-ptx-M9!!K-B?^;V(2K20+@uR5$O%21R%qLGn}PTwlB%^+j;1SsaU>uLHoVmX?H>06sgbDY1Ju~;i^M%2&W9$JLJQiKA+F^c$s+H3@Y(*EO89*+)H1yu26<!{yTj%G-E2M#EiB7{cG*f?krij2aLjzo^tn}iaq+QaTgdyj!6i3nAFpisal7D48CjAsdOid21LiLV8ecIbyyX<<>6V18=%-`s@l70KP87^gd$%vlfa-Pley+c{h7#2mFjGOkJe)qzU|ZB4{-AU8t5OBCg!f&wv$ArJu|BdLy?(r|*39+gQ_yLmDuxbH-Cn~?<Seb(yPht4e+w9Wu6AGFT?bbVot{g|xbmV~NPMxNRf+dbEvCt9`P0)szYyGcUfBN|8D&PlIqk8qt}VA2y#&vg>wfqQ>z3r`(w^$_2|=xkdkzUotqKK`fGrl$g#>uzs(u>Ee&gOM|Xd!V;Ya1Z(yjt3)_Bm-xdkZ^*Z1eYd>c-R_YSfZ!Eb5zTWsntY{>LcLc>ee1OFOHtIc{M_-c3iqCI`Jubsz#RFh5niDr(0(|^bGs6FexJnBbFC(qbw*)XwGR;m;k6&&;pdSe(GYDI%;O0leK45*If|{>EJ`lSTD9(IEFB|;_Ym8xRrrmqs7XykI|5~a!5j1lH|q!Vi@YJ5bxrrF|M@;H<^V~O~To!=i`B6NI5$dQ7gq^BXH4cYWC|GR>@nHH1W6{Ns{Hm4u~cP!sjLwqWZYT0V)FaW9=l>K#27v43Aj0$yv5r!DdxsT-nc9W*A1a!!Xr*d&GUBy{FFRB0&o^wNyO~1ul-HmCw|z8s}L?^?JQGsO9_B232EWVPaTfjfNGv1Y2g>k;6IBJmD#9eO|b|@XKSFnBLzu1z7KPSNE&yj+gfBY)v2CqSBqvxo6INMRm$HQj;)G8);U3Ov%E>tW&<t(rAgpHcnY89h%U23sJ$Qt%YcIOU0C8fe3wEv%6tQkyx2vR+<Sat?inpd7`ryY-HD9*H3PR!_x_!BYdD}e7%{dA!C+MWx<VJm`&NoQ9E#l<|vuS5HxfXQb>*bSlM+>LovqfMWBF14KI4?Xo>2rH=Y%WeWT~pSWPg_MVS>|mDqZK!HDs3g&74CI$uJo9-N*nA=VlbsnJiBW+6}3G$(Wp18^Np&r$9CM4TflUK=G=NT6RK!Ip>{?D_?j5gm~-F-mfJ)@ve{OFXu><W)6Q6jYFEP}K6Y9hnj+O0R8lwX~b2pxd{dk``+dRC?vG2zT7bXq~$1?iUu&WUwa1V1+cT!C%dyRWeXrFf(Xn9Lp6wt(u%}ouTqis5#{I4(zJ)$}z)amPbHmc%>pLvxJ2NE3=&9u?>p4WLIc6Rryt=zy7Vus)lb>urcp77~pDsvldUuM7vHz*fmzSx@xLv&c=+BB(TNUSng9>6bK)lb+6m0MGD1Kt(-l;O`DH~Al76*cAoKRNlU6zk2PkiJzITs1LbDKtOaLXQfkP>Q10Tu8q`wgIuhN)4ZgHQt-V2G#VW|qFxjzUwvRxpc*Q3mAlc;-zOs?tWT^PELQh*iJkx7E;lH5r{cn`exRF~`Y;NmJ6(Vf!`3r?G3;@AV?QCjO?7ih2zVa!C!ma8sN)b`L2MrW`>A?DDRq>^E?S}r@*DeKEgRVmts_e}I7^-1!bk#O#t7WY|qftTSa>u;oshq-)URS#9DjkM#$?@)%>3XUu@0bDZbrWiFw0>Et1-`{eyY@>rT*3XCh|i7dtESdo-7X>fbSBm2+8%OU`KjATR!ADtSgJ+Wu2^xXx!3>q><Iq#DqihL-Ci*jH`e(0g|>Ai(q5JRRXaSRGG0AoM+LiM!`Z>NZgpxnMekLiOx3HM@3(raceu@poSMj2_35`B>;KldfAgEVu?-$5mFo=Wt;pSgN?$o<SZ4=LYYuSam9$g&^Y*14$3Pycj98~N&dxH$@xU?pgtJp+7B|%095WFI&%CO4c<WL~1En-4w_$RniwkQ<RC2t5^|r%s&9rHCUwq^FI%@;gJlwm}M0D%hs#W>qE902A!Y>tk*}n~Fc>`Ga7Lc+&Z`R&%p1O{ffrCg&3$pY;WpF*)lFT0VwAMUyhz+o!1EX3gb-vV>Ash7@<YS}EU_=;V9pJz{<@{DX6M9E1SE*z1`fFVSQy#*;WUP^$`dWZ4IGmH5x=t*mx)nf*IS)Y{e?{OZAQ?kzcg@jz%oLL(;MRd23qtKY-Sy&m@WKA~kjqN^2yaxKblpbL_RBm`ha%Nwvdx&1M8@{n^BM}RI^eNcnyW!w=W!`xk(-Nl_Hfo>;IB3Kh{&Neo20t}{G2D0<TO`;uED>Rjgzt``zH(BSIcT~$%$ctM$QGEjUlqcgHRD2JlN_t3p?bTFu4?GCE=d6Kx$r<Ak38w{i(Lj+V$zNkPCtK;+PAGf<zQu?(#k<%0b=dYdODKQLuIW!2?H?WliX99_MROP<e>6qtr}YNS|40ld}W&w-sota%LQ)I$GEw)0K49iO9{T`kW@1sRwBkin`mk;7hAtyD;^({>u=u7Ey20&@p^C&qz@OTBUIrKT!vo6<*<Vcd3td;%L@6&kIY1GEtxoMRZt>@fds4#_Pu}9@^TmezgM&XDp$v@B1|X?l{gKcG#(ZA*MRwZVtCkNqsFK#%mOK)*qioqPmk8#&{iY_oe=^8YH@>4)9Z$;B5))t!^5&&ffR%x<F3!<$?OrMJzH%C?HAe;sRc)z;^1s#%3beCZ4wEvFJfc%*}*Q-B=Ljwjmy%mvN~mw=WJL)CL1%tm?2Op7pZs(4aY~s{w7N-uOWjx9JMD@M@$aEEyQrt<TI=4i!U_{^bz0E<XN;>x(sy+N~k>P%*ahODJwrfXySEYYu2?h~=i^i^kd1u1`TRTn^L-DUBEj)Oikl25Zx(TU!@Jm^ZZ$Gn||B&x09eK|10pKLi8wyECpMxKk_W|KG*xIu&xm>b3F`57yn#Xo#l5omKo4qfy5Rk4EIGHVAg^Qcvq`d$)JsU~%;mhQ_>Exr2@OV7j>cHC|7_A4b^T+Jfog>Sy?a4wB9wUtIkJHO2h;lDamB7w?uoZqvA}5aNBN_>aE>$6N^1*Z%N^-RGeLF8=rek70QIM&Nz0#no@kwU6=Q>UG$|J9~4OUjMsU;&->UAd9cx=pB0F_CZ%rzj^oBt=5}50=AXgZ2D0#iX8<joh}vqtf}cNDk=@f-&gu-`rp(tR2@9l2LtuPR6dSaxGXA~YrL!}eqF0zTm?Dg-AeIX^=m52dkPgho{~Ua0Pe$gbZJ<Lr-z^JTjy)=C31ZL#}KHC$)<;?5x0h$nHv#in3>uGs>2#Joz77TDwiWI&tBC}zP6B}qW*=0lCw`ss=-24#qIn4{cYt}mJ&q6M;J|`lq)_>Ie>Le$2!rA-&DxQatXBPnpYZb?TpNyw|JgE15ir?1QY-O00;m80C8O|3g;q>2mk=T6#xJ*0001QaCB*HX?kU3FJo_PVPaozV{%_}WpZ|9axZjpVQzC~Z*pyAa&uo|VRU0?E^v8;SZ#CLHWL1>U$Mt82ii1ixy?&GVdt7$ualW;ok`Pv(P%gjv6OHo0R{kV%c=kS%>v+yWXJIbO8|Bk&n|Ws50oUyRp+Euyn$NYp*S;Wb<@)nY`Cj-Xra~y+zz3WN}+&%BX)lvUY}jP`*?N%trpnii^T_n7R?P>FRSZXR+2ZMv^uMd?w~We?OX=VaAj+4THs2+)*y;K2&}Y_YMU=U?eH;@8)@mI<JNkuK^(Zj2vg+n>xVzV?TqfXyRLU6xX`_$uN^ni?V;_hgAGE9-1Y_qY_Qe_0TtggWX|oR^$z8WBuN$v*>>7Eka|%Y-9n|6!#mf=4aAi=({jbPXo8g6cW4KJ-_Y%}__XgZEPi<Z-(l%}%eR<?kE_lu7K?u!nG;wK-jyK)%tt%DTmS$hN%FJcoeQv~vW}|?{c&l9pmYfQ78M%q5Gvj@85p!q8}e2<;LR=H+nk&wixHXArrIt1OpJU1TXftx698n8_|GJR-?hRtDA$9uw0;^WZ}uIsFrDUQsd$TJ3I8Y{p)|6H*H`a8Ui{{b1+VCDFR>Bh$e`<u@<VOpR#LJSaj(4A=9Zfv+;Ve`1`5!d|6%1=z-9Rjib<g?g{)jz=38_uiRwufm4nKLTP))e6kz)f4V$}?MQiD}zt^J!ixZ3vC4&=gw)AmweM{fs$@y>zn}MXmw@^S>4w$6sdRIzelQcVA@X|@Z7(-606lf{zJnmB1-0paXtK}Ms0+uwXv>SHR1{zc>j8ZQ#L7Rd;2Amx`G74k6K|wB%!-TU8`29Q;sbwjXVx}?J^h%U!dBq)Dc`O8%64(zws)%=;fsk!+J{hTzyWZfqac4B0xi&uV`R7|y7o|_qFkyKchvW8XI+?TBal@Ua?D6t95^qb39hHtYxj`RIw#;Ce!Pzp+@vh@aumMMm_W61ccoOO4?J%27TSq5nsx7zIV{)wG`Ii<ym$Z_s=ZWwb36)tmooPy&5l@^vTRs*%TTZ5@Qv<H{tkS)5q;O;=o?BR!%0tg>-?HFR?sOxqV`-Wm<r$Y2;V<6w_=_>xu;j1MM8>cQ?Gv}9MhM8OUC4=lT5$Rr6?EE4x+Ug?5xpT0C`wy#=g^eQgX)$2+#}v2qhoZM3A~XN7EvdR(rj{&M5y>j2^!3D$9IcxMo_?)YusPL4LRf^a*g{e4y78(SvqWO>`zbp$(qzcy(;)J6n+MeQeZaKaVL3GQqNBw9-cy%_nqJl+1OwTY8M&!u?-b#qn8u5*a_x6k)52(%%=0JVrYm^%<euGC&#ww6`a{W)$(g9&SthG>)6BTa`s*NFogs*Pu&JS4;qVy!jDjT(~E__Y(1~QGOz~AjJ~y5$TE44CT1?`P|9&OM{P?#DrMMM)?Oxd<7xZHss3p&5~zv5S}KHJuKtL%)OHOzJc}%*+F^A)WI63hJ->i2sO<x7D(%1l7;3s)luLaCz@nq6plQ(djU&xfsB+Km2nmr0Ag3qHs3=AX?;K-N3A_v4nZCuykPfQ8^&Eo*J1N1LRs+;@P8NnZDe*W?Gmts&I#hyvNgD3(F1dt+QXVOMGzfb%vXu@D^#&BlL;B3|dou^e7tTBaUtOXS>om8{$nK~<zY58J&QT@gTG+{lKf$(hd$@Y{F&Y^##?p$GV-iYLqbWV_8`#!}HiJ(2Yd9PEv@ieC%6rVfcP64TccsGHkd?N0z5MYa=I&#HCu`qk$m}dr1Fo1Klv?8^V^ezd@^uEw^NSxI=xPXPa^L=0^2W>ml%#Y~ET5}@x4x?z{M<|G^~7Gn)lW2^6Z0a#D8icnrZ+(>gB>;<8cOEy4)MXu%d+lORhD^KZX3PfjV((rUTJlM#&Z9vC}8y^F}guIvp0YLY0>=s6Q!4@;mReK(XEH<$woa_{@Gedwdam4B+0{AWi4GpR8JB0is)y{UD=|wzQeEC9HA}dqYP~Ar~|j$6-n1{sgexr-a2f{0JKPASQ@Nd21ago4vJ~(+;)wepHFBdiYj0Ziyk3mn2sc)GgwP1oG{B~85#E4Xj}Y;Hx@Ih_wY6H@4o(;st5R-ZpgdXoC;ViFJ8Z1zF5NvFgDA{@4kbVKcsMS0vB)!kwAs+NTla<fPZ>u@HApu%(&Ld@9uY4!l@_8K{-v~6jq;1?-xh%9Bui<QI%GM>EWgu-n_FQzvWg|{`L21SWecgMXm~Vp1;Uo_?Kp2op?N{=vW(hj|HKMJE5}L95wfj9((Gj#&4zDg~w;EJMu>7Z}$%E`zzL|ZJGkNV0CY*qYfI%vxsYX$WX_UkH5a15wOR7<4pmtUcNrGHMi$KiD%1uoFDGI!_7$7Ty=Ot=N%V8GlQ>L$e_}VHU(Q=yvktscVsw5acK49@dcw%I1t6*b`0%@n0$*4jb&*XspOQ%(-XitUR|@e&76dWkJ!nJz%)Yv#vfw_3-@TWW%M5T{_qYs@bz0(M<zJh4Vc(U3Oi&n<xM&$2dPG@?UD6h&}UVU@vRtDh5p^e(k#yN-sy^4$C9cSJUK*_J@<nL{SCinP#X;~A7<ELZVaAfK&?+v2lbRWdY~2BvXR#_Hk=-|`EX>#ip9_mul1E@KbL_Gox*aRPMpXSD6Y21#=xcPr_7EFzBmj!O+6F$jn>U^+@Cf-dkxGyW)rRqE}l5lBWpsJmJ<sd`m@8<KZ$#T#d&`28}`V4i>FEPp~;SmZz>!!n8V?rJ*7*79dcKyt>LZD{Nno;8T1y*iYTFPkCRfQUiy}jeoUv;CEaAo|59<e&ILMN?O2*uUC*Z1>%fOdk}P<FGlujzqT_$!io^<u+6Y<u`fwrR$;#I4VhfD~>>KZq#s2|NO9KQH000080001SU0IqeD3=TX0FxmA04D$d0B>+~X>DnGWn?dFVP|J-WnXV&axZRgbY)_1Z);z5aBOd3WG--dwOL<}+r|-pzfUpU)Ht%Hs5{$D4cR;(b`r!$TpNj#Cb?XN6}b}EM=t5zr6pNV(I5|fC;}u8MNt$5>Kbj476k$%Kv5rxKAfBHp?;ELc9#_O=X`d7@<q~ecV>3}{$@l7dB4mGngA6El(LAKQXtqE*o5W^xBx1E3(X{rHGjz9Z8{od3=~s}7a}Bt?C`uSq=o@i?A0qfLs{gIP|Y;Y8Mt@#?|N^l^Q`{Pi~8GrRS0{&q&m&`pgw+=>a^Y@8zL&LG|%jTHnWn+VLvV^l-j<nDxTDD@%xpU#3Nn~dB%2})I~h$vx&yL*4xLDYt}C+T~=CcZVLxg$7#Je&KMPa^WLm)*`HIL7O|{P+`0Mg&5SdlcXoDezxHAO{k!kndF}Q7jgRiV@g79*!#(%`?C)K=;y!kEb`my(%Gj*WHOrOvkY*M0dpiICgpk+bn3WoI%HU0&F=t-02~<i;rl3+Rfx#4bIUVz4#I%AR83(v(?1_Y!(2_F+kv$n&RzC0{aK(kvRK(1(4j|#N_JL)Jf#?0W5G;U)J`4*957C3HfQBrt4QZAQXnY(162%RUnY{u&g60e&^FACg?GaocJ`}RGmNQKgswoLzHuu-})m$2yd<de@hV+sVK9X|9-~$x$4Jixhk+;*+Z^Z~Er;lTJ_T}Q_mm-B{Uw-%X;^avT`t&aX&}l+q?}X2kvn&CnrHAzW00vXdl!w9|lDH6>2~CcCXTA~DDU;|YRnCM~p0%f18u4956L|g3%?}MH1!Jz$qSD~Jh>H<WIpA7SIRz@?6hjk(Mu9FM;p(`tWWxm_I6Sh(os?80Ea@95A%eZ7ad^h?3IT>JMdLWxHjGp63?(Zw8Z(c4@(Brmbcr8IRc2g!q)P(W_YXV!M{P4?On6A$hXYvm`EyJ&Bs1hW?-};+(AY%7gX;7##>(v*ZKI&YQNXMiGwJyu6Ny4adL+b<Bu?G+6H}3JOjJfw-ck$xQhJs>hv*|-uG7Rfb&?4`<|Rg?$BXa=D*o0Ts|Fvas*SjM9G{%!11613xiOIm8Mi&;VpwEcX&aj8;CaYZKjG3_Dq_on(VmBm4bN23c{D7?m-B^CX)3kj^Rw@hmVR5NTXWKjpsL)nton*SVoO~(4V~{!XSx-+w>bHg7}<!^vJfN47bm~bkU0l?zN4MtK`*P9ps}iCH1TrZ-?(Gj6BT>mV2RVk$>%m}fiWzk#p3DTV?duZZ@;4E^6gt8?>c$(Z8W${h5{|`x+=xG*&-9}?%=A*HjVru2K;{*@4PUr#W#^!>ts_j7lAW!c+(!}-<y_99fgxTL$i+ynRM&pW^o>JrK*8q+H2NP)i$KplF9Q7y<(_&4~}gKE7Qf(C)^tTCxvu1C0(36DIr^&{9;`ZO|b}RSu&A$-gXc0?J;-Z=G0Q&O)K3@zx$d(eEN-*re&=d>#SLP_T}jl=d874lIco{B@0(V9aL@A(B@kJ<3b+yWl`t=m`KWk3mxEzkz`6m2vD46aSucWUV6<T%lCdUFjs~MNWwZg<sAYS8z38sV(j^$(sGDDJ#yir3;BhlbK$KEw=cZcAwFERNYnGfluZtMuiDVPUko6!e7itgZ6(lc_*uOQM4J$q?Oh<|w=L(AsVZx~J7&7`prWL8FwBp!0F+dsKB9ZFhF|Wmd+*kYu?j#Hm5goPItpr>fEj>`giQkQR?TlwTa`0OHEXMWzvZ(E;avohg{m5Ok{tf@lX2KNx`?6BA%Ru9!Al`-HQMB*M?&Hs_V)Moju6L@gxRa%lhFao5POiBk*l|NDVcX>czV9vW$%Lc(W<cvo;`RGHp-ga8(NfaLEA>ILJigNFeB)wq-Q`t0K_&1q-RB&uM=+0*`M~c$u<q}_i7i7mcAO(IAwjra}l~qmzV|8B}FVl^&sP#T_(Y{<Y3wkQy-(tms(C=e=!*>HjI~sa;TCefjgfwEqRRIu-?(Qi>Sa6S;Bf*jDsdj(LWyX(zDLh&|h&5Dm6<WT4~o>phe5+YWm~DhJ45{r@TOISUs?s>x`Wz82`;Kx3pRt+6~~6U{2+6Rl10n=)iz~7hhdjj1OzqIC8nN<xOE#Vjl67rn*uQ5sW3OFWE!R#su%_h=xTh`?yKj?q*1NR!OFaYqm&nyxj}%DPkFI&<S8nrQl+uns@cWQc5;49poG4gBJo_hE3w$dDk#HCIOhw2=&1xaqIR*D=ed>Vo6_-BZy$B&oi#vlpjR`*fSt^{RnVnsu1RNtCu~@zN>Bpo!aA!IE|%<@#>Cq_z))$#CPcciF>#bCRJW4ZzVRcw5Uq5K2<U2(VH|=EHK$p=;)G<uj3-&Vib`|4?9;~&RH^6Ri0BheLhPUKmQA-SKyAwrcJ$|lGX4s35>pM@nH!Rl8?BcnZkUFa|4x9$r4OaHWP)0VBKcAdTFOGVNQ=(-@Woo3}C{ltC=4F!L}%<<?vUkx;qz`6zeLxN{eyd4LB<9ZK>rhi`3(z<q(5nqinv~tc1gtTX&~AMSXHrW^`Jgdh6cpcW`iF7I(~crw<8HO(k3oP&{hJDv>Q9f$8-E$g52AGGnI129PinOKxH&A~#!@8;1y=I=c)itqaj&a>FcmD~F$bu{inM4j(=xfjiSzdYZ9_s47P6LyY`%kg=py;uXsnrrH(*ysKBsV*#K={~zDF+Xnxj7bky({Pgh@;>F3IDGU}*A2&d(t{^SKw}igyR_!P#+6y4(!hFz4w7;g3J2&6$IchdJwrJfk3!ZOnVI0BSgcqV8r&Nqs;@Qtu5kVI-n8hG9tU|Ud_m_hUHzc!14~bx7a@1-#YjuToKo$Xc_C<q1dw`G+;YxVF;KIYHWyn-aOXdmNoZ)cPieVdYBaltb8x9aIA&NC<SlbVxW;Ugk3b5(5Un|~A%w$a$nr2vTsICV_ZS#aVoj`=@_x9ape{)lPTO_&pwq-O6ws70LJ2TYgWGNx`sr@+P@i9xheoLA<mfAPmyfb=jbHsW?h=4=7a_fyL*Dc@PdWF5sR|ASQWSSX!;)H~pjVcTnv+OqU1g5cbflB?f2W@9e_?$B~+Od<-O;V-gLVH8<><cPJ#(6%EVP@uLHAR1Jo$5b7{}Tydm{lrW(^UPeBw4)1OSevIj=0HUYya%{8pLU{fBz9Lx3{$Gu+x-4(;IoqqjoH^wPK&GLQ}*RPoF?``fU$pD~t@p*fn0g8|?XWSUi0q(qQYZO7I62v)hhS>{vxgs{od+wfYRnHYy2Asxsp-)hubT1T9bhAx7Id+h!o;TG<mOOPj@)<(_I?_9c5zF;x?Dzs_+Xhwm}hp%ZPlp~*<5xgA^VR>X(~XRW5F{Ul6Np0)Q|{Aj25WVQr>Z41R_tf8@J#R0T8)!^9f%N8fU=7?anYpC{)8Di31blcsdc@Jh*sQEeSF!DfTCX$=CwLi@si&@vSx6M~yAa|dc0Nba}-o262Z8Bdb^)pYdWK-8Im@b8isV5JKh5zITz89H}#NM~Lt*eGLXxZRn(5o#F<`2Yh@ah%2R%Z|s_pUkj{V-wXB{6V#%M`Y@ts4uXul$a~h*JTR#mV0wJN+kkE~lEA*hPbMFM?h?eOw?Y@_R5_P4=BuCFq5hhI1J4(?3kl@e>CxJp;goyVuKu>lBh=EHJZ0WLDqL%<KrHk_{t*e+l$xX|n_`ME4OduUXXuGpDRMf!*g#<^gscR&*ltxX8F*ok3Q_$Ja`lV877Y-zz7uf3=)k!+**~vZzGTdu9LX)k{|f*Ww~8WN(~u&8`g!nK0RrG~t!%y;@GLVWZMPOGqPRPZWZYgY$&CXQq+Ii>Lp>X)I>tk8j<*-lYfEyXC?4^5C|KT2t4|q|i3mp~n6?r-Bce(kA1>!_(h`PQUvaN4<ZHLGgseREF>&%c0#_8DgKoF<+c~mV=hmW`+<M?w(mT2o^Ai|NWJjPJZL&loq1)$Z}3^TAY{bo41Emg6LF|7e^*ePCHa}&fNS(X%vWZV*kCI=VOnTt-+sdK+SZRE4#MC)j?5Y>sV&Bl?sc2ViFs}bu1rPup1F`Y^5~{aTj7ruMIEljyj_1uHl0l#l$;YZFH`umk-+g6S~N6`Sosk`GELAgZR%JoEX(Rf`)v3o4OA>@|T0Pz*(rC^0>x!=0>|rl3t^Y0Ei_(dX7?ifM~zWcm4}dO9KQH000080001SU0E;>^LPLN0G9v&044wc0B>+~X>DnGWn?dNa$_%KZ)130Wq5RQVPkYJUtei%X>?y-E^v8O&P@u!Fc5~}{hZ=+RxPw1K+v@-FCY?PI)T9ChjfZPzH}w-A70+rYsT5HtK_!|Qf{>yf{HJ0sbQb+>B+>^&Dv%wk#k<vIb8ANAL9x92l+)ri$r9f$}nQS;ZciR2mo47#Rk5MYDGK_j>aszbsy}+BYgo-O9KQH000080001SUBC&aBohJv067Ez044wc0B>+~X>DnGWn?dNa$_%KZ)130Wq5RQVPkYJUteuuX>MO%E^v8uQ@zfcFcjT+iYpdCB79ZHNDNhmcB%3y%2q{|YYbQgo7&e+J|#?@JM;ni5M7(0OJ5{!QrSTMl#c~2_ndprJ=Yd(r!)f9RY$eqL8(;>88wEtpBoNhNVA4hNeabwDQvYsPL##Yq8afm#ab~v!Iz8Gq7htT5EQ(GmI@gs^8f&Vc95UDW$$!lTQ2e0MLO=Zs`AjaA$*6LG7OWEmC>RgbcDwq^IC4E!W6>UOj!(5D0oS2gL&w}k)bQ&RCJkF;ix7aQckW@p}8?}p(IZM71AaJgA~m;gnK$IB*ta9T&(7>chLT6@3B1`fKgWSIqW^_Q{+QZ3pCNjr8WO*xj}P5)QBCKen*==OrdP7sq?kAe9*QPSYND-Cz<*5kpMNoeDWf*rTUBQQee1){A!P}8#)^{s+cANFCSiGzw1TESA|mVV5aN{oUC|X3)oMo{%K%jxI;LAJ5he#hRc}YE_2n9%f#8SrCJJE-K57~Ngv)eS=XoIF9DUHat?b3b@E3S3&s2)AST+(v$th$;NUE~M`jhrIP%*fCQ9K=GNA8zEf=fvCLi$GWvi})o8-fK{d~RM2dvvod*WT&j}+_mOUL2g&(rY-1JtMEZw4q%fBtrli%N>(8z;sx#?3_ztL@9__+7s4feXD4XC;9sfe=YsP6*^VgoL;YOh`DKs^K;V{{c`-0|XQR000O80041a1=J}(WCH*I)CB+lBme*aZ*X*JZE1RCWG{1aV=rWHV|ZU>cyw}MV{|WJZgY2KaxQRrbyPuX+(sC^>sLHxdyv+)LfhMd3wCR4(|FS+4it8`i*__C2}?6#G}_ob1#$|Zl!YE@dT|omlBB^6_9YU*2buhj=`SfWD?1LxCrLA(-uLFc_f3-JLMlk+IOCnQ``>Te$0{WAsmSsqWv(PYyThXExr4Qp??waP8+rEo3*tFW$fih&vB*^-_&5vX9+R$O`>KaRNe?UEU?N1?a{xdxRZ4Cb+_sgU1&YS@4+sfDJkN1DSo=k&z#t?!CL#^77wh`B1Zgl~DGFWBL7NvPT=K1XeAy$*SiFAa6A!cIs4Z<ifC}z(;F7>2zE47rcnF*Kpm9_22p98@|LXcBg=)@lG1t}G4DO@e#DkK8M+gPo$A6E1M}O<%Nq-YFUA^Im5?!BIW=;n*79jn&InA)`8zMh!>FTdOwte4z4mNj3mS3#vUpapAiI1%yfn*WeM`-&0nc!mHyf}cB!|7o6Uh}w*bzT1fuIs}DVaX#DEXo*HaQEWoj7OlcuAd^8Y?|zwQ~!gBuHLm9uXqF%>-rsq($)FmJWdDGBxT5S^`^AURjjMygI2DzY|c2+=IE-`#a!2i6sfM>=5VP9QdUrgmO!NI!yGDZUT03{I#cV;!n0X(BtV<@Ji=7SEKr!)1KW<C)wV^mhi|1McUl>V?~H$Zxba|j`(A&1cWr&+?u~d3i>~p0SS$_0rD4$>_`T7v=$e3zA3R*&-rd-_{m((L5{`ymTYnkoJdi;ogFGG=Dv)Yy$=(`lg;jit@Xf4$&i^v=o+<yF;MG>kL^2*Sd1J1YG0l|aP$;>e(7}!^0&PyM?N2b#^|RKYE91P@iM?3p>ZM7D)YXYi3KJ-T5?oj3Wjh@$GL4O;CJE1Yj8Iq4c?5SBf~@-lOCTUEtesUut4$AC_dz@#n`T~GG#-@)&1-&>!o|F7&g>wZ+Egsk?b}<x^-$>Q19++HxiaxRR))!O3EDiTF{rNo=4}TRC%QUUP}0s8y&lXxI|^pssA6Mh)Otnln06<ZTSgAgw|E&=aqDemE0QU=P{<ZeTyx(P+O=`Yf^ZpBNVW_mgCxZ&GCz`{%nNscQW~q%gfpxfdk0^St_R6=N!Ms?AUxCYQ#nfu)~+utz@7gAP)h>@6aWAK2mk;8ab4UVB_N;%006-b001Tc003`rbZKpAdSzrUb8=%ZWN%}5UuAf7a$#e1FJW_YWo=??WiD`erB_jl+(r_9*RLpgUZg{-<$N#4z#N1n7n|cY#`w)iL{F<{)LyAu>2B@L-sK>{566)lIYS^A6SC_iF}|Ehd>aUi9{ixi|3mpDM|DdxvpZgsBlEIScXieGRex2b6=kKgh5JUyK=@~6h9}K~nN}rOQF7?lDz0ZKQ?;~Q``u*NlBQoM6wSCEmhr9MU`krPu*1%tf9*GQEzPi_xy-muE2`;S(<<*xES(nos*zjl4Hji_M2myLLRhZTX`$GG34$QwGg!M!F?3i(u=58nR*wS!0L^W!B@mL_S3-t1ad4FeLSUveSP&Ab1pS86Bp-(4h{QlFiJ~Y7f)QN&t7|?1n+ve{s~@`ND>;X<Yrckyhu!jN4byJ<!Y@6tU^ufv$(@|jOvpK8-SSk<f!w~?Ex+A|TL~mdg8ujO_Ph*6a0}ldSKaa_23B{?3kFtn%a;|9z4l2Bza6_H3Eb5f{V{{Vw`@<2;~{R>JcOAjIF#M;yBb(~9*lssrwlG0Dg*(f<d$aCQUbDT9toIs%_9g8Gax%VJNtKUBdon#!4$(=A2x!6y!}zmVNrL>Z!F}ZYyK{S5m>F{94ab#0dM_9SqeXY`~~daeD(eQJ3cOR=WO0J&)9CaJd-)dZuz3@k6yxd@vuFI!*2N<J|fLC7L0nowtX&hIBZXpM~rEISwdA%#&cDmEUju!B(SdehC$fh2gAT}K888kNW7-+?QmyZ^K~$SlG@xJS3Lb)3>IM`jx!)uwEwON-0)dtxx)3OoWqSvq%P69vEqW-K-u_6cu7$Lcag`b0NX8373A#+QWB2$xF<gEntzI|GTGfM3B(eN;Llz2*{zM-b<KlIsj(Li<h;imgUnz3(C6bHX!bc|?SJJQG=+n_Yd#lHQhiWYj;QUL#{v#=5sYBoEq|(jN|r0#OD3y5`{nahDlu6SLND-LC+l4lZj=NgS3T{MV<_5_UZ}2LNV_wohG&rTS_@+Zs^#24e2TcAvmB9r3UlG`<c?^Ce@ZwMW6b=^IvBy9Znj_SzvG23Q6c^^ydGRirYbvjJRh&9gdPdz?GqPwD5!%RHJNwIa~TJtp(y(T*f5M@uylw$=oPBGeO5vHq;v^-io)oYXK;v7`V8x?`BKiYQi74Ab0&tu+JL-~YNto90IUFDzK*8tsfzoHFS_PS;p*~%bi(`;Y)+5iu>DUk^0kbb#26t{(pXJ}v_N`VQ>t8+zXGPI$zd0Yy8TfOuvBfU+B))bo?kqpBuawxt^K{5@7_u8ybTl3JYlM=M8QK%KInDuk<-CPP6zKzNz}J$zOb4yn`VkF!iw5_jD0$W-^3uP6^A^<lAP=-$s^yr2_bL$LxXaF$_$t)3eGINyLSiftEqt_VRMB*m9&&uyiE$F%6O3wLYymOskS_W$)$l8Ms71x3?^=_Ib42`*#%67krrxRm`SEsf)6oJW4QGT{xOD(&$vz%)04Zp=9f8bJbUM~DAGbmZeRlU-uFvRpQ*?&EGEIN;*eswC^8&W8{;p;&?xUdRrnJ#Z<<&-hsgvs23kcn?0VacfWo+lF1zddO-gE69$p=>E=H6Xt3?$?SNv`<DYy(*Y(4qm=cVl~<!a;eL1Mgw%V$=u5?WPUX5rvuZN`f?B*IRMtEt$A;o9x5fiQxjF@6l()<1vzt#7*@)-<>_@oPZLE|MdvrI7QL|4Y${RIt@<hvqg@wanZs=B2Q%5N35zxIXCX0Do(og9KNU+zv0y2BIySdu#k-FWtVuO@O>7`!B7H>#nFaJT;stnVGEugq@)_hpI~y)2Xr4ntP-_OWwzGn|?OG?tFN6Q+=yp1*h_LjRsw_LU3s}`^+1g!-jX5Y&WsjbG(kmilgR66^9)AMYskP{r@?*Vw@>fmt5Mx{!O{QHho80Sf2XsZ$iJ{=aAXOcxZ|-4u~`p+U>ay@3okfNlJRaGoek`o5jEv!q`+DxEATj8j*3BpgCsKi)Up|aF<a{qAj1e^IC^E3_w*}hS&J?d=k5`B#Obc1umJw**+n)o$dUdL_iJ9)-^CoTuS;>M61p`lQG~&H8y8gxmsAuCH_BAnsB4fTm#CJm`R6Yx)pXaG=}-b1f`g9V-x%@@E3DL<0PxgD*P+)GmVTLslgrk={r-uGrYz?6x03UN{oMtVHR%<2LLWjt$9if6Jqj5S{NRKkQvtGjVSmZP)h>@6aWAK2mk;8aa{slB#~7E005{6001KZ003`rbZKpAdSzrUb8=%ZWN%}5UuAf7a$#e1FJfVIV`wgLd6iX5j@&j7-Rl$ut|VI#kO0X-fB++tWSIcj4h907EVbxRq(YVq*&Q7wi(FzhIm_H6AoXv@hE1S9B&%Lkz50+b_F%O}LBO>JX~oc?A$V&q9xm8ehw$6O@8B&ciH4Nm8dS%x=)t$#At=+JLnE+Yj3r6qYzLAP7n*yI9#q#`NBdE-9JJiGT3u!ZQXQ_6J-VhAb})o)!7Bn(chQ>+j^wzYT3RtKBElNNkyeR^d7C83o85H$h95bJ_NOJ%G64YSxcg)1OYr2N0><Luf>^Ft@E81Q4MG)vq6j)?-9ar$BsL&4aua>4FTqyYb83AKLMt?s%_co@M<}5>1)OH_a?OL!V<vLAIy>|pDtLU#mroRItOI2vjyVJ)pbVfHI&_?HHoKS$i`@Htvxl97pb@GQsZlov!%4lvy5rPVzi{nwc5hvqn+5NCG%`I=n;zh^vtq~lUYYA0(CH=wWR=Wg!Q{cG9O@93ps@%xl)2VcxN{(3IqWQ9tpcSDleb6<yoPV5y1rQ@r#Y*~6}VkpPdEL8?K`)Ru5WtL+b>X_Q@E<FFuM$n0(*j=<BzfixDRLL*DxOIvGU|n97Xn72Ogj1+nzg*wg0FN;hVKYqnH)0{%pyfip!xu^)8CeMPE1+_G;7zOati|sAirqs6E1MO1_AJ18m4>2{)!W;mFF^k~y$Q04t$kVQNwP^863M<>mQrRl*HNF|lbcv3e=3;K;N``F~U#!r%IC%|*VKvoi2O_35E~XVMQ%sG3c2pIJWoe@=C{RA!h5O+1KQdat})Yb&^}9r{6&-xyjcE3Qb#68|J~k+?XvfLitP{O<rtvgzYS*w)790e+=+Wo9}?5lT?M4_&;vo1UD{PQnZI<cpz~VXU1z3Abd4>q?{9yDC2k4MzPrV3fF}ailD-PvKks-S<DFp=Yx~sw;X*vn+-bLO68uDVjTcQe7BJUf3Qh7T-lVkA8L*T$5M(>J-`|TiTF4!Rxu$cEv3Ib4Y_T$Cu&u>XlT2(xt;9P?p06Rj$3A1B;NXyobJ9EHXK(eqtEpz&00H34zjWf5*xZxZz5p3{T6Z>W419SgTV-*cGw-X5zy>u)X6>$$wBw0|XQR000O80041aE3cbny$JvSlOF&8B>(^bZ*X*JZE1RCWG{1aV=rWHV|ZU>cyw}MV{|WLZ*FvDcyumsdA%9mYa7Y=-G9Y*n}e)4d6R3ePy{+II8HCrj!Vw;0>X&V?yPo_-5F(eWXp98g?lJ{Dxn9Z+(UDO<49l9l0I1v2X)&2L;Xw2_s#4tX{9Jfs0p?-^Ue4BcSJT(+Q3BRVm4tW*%9_Y<vX!f85ogqaAy;)<6)?#(r|5WBW75bvOMQ`0aPA!Ar?H1EMPjz(gK1T&AB}eiZhun9c0WTDKmW5a;`$%aw?e-r+jB;=d+T99r_J>N*X>hy&V7m&C|RGkq8ZaMcz&Qz{%GGAr185HXWX6VGNfR4Ek0wKjJZ)re+wk&?tT0$IA|a!wrZ<$|cJ<?0yQPbHj7OlQAh`o|_(|B43XPFFTen(^ANBzV!pr**oy5M1(9Juw(uR{G-Fe1NaDz_8yW-&H02|v?|oYSoT1T9&^I{KuJyr_E?&-QOXAc60-}r(I@m9eoIg%?l1iJcVB}M*MG|(OcWTNTZKnF=6a|?-Gh-*DfHnUOLI<l6D}hm$M$wh4IrzzVTDmQYk^)^&)Mox%Cb?!ERgTsF=Kc(F+A$u?v}WwVaW5mb`rBR9kK999jJdhxZP>+fLeEYa9Xpf-YHZC!&pd>Cze}3=V{!rj_m-D-5mOs!^D`$Q1ho#o*QB&WMRUGiLl=^&y?Xqk+CtS1EUN}EnnRd>GDs`^(oiGiRKfgofO^TR7kX@m28ErU^<DI;ly=N+G!PT;Nc}P<DH-^#rrWgzGu(7P$R&fPMHx(qTmBzcm}c3Kp%tzB3FbSh?p)09(p&T?}Jx3yLZyUOT|aZrC*Y2V>@Odoob#FbUw{|sSHuGoKtXmod$Co@HNy`chU;9smYqEY7SF#EtHYabwQ<ks78;wz@^dWH6P*<!#P3$YIN~6s&4fx2NaTgm8UreMP}mi)c5Wl?L8XqKRSMRXYY9U;Lh>=U+)hO?;L;bb-}ysb%Io#aqV{qfsk<FsW`TGUGQSe0^ViGS3%sSq6S^1>9ga!62<0wGu5(|Q%G%wX+tvBYns2L*Sg9F-g2~iC-6}pJ}#daeZE9lAOWXmasZhJPkm3LbRzOx<QG4~F4fLL??cUVm7a26i>L`GVhbp8lD<iFt;qO8{s;Y9xeSm!qlR?jBibe{a*o0e38+uMyO42{sHpE{hR3_7Kl7UJNCbi$5Q+f`d_plqMSTUfRo5<yE^E3b<FTGrBJx?tCkFQEk5Cd=j%Q5;Id&WcuC>xOT8;MkENqI=s>xVLmZnR_-XMf!zv*bZ&I7l^Q4|l$={CN(yM<P(9N~No>#R!;VJf&ZeRQx|ZV)jA8sV<j=2bD8WvWf9<<Vm`!pyadbbVln*s-@h1kQv>ie51|+IzT1@O}qfLY>LwBvEmeyYfjuf##taOYs}-4}2ve8=+m`Gh>r@^Q;R)Ok*g|NmJtW=^CrFrL~mV@dxv1TX+{tS}nX+F55YDgqB#`D&8t?FO-OSaN(+8k1W>(&xT+RVh`I+lPs>jGca0QeJgr!!Dr#6cUeg>wEqeLs_H;|v&keqN8cGbs=oW%sa$;(#yWffS#gg9x+`wk;;CI2RDHt|tCVgEcH1-q{em==2zt6!N2H<=GhqTI5sRxIrXV8kvL$B^9Ey_?e|fSTOH0?Q4p&R&N<?qN*zUB1qnKvUP+UfS?4eIw_Ta)Hyh(!IJh`lulh$-n>u&hWI1^4ZkHxHqsl0<jO>SzLiO{(G+bkz8$;aM>MQ}O1r~_YmHEK0t)Ty_UatO<rI>;xfzzNjHZ=J!%CqbGUEhfGf)Vd(0!5ax#Bd4+~b>w);vsFqsy%qfeDS|rLEUl?8)@%%YJ9Gy|AzeB45gj^bsL+Y~-qggqKVMe5n|cS%Ou;O(Z7`zTY3nKIbAVh|Jq0mdFCg|oYv$hUBsiSzerhMxdYb6MSS;qRV3M+sCn}A&u1T`$j;rq6>`5BqLcv7?!v%ITJsakBjOf`JBEz}F%hgnebS7jfB;Q4i;6)MD^tTqZr3Vf^$a)`=xF4R>-LB{bU8)jMElr_M6;6Q54N4=8rcuNe$mtv%jlR3K+on#t2WjhmY!0F+blJn1NxJp-13anq6*O5ot|QIPgdA_<#3CElEQtNCeH$3`<v!Rx`jq-;q=2=}tEq%oB&@~5jotUEi;Pk+sa~(;fFUk~^*VO;DWGCOjvL&%bwQ9f+08E8M97<+dTy&2#k*xLc{bw6DAFg6R%We+QLHuIK0|DrA2x@ez8}Gb7SSa<Nv{7X#~686F9f`K_Mg8m<}bpwDPTF0T7w5I-iVw`<hhV^s0kjaZZcUcF*CVLWfS<YOu3g{i;zWjleA)BhP@}=kz#9bi)`E)-PnvQPv_Vw<wje3_UsCg+PA9aG>zwZ)Eq3XUdRLvnMsb%CmfCr_QC)1@ZPQc!+RabnH;-S9|mDF4+bTwgT?&M0v7YXVt)P3z>~%NHH^^0--IBqpDTRre54D%{QUas2fu(&vpgxk<=ogT7kFN!d$Lhuh+jX`lYbW-JVkj~Q`S!gMKOw{U-iK6)~~*$2JJnpl$)ZSiY9gAh*OlkcPm6|O4oMGxd1hKarMI5+^e-sDlI(i47U>;7s56#yS-yY_Xd11;UWC%`%i+O^dL+Y^FNSV?YqSl_{FE-?iYv0pTMp1D)?y!vc>$HDnZn+hDtzs{TxEeXR?^T3`_O{IP1AxbNKI<atz`1Yl8WTIC#k*y#6+XRAB>v?9>oHN$|=aVJM8r+9}B7`U;)ZTt63(ipBi*Q}|NpsJs}7#r&%&ym_{m|0xK94sEMtvY5Y;Rj+fyvI%dVUB50oeUeZHY)El>{k+hBAvlZ|SFdEfM=3Y#HS~VlOWc`HAz#d&LAaQI&tSBe{{=a{`T<wzQ#tNbvFp}N&@T<UT8tDHqoafUBf4)^q={DJmA#esE3L-&?|M{;3JRNxk=qnFblBP!PT>`OhrQdv!B>Wf*ektbw3z=9o=#a0ERX#Qi}nLu)+lG8y3aPtLF>+^Ax(RqO$DHOd;>I>XjqqTK`yTTkx?E!?Uj8Be5QM%+WS=Jn=E#U7ArxU7PI|BdD+_P(9*l4^nYk0?AY61-zn4Qw^#h-S+(3gt!%Z|?z6Ydod^4c`QAs9W0}~H$@vT?|D`r%Lqo%7=0`L!?`{Tb{l@-b#Rxyr02BNl*#>uMfvoyJz@1jJ)s`)VhL<*71=(#DXqNv**R`}ZTY0jwQ>kiqv9@`4uUIqo{SQz}0|XQR000O80041ab+SxywE_SDf&~BoBme*aZ*X*JZE1RCWG{1aV=rWHV|ZU>cyw}MV{|WUZ((F*axQRrbyUxb(?Ary=dXCAmjr5x*Gj~LD=hxHcv57UW+s`@$xNJ??zUYP6c3&kyeuNP6cJfr3wmk}9xDAm#=k@+KiX#3&0*TS?|U!beDBS$ypYPkj1?*4#AEeqAvK#6$WSj;G6#$dN;uM*XlNLlP|e_zoVO4$ijX=SwK1VR&%3uxs#dr!lW|Uj8F~N!BZ;TOIy*Yp<HR8qpa|DPz$`Jad3vDsG2FTf>6mjTA1b*HayreGgkG2D5=T=*oGMIG;1c1$0YlJ6g%0j&J7R{mCPmL#@qlqsC4_)9%xaQ*uuKk|7ctBtMGy}BBdPEJH`Ss^&i!!1?!jYQo*9=l0cy*60o;}!K+yF<Wbm4<eu)f{)laa?xuAomkM;(NaAcSinz91K>Wi#!Lv8tfr|wQ_LGALJfJ(=K^RlBRauF~}imbBd@7s@U@}JZ7!j|)7r);8-rjRjPULr`SEiXg{hpQi8D6`AoW4H|kQl_mivdc>Yqp>Z|l4w)LsOi?Qw!!~%?FO|w29ila3Ih+xbWbU%ZY=vm3bMiK0f88fJ9IH@zb>{Pa8D`~2w(zckw7`&Oq-y~U@3@58>EbOlhVk~31(>Ehv4$B!z3ROjG4&vU}r}sUOn?t8i<?I;LegZiis>}<}2Ks7>Fxwlrn(>p`$l}f8_VUcbFfBO*Pl!7>-Y>H<oU#VH$)ql|Nha+=dn5Zp+sMw3*=*uNihq2#+#i0)HRxY^mxwQW-J3lxJ1M@6viuhrz5D>+@;`ZF$Pz(3YPV7`k3SZp*JpJ<V_j5<0fa?*gb?ojcaMW@<l+46t2Z2shN-Zn}#g-d=H$kSce7f^F!PWTNhp-*2RcU7^l~7mDGR5LXo&o6T5>&D9aO9>i|xSNzy5{bu~)`X}`M0Z>Z=1QY-O00;m80C8Q{p+}^v0002T0000f0001QaCB*HX?kU3FLQEZFJy0Hcwc3BbaG*1bT4gTbZB2;b8}^Mb1rasHO(;!!Y~vD;Jv5#p`%a~oLmJr&mbW_n?jq!<hAIgcmlx#bQ1+}a5FgNB=IDI;{L&xqY<nY!h2$0*e%|>6KpLcDN`@WVE}gVCKbK}K82$$!OrLsQ=^$STx2=sX4%#R9s4cHDup{ZUNu;efwv^l#U3&~z$%3Uq)Oo;ps2Rwz^YBpvj(^Eg)W6Xyl9JT<xGP@{t{G;HIDx$kVBibm|!*@j<OF>O9KQH000080001SUCcUDo0bCr09^|J04D$d0B>+~X>DnGWn?dNa$_%KZ)130Wq5RQVPkYJZDDk1Y+-a|crI{xl~zrQ+(r<+@2@D@7l~1n`(W)Q_~wL#<PeB$(Q1`G5~*8mwU!x-!8b4D5DW=E26D;482e_}!T7Kt|3mvrLbrNnq>*~;=pv@8>%FR{H`Sd<Q9}r2-3Y0Gw`XT5W)Lf{k(ia602Iyr^%($wMAb<SybYgDqBeyUWtB?gaYYqwJgl(x;j43Km7L@g68<bo?!{cB=rJEMA%Q{3!K7F;v@i{o$W;IaJ=Sy+OhU**`UCz5tl#}jfe8-x>-B!~Ie3>T)N9_=0n}^s;r7MvAp6~Go&)c%KXq`~U%%sdNC+{2CKVzWc*CsZsb@MSH5pgok6SwK-Td^S%q(2l0v8It6C44n7J(4krqN(7m7+OBMvqX@+$@#Zuo~%)V~Qb?*e_#JHtH_rsA!^yEq7G1r#327xKSV^5Mqke5KE;x=+&6s!VV?xVO^=&LaHQ>GD+Dp0;|L4*LVRC-qpm16_DqP;S{+II~m*WyJms(7zqcQCyO+#hWIQ^^5rzKgC_YBn&pPP|4+4txO+79XBF_BuB7R>t`u{ZFH_9N&P-^F5G`pn28#!yAOGc2%#DXNoy)_vU<-06acaW$sZ>Ad=I_W>Kpq=4Oo4~*GEZCDwV=>Mp9J#&kn5R4JRS89=xHOJ+Fi)8cDk)bjys>Wmq4U9%wQ7CzbPh2`a#ET<mtlhGK0j;Y<q+Vk}yX#hP$K{@dMrV3=mHP4tk*r&>sPooTFB8f9xF3>bd{-&*Wq>YS5~h2txe%XqDtqg_$y5z#LY#gq04+GA~R7dFrwUr*b=X$URUuZ&?9Naq}1dTJ^g>c)o^TZazX$_PduI^t(SnCtb*MR!H#b{`v!he_r*wUwHwyS8om_vO>l+Z6JbY<o4<nSsUwihWGaBx8ZjqMLI0J85T;e$n!l$yREgJI=WxuZl?~F9hp_{cYa|8k=`&vVvF8mr>!zn%N$l|xAg(g7$UX_gHR@|-PT7tMW<)be*Zz5PXEzPPuaH9@-I%-u|jQ^Oq<c1tLSc$Yo_TaGC9q1&M?h2^EAiCPmNq_YSAf|UnyhD$9X}AiRd_zY#sL?p#nVdEmcJT=5;|O4FCf$_qG&y7=}I+!uVR#v*O?P6gK_tEmSuj_0=iuw*Bs<(l@|UH?z?dQ`B`8D=`k(%a`2F5Aef9&4h5nuu<^9yh_1m3EGTj{{c`-0|XQR000O80041a2iF2}6%zmeNI?JqC;$KeZ*X*JZE1RCWG{1aV=rWHV|ZU>cyw}MV{|WVVRUG2VRUJBWiD`ewL5E%W4CqR^(&6)WGz{DHInABv*Vpb?D4oU>V3f84Fb$g2GqTjMDG>vH81b%jK^cRP2xvmply=20g5I8+7xm7DSf3)A0OK9+W+8RQk+BHr>;htEh=DV$wTtIc}O1eGREE&Jb)yP<47A3z&&oJkIrKO_Z}U-1*h8(NG*I5p1%fKyf1tMQu)y=4uuimN9t7QL`Ev<wOVgT#q~La{5(p{9=s_MIZ^PQFNN~OD+$EuD3r+*i8$OzEPOeZJ~uK_9?YcHkrn}P)0!F+Cwm(k+Kh)GH{xt4dP$UOUyLI?5j`c$2BO*SP0cKX9z0f0RrE|jVsz@8REq?-79bF1;<b_pd>)2EC*X4hX(B+*_(Z^igLgry#Go3YF(fh+%7o_-Mu}7th`BZh=L*CJDKXk>F~(YB9nB!%hWjB;5|MzMp$ZF-2V*J1pyl8)%1V~QR#uHdF_xidwOTX`9;@6MXpHuEc$kVq6r;1(0sx44lC)Z_Z{?KS!23}kY>%f>1$!Wsp&Itv1Cz#~fPH9p2<T5`7-*sPAe4!Leamb+5M!Q(=48x$6Y2APZ0zK$p|1#SX#`>nCnr)#b8^y7L^$?{kmC`52<#!uv&A?{T*;48g*0tr--CX2P-4zQk-$D&Tvi*#kunHBI<CU~G%?ZaBp=sr<vbexR5<=ri8(n21$Uqik<Ph;E-1JQp-}Cd_tFsjX)Y77U?TQNr^@Fve;ysa#jJ|)cZ$Y?{X!Q<8L_>r&{Z{Ih?9L#IaCXiYGrX1f{o6N6|BUf>q%%x@ImLOe_V`7r80w^C8MM=J@o1m#GJGXB*TseER@ECf_19M>ZmI65{I2e<{Tp(w1XS755w(k5Md(fk|st=HK~&lDkv+qH<l`BkJytwTktp3@}NK5DtM7&8h;1+YuWd)GD6R!%1!KeT4rfdn^3{cjC|3rY?~DpR%tNFd^d_Q^dwaec0jlRyHzpdxI|~zLFJCvle67E{$tg?Ri?H|1`gy*D2ncN-O~j2AQmdXAX%}u2lpc-@~$L^9F0QJuFWMCsW9?+n4zglkYSezsK^l3Mb1fOWlmhF#zmu|7w{}`T>KRTY|k4o*lsPu40rFsR^B>}G@O7`pt+ieHtDXW3R&m~?!wlhm`nZAE_&pWxw#-!(v)1Yfr-Izx5h$Q2^lX79m=zHu)T-?e<LmU)8!-Y!eG!maEXj>t5Y7zpeAEYJPaGx&{};qH=!+gAiAGa&_<W?pugUlknvrZCFTL5E631ignKGGzdQ&;Y^M8V{Bmr8a2IyA)*?&tD3KUsJe0GFCuotRp{XvQ&!#dI@JOdM1sYzP*p=4OI^NxfL1Xl>&39qg@AqqQc9j5H_>m44_;=Wewf%bRF3mzKXD!+U`TVxn+lEh^&KD?9?<)h3CDG$CI!PP*m(1!2SgV%ER1Aw!6ce)V!C<@YSymck0N`bKCz?o%m$(apM1Ckd;xF5m;&$8!WqJ)n+5-=U9&CEB<-xWGJ2gt#O0vS>c16n8q9tZ&nBJAO>f>dRV?~gA=k5E48Je0;EYN5Op}7!lQM?+#@LDV$&PE~#L;yMsg>GOjw`3r!+u=HSbX=u5u)7~(G7AjkH{C&JnX-qgl)bZwTigP$Ijq))eL~Z}mb?d3uDNf7hG)`DqtrmE_-rI6X_O{#@6kaN;8wAY&78+#%zfd3Uz&~9*=`PoyIys3_<uC<U<HD~&I>q+n2kgjW~|q()M(FAtS#}u)p=X>@T3MW?ZXZ&(6|S86gt<k$73bVV&UU@=I5rdh-5OI2~`iG%|3}%@p3D5yo{G6zPGu`_jX@so*OZX@yLJ0Tu($O@PO*UDc90QzuG2rSsZg*7UOQfV7P?QGV*qH#H|i|1nm+LxN`?~8#r|-jw>Q-vn?7#^m1&OeCtMut(75}4#F=Q{vP5E0wV#WMU#z%8_SKzLpf0>9{ztF^DQc?g6g`;+5Z1=;~6(vo?Y$4F3xOraC!fZ&<5A{jcKIihmkTog!j|L<QcMJVb58Jo+TJe!{nOokNXB3j8?}op@=lWUZu)i*y`V25+PUYc%9_M@v%X8%&W=Do`W>eMh3cDosEqx=wd($U9Gb$h8Bpf7c%czCiCGCN+p^&UG%m>>ubB()!wwilPZCHi9<C8klY<g3>AKVtI9}Wgwn`YH;a)XEXLM^z#XXvlzz+1N!g(*9I+v9*L0BR!yUWdy|Z5=<|qx=0j>f^^9Fppx#UEU9Y$!(5&pe7IPQhfGodm4q~K!2jGT!Clz0X@nsLQE2f_jd-w+Ev59LTpTbzxw<W!7OErrIx?DC5Jgcb4V=Azigs4<e_BN%S)G^+b{%~bNkt8x{k4Gf7-z4u|zcN>z;DztB*rR`FgbbYg4GH!GKo$H<~s?%fp+S9|2!eZ@W%T%%3xHsFmf|Z6v)0vXekxRtCwNzcNl^iiUH&zfX!MG&1=e8X$HJ^i5M3n5v7;7;W8ZT%XHCM_2X6%<i3mGP<0&!M4)K;c{chC@_X>M0SM(e8iLj-8XlvE(c77ban->k&?3|ei-er*r*a(*}VN`j$iVx~6(K^Y24Bb^*PxOeaH{v((P!!aZ6vH-oSqO29U%t1Ym)ojUeHTCDxB^QQIE-UGtJb%z?wN4Jc{rbCj>mq&A{!qqkHvi51ALjo!|EKvs&;Moq+xfrF|84%e`M=NqWB#A>|9bw(^FMh0{pUY;{=?^=KL70b=g+^G|78Bt`OoHmIsf_mm-D}#|7!lX^Iy-uVjhuevnNkjH|a89<OvCS0pnpj<uDS4!-UUf90Cy<4kAuu7%BJ=8}Of*;$ImV4o`U;a|rot6mXcOpi(%K5KGY02ogD&aWFiE)S)=#I-W|r&Vo3R5Y5Dd!|xHTH;L9GLj8`#_gx7OCA>#i9y<&VrxG3}5+2}C)=T10qDeX(tFT@g2i%DE1^fE*AG1AnlX<|t{?QNFo@P({AToe}|M?55+l7Pxza>B*Pr+%3ef`DX5G)iQP?xX2_!ZSBdHt7ESHe|4rz($Q9i2JQ_X(898I5sxKmR#Fr5c-a+wXHb6iq<YFD&B7<d9DYGLAG4LkF_B#hFY@QXv2I#{|}LGBsT5=$Y;P>7O`R#%1Vdw)4BS&Y%61!0(8$so=>-{3$i3aT)(ltp1{Bss#PPUlDZ1&2%)nc=ysW{xfR}sr?-1^FJdPx`yrZhqjKpJ)Q|{$tLN{)&JP`(NW+4fA6$ta=icj8`~!vV!52fuKIVDlQgS;W$QevTdz6gT194x3EB+rlQZ6@HV->KNyWH7tkt)u{#stY<=QVht&_v|4qktFh!>{pg1{Hp?UIV5q!TKhVDXz&{3aG(rsB(3yg|hqSo{VRzd?kaP<h1tr%A|@DX`1d$?FFPhxZ@7ht!7FMuvH?Nkv=UvSnM@wG$%@1oPlFR;JuIxwGG3Kzb2A^I$+oqf?=C=R)Xa8CrM|Hr>jbq!P&CTZi`#4v9g`!anWnol;DC>YdVtkG0tzYomM}w#PbHIAVM32n)w-j~!#7Nsrc6AF)055!v&TKc^NvxKaJfKeCfz97d70kRRFEQ-av38Tvwo7GNi*wg5LJzO<ebay1E2Z|v(Yt#9c9@{3>B`~2doa-YkdG-8&tJD06ii>~YQ+z{of-G)6oH`(M5CVub+C<+8!tJ<b1*RT)%R8xw`E--Rz;Zgww%Ee1y0v}-~n*pQcb#sVYdCLFY!8{7>;t7$vWntg@m^I+CF5a66)5hK&jj^0m7tQ4hX6cm+Whmis_W~K&>u-HGTQ=?19N$BrBJhC#@;8d2ux+(P9M~jgzk`{QZMTQST?mfJE{k5x_CSNpjyK%0#}ztj6N&3F?AXdub8vxwGoCZbS)RXxC;7G;l2LUVDC~HQY&TrmyU40%7Yw=%+C}%TP{ETUy_#rL$Dk4IC<=q*d^U>0M$|YtU1C)#>wEe8lpLK)8QXsLEk(b`tfDv4osEJv-fws?sNZmA{ICnwrE-^DA~zS?d+ZhF!Ok9g1^*xHV&N5bd4w1X66C%akt_-}lgL^v29&f>?(6RO3V*1k1jvgsZ>YW{uoFKKAEda+r+J^$CO$5TrAIZCh;~v^_v)h=G!A90F#+tn%f*VL{xRu)c*_G!<_~YzEYO(KJXr`gC35AY>&F9hyK3HiP~^=kZqvEOk*LsW;%gb3f@u9n6h>B2C4J@;bEPifsCo$?$2OaR80iexkZ@g{>sRS)Bs4u+vz!yYI@jsQ!v}tNXu{??A9<#k?tyGPPSV&0kxJXHYa6Yv-S-N<Hga4unasHHrwfdZ$MOtQ?d;@&b(3^N6BN(8>Bt`L80%nKkbRHZ<Dxyz+w0;{>?6!;GkrLV&AC%)4dI9<6heq><wn3VKy#@VN3lJY+NtGOt01x+T^0Z%5@JSqx|Z3ij)ax5Ct9eWzM@wm0b{-QBdI7;nsl;|&@{}Fsn7CKNOu^k@nmrdG@=<UqFJ&ZX&he4#w^^Ptr-PYPpB9#GTb$K$@19^#l>Kc;ihf4$8e{*xyLlSq!q9n11q^!J5(kOuL#x%NW>4JsNLaZ?;O~Zk(fwz!L{b+mlv!{oym_h?%u8Cb7QXxQXL#6RN(NjMoW}3bd~#u#USpYT$KbGthi=)(TIL^R^P|1gnfR}5Nv2^w1IqG#T+lyyuIwG>YLqg1xx8j?#V7ndu-9mB16{qP(ij^$D=ES+y)uuMI*daxqDsh*4dkA9ug9#iYspz7wm~)Hfzhaxb40-@P-uW^5A}FA8y--hpp8%N+;)yo$?dZ^{#|(<}6gxz4^+lme`N<S41>x&H%G!B=@>wU&_ZW#2|WBQEUeaTqP=<>liJ+1}_jeY1i{#(n-#2**>Kdb$yFPk3~u@X=kK^l_kLjIu+%^Zwxx!8l-&4STo{{aak!I@t0<;(dSiclkhyAq8cQwAnwqAq_Qx4+-!mh3d|*-r;9Vww&>i8<Xk4Vc=xKSH{~C$95`GYD9(Hl8~A;3POtfaCkT2$4G~e+0s0I<ocRs$358=_{EJyxd*n%u`o|bwNC4KQ{}x<?vfoR5B%As)w<BGdLp9gQx?+g9wv;-eGwY%oRJg$_KiZU*>Ed34fKKAU@pZQW$p<NZJ3>6rjl&i62Sln}n=>{VM5f?j&Gkzr^w~)x>B6+h2imZ1n;qq59UOPracx(cZHc@2iI>b@I?z{4K3E<WZmlr&;sItIN5;J$5@<3r_eR-U>*B=nGCb5G5&Bf%(MM=*BE1m^713kfNW(-&X^i?4r!E9@NJ6fV+=%O5gDQ~=+&6U@Zl6Yt+959<Vj^d`iUzAyG`O~kMqD>)$B$++bUR9}q__l7>T|)QE((>?qYLR7hii$Gl$5p5F*|-Ogv;W2eO-&218;NaZPpaI?t*iD1#;iV1j&*>Qr6XG4S8eK8+b#n%=<MsIkl9Q<rWHyUJM%Y-+4~FocUgs62CmksZ_HLJj8boj>tv-Y}kKu$LpUB2e<o|-q1mAS;+9x+q96wfrV^cHX>d;pO#hkLb`HRTdgb2o;;I*u(t~f7A?FVJK?RvGHvwFHv5~G%MeiW@Ur?SKljX|{&AlDa(?14ujQ?;TUlv6-zYq;;n=cl8ZHg$#XbDCH!Qnt6x-NIpV;-!hQr|-?xKsKsfWV@suuM(ZM^^-lIu}@!NByxUiMZeNoRIY|E=tL&DIBfx*UrN$N_ouNKX7Tc93BP;W4|sB4%tZjTyDL++eO;9oG&oA=eX;k8cb!qbNA{AdO?8J)mbti|d-RY|h6~Grz}On;9c4S@5+e^QGSA#)w?jgDR!X0I)7H)mk^#oF7CkGS9l)xX3*f)6O=fG&5p+VxklJT(@2Ncv916qp<J3AGJ-Hx7x<iYnF0)q1gftRCu0$LoAcx>*YKHPAIeQ*-D_wi;o53xQefRB+Z}DR8OLmF~`VcF=r10JLN-TnLlz_)W=N+#g7HoIVbv&O43>O0aG<+m4D{5koeE_K!-)rM_e#RuR>|;N(O9>lxSy~sNO%gPjIYjM4@Q(3^$Jnf*%UKieG8cpVdzA&S7B#6Ge6|Iv(gPviq>Q_aGE+$xyugpe>``8~APO+Yc<s4xYiHu(W(RB}Ad<*@qBqc9;>l&y744Y`u;A{RnCUF**o7D}@GS;rm1`iiNU08c6ogZiSsFsYXp5xC7<a>np!ct}`_yhj2{U%8C87Xy1jiUaDg5KRv;dW}DslHuD_gc1%Pbe)X8Q^D$obMqZA~)aU@Qo1K~P&1VU1_Vm+Hy;&N!Qt<3aa$Riueyi%S)%azp9hHfd%H0xw$DP*y0#Hi>1QY-O00;m80C8QXLWpJ#4gdgLF8}}~0001QaCB*HX?kU3FLQEZFJy0Hcwc3BbaG*1bT4gTbZBp5axQRry;@Ii+_)9L2gr9|dD0|`Vkg;R;bL|>>&$M9b!>M$yPK_H2(<L%;aVdqB<0v~eeUNd7JKWVKznKS&_jTJk9?Ejk(5MfG`52*ItN>#$oC!}-=7~JNh$q`>0|*Zc`=_Kkp1mhNZ^7c`Vn8UQy?j%RhjdIX)X%Fij<s#R9qC05>+J$C`A&MEFhwk-oJPM9$%J1YO-Kzk@J)0{U;@g=0&Lc_cAG#q+}X<5!Zge^rGpcUguoHQ`a*QMN&x#g^sdHR}xe`XugoZ(gPv#=>;T}7SavTd<k`@5@Z%9qAE1hpDUT?{3I%wR86-^=BmuO4)Lif%(FKs39l<qT49r+5lN;r$cBrAl1Ye->af%OdnsfjmZH)SK#`V$7dj-W$})a2+PZ&_5JE~OwIbt&dDC%Jf7c5#w%sCpiKNn!mjQiD`(5;cJS6n-39cz6Jaeqbcuc5kgj|8?8A;2IKBghr935>Oo3RUExn599Sc0x(;b|kVV#E^4oFxz-p)vj$jl2mPi-@r~sAqY?6p%3yDuUviOHo9ppaZ(QzdhNV92`8K%qMh2gk0(I>iIYE_GJ6TG=8!Fa$3*W35FIE7}7$TmuMlD5RAxULL(yuHP5L?E*`;!mMqZ}8$==_XJN){@83&urW88BcFv*DPHal)egWhV6dL{G!$0kE4QP!8B*rC5kmM(E5)$zSed6Sj=mn63<vA}-i8jU=A$_EFfWD(N6xuiq_CTBBXwHl*MGE<2G$JX`vGB9iWWko8N|pezmc0N=Pk}rK8IfH$WyvL}4M!0POU4UQDRz<rdBgQWR4q@XSq_#R5tS}09W}G9MK$Xf^MdO*4iw~BNFIa@hw+{$Aav3h8?X(=#$v_af^D$0IzQ(mZh*3ib#J8<rxKKEUadd95%LVAZLRIePCw70S|l=Vq(j4RpA{*DCOFpGzEEgbP@D(!`Tq8+c=~ESn{3bH{q0#i-Ftqpzq2=|A)&RYHJmV@pc=<y$u5G;kYu@FI=G46eCOqK|2sqO=>{E*`XpO@wN#h92%bI(2`_ZueYdW$meW_0`Sy$WaOclcWcE4o>N9id#6aPmc-B6CjG7K+(}T%u8t=TEd^0^nhMROmem5p`gAOyT*RVUy&+RPxeOj^FrdGTGo4WaT=jr^gKl^q%GxK>itUzm?MuYaT*Tb1J8P|qwVsQP~S8<(BA{SnQKEnmD)GF=jy8Uzpmo}2FozUc9C;oQ&qi?}H!*QPT^qCydNu?J;^0y|}(_=Cw8T|^F1WB&W;PSe^w5-yLSS6xFlX3|^QVaIvv)=~+3MnLdQ3ORIQV2$)XaN@~KLw@znI@vh_-Q2}C1Wq5pv3;$$JnRI7c?YF%b=5tR+iM`Ft)^T!L$rosYX#Q-hd27gcszBiY!9|*}y*;%N5XTE0D=W-mo60^u%%KwYRZ-@Ux2mRf!10sAw*$ey5bHM4W?F;nyfsKXge}D5MaBltK=(QkoeclJZ0cT4<J!H$qYs{B;HR)M7zM@!jff!L=H13`P3pCt)eHgGJbvEVM4;+WN5}V?0MtcMG99DyZ>BRFiSfHWkL*9yif)17259H3VH7FEWsIn^D9a=vgSp-}<^S%2wkIgN@p(3J618TJ`gp12x#(8*t&L;n*Kz;)^ve&qap(JUxk$TqszHGA=%~Up>~fWv$5C@NG=5$J0*cCPVESvX=emBtey;FtD*BC&EI9sXGWDB-DY|QAk`=8~Y(HG<yP3>2dHhBv1Wd;PZ<l;|k(T$QWIsQohFUw*<{nrWqNxV{n7h@3-RL^+1bkd)ST+jb3&CE)5InL!(P?JX|DDY7#UVW@aCzrk#r-B;T{Vf+-qkNG4iKeo|@s?k_X`O6KRedg0r7;jXYBOxN4u>N+=*ZO7xb4EK7=vy@BRb3mch7$=F_Q?7I@&dh61XWeM)0Dk1)deCFin#1OLG7*?vB?NSPHl56;WIp+7cS?4?CVTsHGJUmkI6ovJk;G)Y`SjJC9L#oJPG&!nZ>K+oBxf2fY&#Cv`)+r4)L-d(we}C)denjsE=M<BA5lNl&|~s&Og2{s+$l`l8ZK5<Y}1GrFwWr8#VIsWTrJ&!><%s*^*px+D8inN4MpBt0YLoybauG2zvuAj5DjXh_M`JnL<=5X4R0hrn;g!kv-tVJjz#tx7;)f2AFssn9xTEr5vA3#R6#vYOv-0)X|1~jOXQ(ZU=K5SiarHxp-t-aeFm2wofw9lQHnQp%=I1MaJoC)o;!>Fdba=4*!>SLrn4zBAAd<h;|1Yp6lI{3g(wEC$Eik&H&}Dvnc-(3)GEApuF-LbPw@xGd&O)OJBO}1OU9+TU7;rWnZ9?AbFEUBthfy5&fej4HYfWtGMgUkPPV6HXK%i5wEO*J_q*vK3BC-;mm^wBwVrMwmP_7&*T1C}J%yD**?`&>1w!)R!Py%oPdjQ;nz_v77vWYxh-KSBhAXQ#gb^CZ)ZhanS}W8r5@<wi>1DiNc|N#D@}+ZPb!S5!)f26?^hX@SlO^bdNIk_#P*sj~w98zu)Yp`CEG*-hPd34uMn7THz0Ci}Rsu(8oLuojX;vf<m~UqLGcxws?K|LVPK{TN-Ig5@5^4;9ZV_s#iI%q+k9(w!pth$e;nDr=897_L{})!k<cD`ZlU+84S9EmU^*JbNXG<1UaW>>^ef(o)14Up5kT1yQEuhgF>Z+};0gx6?4dsfA=b*+%0W}X8>N0vIczPXO!G*r2HsOs%B*RJ#PrU?)8%HDX@c>=yHNK)eRdkDZ`QeCbSYm)gdidN@8C?(1X`t<AcPUKSa#35J!sW!mj9=)oHJZ&QW@F^lzaDriV1GSvdeH|Xd<~N=ppG%eneFtmjb|Zww(;5Zx^T1#fP4E}{Gmk-<*~a0di*(Kz75kmx>atr${i1I&3$y|RDcP?b5YkhBso_dgw$b#a(hTUFrH;p(i*y6@DK0)Q;{3kFIia{-T7stMtK({Bv!RoYi({|9QLt1v6th%W+H++#a6kt%HrvepiWoV-RP@mrL$gzO?EbGINe!o$@_m2o)RtJ|GPMS?4s4;!>>P=B>(X1U)M=znI8$B9ycMzg>?IGV4UVlUYxGji)*GKxw;+=>lXnqPC1|smm=OYB@28obqc!9c)Y}wQCHcxLW0+I3bSFtiD6{QXwxGk8~qKkpD6@Sx12J}NE7ke{9z0DZTh&uZh1<to2cn8xo2~O27E)H?Wv^BCcQG)fa1E#yLLW9)COE65ryW3U*)&~bPp!3UVgGk=*r|Ex`iny6LMMQteb?c+pVo1xksNEB5$^RKW47k;NGzuF~2O=R3->hNNioQSS@pA#_}9rq8A_D{i4aqhj)MDTjUBZ62GYZ+uDG06iciNon4L6SgHd+3UXuf-Ev?lK|z5y@*FUH9Gyy0m34rQLt@_1>g(~xHK&?eB8qd6I>2b&j}$%U8Jm#btNmD6)8;TPjcwP?S$ELx^&4NS50(sC^g0aUcO%I#FMmRP+D+xDe_Zqh*=(R<PmQDIBksM1(xC|>SM<%&6#8MA(d$~@aT8|xRC}t$BiDhe35c<p<IEw+jfIy&<O3#d>;;iOS31Bx(UO&cTi|d-BEKBCijO+P?n=ks3$^ubn@m)-tZ6`c^yC<i1BK+LxT!jN;#Ed&{h_WKN?}6;wf>fu7I@w$rZXzj(f2$Fi8w9zTMPR`WA2@mqm<5Eal%2olS|y!;UPLt|Ii#$kTZbN6U-cJgrtCTknTE+Z?=;YdwT@W<m)qk41<JaczcAACW;Unx3nPa9JAqG9mf^H`!h8`R*<S-=Zxn@wE7yrPLc9+o>nZ!dkmJALbb6b_nK<~TnUV6crU@5`A2WrZ#z^n>a9)8>U!9e%n^&VB6Qe(S9?XFm;4=~?&QZCcJB4ct>-c*)a{%7wb<xpEYw_2NEZM72P4UccfV9*@&50{!UY=>5i#Q6`Wach$=lT1oXvQ%FbP532`Cw&J(by`WlJ6_xy^w3478^(%Pvx&Q^*(s0j&R#qvKJ8d9O$V3k+B3tT6?9+Ssm=Ww55Re<0akSn_CiJPL(AI3`EOT@4w@n}q6SW{5SHnChP`6gHajO0igkUBRCA+now-c$(Uq{QwYimv*W$Wg3FMy;*jh;?00f8QT4Q$C{0_Q(Cj8t*ZQWye%FW5wc8yO`xneg<vBz6LkzmuF}w6d+BSw#yKW7nK4I}<d&oSuuq|vd3e^Bb6~#f(AB!#?TyRPbN@bKyyMEO-%d0~-WZZ@h}nMDcRP$e5Va6=eN1dvt%-^qQPz{`+y`i_-By(DwpU7kpL-r?N{K_`hwXAGef{5eEWgcW#Z$#68{SZ*rwMg_xEv0dH+YGw6O9{jo1M(j#_<3)9P%i0EydMXKEi2i^|8wJkyFg^8ZWjp?Z&v?$bR~B#^;?K*?+4kpmTq0T_&i%2j~q}1*6c*X`$6Y>q_ggXcz4uGDob`n@q2O=Tv@<R;@aT{u5A<blW>ayS*T6JU`YVHli4mu|-ZUHG*ik&|CK8;?WoQpZvu?xTKVBi)G0)KjAsoc;{8iOJcDUP6%(!=HPCx<2786%HB=K;bvRb-0=;JcP9RQUr%V;9<`eWto_?+i>tHP4ejyhsA)fLw0r;F{{T=+0|XQR000O80041aaI*!L836zQU;+RDAOHXWZ*X*JZE1RCWG{1aV=rWHV|ZU>cyw}MV{|WWb1rasrI4{|!!QtrcRj^%i&LuDp+g5j3WlyB!9y~XQiLMgqLxmPlsFLl?!`_zwa`GPJNSRT`?wvz67bTd<V8_L)oLqaPF}mS?P>~ErkGP`ik`yW2U0pl;~Lu)Pm9HKVWn{cvems5s%xY4E2*Q^E`2jnJ0L6XW$c6Oe=1b}Q3$$|$&ib!@o_-5y~2*ci5qEdjS3X0PIE<8oNPBBTmMy~>DFxw$TroTl3uD$+34Tx*vG2ez!EOlqktU;v?yT30VN73Ibc8m6$d<_fGr0k6tL!iehfC%z}Q~baG3d^YjYM5Q{a$fGi?tre;KD>004-1m}Py&GoMXKcFFU6db-aLaViC!j={m`^>$^x!{Z2KUr<W}1QY-O00;m80C8Pg(QGF91^@t`6#xJv0001QaCB*HX?kU3FLQEZFJy0Hcwc3BbaG*1bT4vYb97~LE^v9BSZ#CLHWL1>U$Mn~D4EhMB_~dst2&cfNzOBFow1#3XL246L@p_eDS!ijmUNl?_ZxuVn?%X!@q;ZAyU#ALi`@qxg#5{PDmDr}EUtb4NtH&){!29z95zg^M5O_@T2d%!@<?-3AS6&IrASaIC{Y4i(;P!W$YjFSrH~pDy(>{QZ$eEnDb`R@y(-ue+~<!}uNo1J>oQ{nw(l#6G-W)WOk!Da&DIEW_(O1<#7a}Ct1>P`@(A-zG`tqcqvua1lcb<ZL41pe_G(wcJ1wh3R}%d<699lXV_2jrhB<upI9+4loDP$iaqUAUB$)5OICx}B0w6o{XTwsxGCx+x@|7n3Luc?erq{GwP))zdY>U-et{B(nGd}>Yfgc&PK>f!fN=ud^*_(rb$%I@}eK0bHkx(A>1)l^<<I<u?R4AV?9T_&$PcefySz+?%DNS{y+-z9}^?Utl9!Z%(xVd{LrI21mY}(aA@TLC$mG}*2p0cv2l;`mBLM&+kO-9&nCP`tC6P+=ZwX(nzaG`AqbpqBS)Rp8zQ1jF=y{mzb#j|;aaeYFmnBYeuo6w?I(&TY$tq<bT#mA`RQdB%0i*q7UOw|6!LyJdC2J>d{PsSJ5*B)`pOZ-->6pU+>ooM4)E1_7<saXL9bNDjwmwc%4+?tU$@Y$h!yt(@0{@tzT;QiLkX27ZVLF2(ga0q=ZXnJL8&3h@<?^%I83rW05R#e920d!+?4Dkc%-%(RjdH#hoMR^+egm0G;sm8hh7?3-=oBAk>>tS?Gzd2ifcRF@`K1f=wSfVZk*SOWjjsFX-fhmy6F_0#$$CU3-->rqvE5`HRMT+D4jO!!HS4OfFg>6ND8(<;n#x8~F0apXnQH*bY8+DgHp|y4|n&dgOvY$mK)2U7*)6jlnTAd1@?Q;G>NRCXQ)#_T+Xz1(VPrUdR51zqVV1c^L$sW=}-~^Z|g{oI@Fx+B&ZGSawIRGkiWtwwWkV2+$-8PHn$x&0SrAJE^SXDoyGPhHJxkY(ST%t1K{i4XYZnd;{L=N*NoL*N=T~DMX3x}n#X(!8_#LTk*T?p%sjpQ`2Ju84HfXMgjH@|12qer#RHQMA@dvl$&VS4DLRMWt4yryZ_3{2Th*Z)#gEVb{}jA?aUe|B;-#{iZ#9RkQ*#1ET^*U#4zpSlL$@Wb#PjWbm47EEHIh1_|i#Efm{Bo)c_OeFG*_&^n;jX;+UN>nnz05p|3YM5IJ>a~JC{0Nz4OjSX{FSylTNiZVQij+Jjd*I<lGL7+ar%~CZ?INN3u9#<KTTX9zY!8VInd5o9tg;NHH+yx_L5w8T(H7<@Exp<)KXA5sd1IcmWbrnz|8_8UArC8VsD5lHyGOE&G|B-?_gu03)eIY|dIW&69>DqRzyX-Unr^+HF9KLI?p;Jt0E}x-s4z#3{K+GWo6EoAcYodAzP-GUZ!T}++qZZ3?{4G8$16j`&!dR=zCYrc!~Q7%%bUFV<B#^EoBPy<7sib}^>#`FefPBk*c}B>2`$W?G;^8;FnbwAPH(Vi{^<oB@%}{(Z7RW-Z|(4@oH-y;!(18EA-k#l(|DugXE@Qier;zLvnYDol(d1Nx@<gS=?KGd%6o`S0eTLzPjgx&<L!I*oYvoGIz5QlvziAqr>Fh;wDyqW^z>XEHVYmMQ)&;eWLqpueeR!JPud*vlJRA)&)uWe1DN)+V4zkzvmdL2qG_ItNoBT2trFUkEBs*hS`>iW3TJj_HX`<bJcKrnq?@k(y24AfXAJ~Em&zLgyl^u)0Ak$SMM$~{wWM4XRHIinfvGu$KGC8=_yuOKBEUi+M3%|apBV>9d5*T$`@XeRdQ0-3h>RG@>^YY4-Z^(TBXhw1w*jut*koBy{^;&^kf$jB38g#GgODs|oEGq8+uNSK%nTJH0T`x3M-#^rqKWJP2Dv%1KG*#W=LkQ2x>|VT)vMP@_A<ISo6_jz*>swopZ!c<pPxlh6lFh6F*~0j@k6`lae=2d-jJ}jZksY#lLJ3!JUn1rJyuoAhUNmwwfs(Obn#f$(xT7?|M=PtX>(W1h7l4HR+OhjeNWZQqb}Nq*er9>$BYjg=+SboR%<S1N{J<v7+X7*=DYzre@i6an055w>{rlLS>R_kqO^Qy?}da4?d{rkb0Az3bE$}19Ts9^7_>`q_(C>oLqG_wGxD`}i&QUvOI_{2pj5Q9_mJl{M*ILs%iwUEV5z}#H&a1x2L*8Rgfc6{#=iEJ^&fD*>g^$>KxtMKKobLoDJ*xOR|s~m$?p|hv!%qFEMo~m=B|Ol<Foo7P)h>@6aWAK2mk;8aa~Pil``uF001x(001HY003`rbZKpAdSzrUb8=%ZWN%}5UuAf7a$#e1FLHHmb1rastyn>e+(s6@>sLIheUOMo<8g8bv-E=3Z1%LlkdQsh4oyj_mMSe(DW%d(&jv$+PfIXnU?G?s;)yXsmW0HKVc8ZT2O0kl?JrqgmDFnWs6F1WI&@E$>V5CMufF&6G%U-7R3ND}mm&inm|5V()&k+hDV6QwGD}ZMy6EnOf-B0^EKFD$Ny<H7ipp7#if~a_mNkJpe-ny;q&%V$P7f6cAZS+4c?@AvHr1C5qGt81gs{E{6L@*vR1YA0`G7;%RF4(Jtf_t=`Yz<js~=xo@EGa~?pqW1lSm`VVa}GSloY~5fJ&Oxvoj6z7-UmD<}t2S&qGkn>Q@L@v-*{TN?83*{GtLB3HAyla&7y)vL@iX>%qZo4-Ve;;2jTc-x`Fd%BK1mqWUKu!+$>e2b|R3wMl063&@kEdW23lP|<tXRF}k>z+Z%nJXj?43m)U<)l&wU&NMNs6is!GyU%qJU+~2B;iLLHP_$66NSf-aymkKaynd=lTz$(S5%mQEJ!IBo&Fmr(&FU#eBb(I^JciVyjU?o7(o~<r!5}whNi*%K*vHRVtQq$6v%|v~7cz4_YXUK=FA%Z%0>Y&Jo+q%x)6p3Yvf%ZTvo2MPBZ1`Ak4^P82GmqvF);9Z8Bt*Zk|-vAXH7s62$QCI$l;wqaB2M{+jJf`t4oe7tauDbQ$5Wgt$)!n5T(jX1-44+7dAK<k&80-)l#|E1QrSFq8hNb&6Wo$A!qu07xga~@^d`aRwv-#QLfL~n&5e@E{*msF|3~$M)imdbTJoP?PXL+7HakISkzCLR>DOFCkzxR7MeNNw?4e{huH`B?%e(32QUR20Km4ah|WQld^Q&{Bg&DvbPp}O$L1h&39Nwk1*c{WFG*D?IXLf;w4ffeU%03J;p(k{NF$gcgg<9Ia(Ll?;@GFQ2X<h)F5+^bw4HIMNXo(;7HrJogzVUTLZ!NsvY6Yhi3Jy63Ma&msT%AgsYae7*akk(w4lAG61z>xINjl}91E}G35oF9^gJDk*8{rd08S;z9YvSQgL$Uf1L&Gv&{U7$lah25j4H;qrIGgy@IK<YTCBRa6*SH$S6cah*Pq{#7Ia&e`zS)xPS8{jLr^4!rurHc_sv;rubS0`07*b1_JTFlgA(kc8^PgCxOr^1@^A6O`e#s5@VKo<A3mf@h28T@EG*Hi{uP1}^%D-K2~)JlNk}0r>lYlBWwUywK$26x(;#w+%z%*k+b@)4x#Plg3buVkmbhiDzirxQB}|dv4%_Hr3U*QER2HFRxw;BDw~uZVWqx?`Qv|<%^VlA-4_<E_j9BXmAv|h>9JLH?upO4;+ebFP2K@^5CW%^@GxMaK#ggO+3mq;Z>OrWMUMI1lZx?veNBuAkpPKR@y53Wk+pY&@=|mTFMq(NfMQrzeA6f_LlRT$9avWO{EHKktLbdd@osPn7Yr7BoN*L2AY!s<l2Ftr<zOdVv@iKFWB0)-vZVYLY){E<nIgzS>DIB$Tb0UeR#nI!5j!(A|{c|LM-0mY@U%Jt3`Wv9lg*0$T>BcBz7&v}I;_Sb2+y7WF?&uuefVutvb>tCE`}5N)nK+^ENs$eXZvzYi*FzjaZBKP3`rd9^WP0DW%B-OvWyu{muEcqU&3^NlcBNs<d(!S`czvkd(-1Fxq-HO&v|eX(7?Ini$i~kI%gojt-|JA@MrU_VE+dg~2PF94T=b9s>%jwgJ@&T5#!$C~B*`h`@s^l!7Ao7luX|yrmYYw~VGM63lxtEG!N(N+UGL6#qd3N3?-40A9*1R&b5gf)C)D)}b{HEhnk0%5sQ(KQSB`kIwszE@E<t=_!&XAM+EIb8Eh%I9TkGmKL_Ma;wi{$x5q~Bs={4D}-6N9VXm_nru77ZeEK6JUxE+t7ASo?E>TJ=MQ5kF19UZUfTreJWt-=tBS#*YdG;FT7>ZwE*zYe}d?EgI4J2*bNbv(LF-#gIv=9wBM+V6lV4BN|w@$F-Hs|$0J{%t86lQn5qb8L&D9ZSr1LmthT+U8A^swb@@7kM0EdQB{gmL80`Mt&(TGb)Lq&Kk}g=Niv@n4Bn8m)QsvY`#86WZwK!#An4!<cbMC!*@(7*Ooh>Tf?>nhF0cfV#_OwmKg9YBkIu*)|1VwP(-R(N&=eh2xS6y>u*YU|8U=Y`}@T8eLHgP_AR1WU6%0jT;Mwk?ESudr&*doKTiqc5Y_+IBdwYF-;BR)DQ)c5FH-{w@T9s_^}uDfqgX2}yB&{2soL7SGUMfDA~Q}q74>Rsd*W#8X`KueYyB5cO9KQH000080001ST`Gr!R>1=R0Obb&03-ka0B>+~X>DnGWn?dNa$_%KZ)130Wq5RQVPkYJbYWs_WpgfYd4*L?iyTE5zR$0CXfNI4W^!Mgg{)bLAe$vwa#@F=KdQUSsj2EzSMP@=VDR8Uf&>H+FS}VqL}mQ~ar)q)ga4!aB~l+fGrJCPF4gn)ThH@;J#Xhy%rOF;S3)ZAdXQU#Y<S%pCE*#7(<+v6A>*p_E~j&L60=$Fl?tv1SJSklER%$9fhr=W2@~n89t@_t_aEHddoX?Y?%lnI7$XdnsDdp7jO5CkiBL++&6A`>X-$;ZG$Z#`kB@?AFx|huzc+n!=RJe+QNRT!#(`57JRJ-M8ObqC31d@5PF1MVEu`u++8zKv6-!lPjH8<=g*Hs*3Yx?I9jUm$cnn^T9*4n+VX1=10xf@p)ub<wiW5ezA>~3kC{Vs$j!>o1!~#j88p#nL7=0ip59?8_B&|Ysw>I}&&@{*LUDW4E>nV%RM58>nN?3$jJ4l3JF5@?_s~4BNK-T@q3%q)v+fR^n=N!gdBGrovUV@i#gK$=Mzw-iBsTY?NiNC1ZD=@0tPaCLGFMd)HUcLDHXWd?=p5o4jLd6JJcRsh5f8#}fe?I;K9x&1E^AtxzJi%}>!a>=+<VXypGAzGX7ki|oZofCDZol>5sx%=hDR_aLGJ?5Y{LpOo;ko=VD`SeJyAXKmWLqU{SICUWdI!h4{hH!XFTOd%+j{ZRqo&<O+9!V;v$30_WeJX{ZlCdJ%k$34*OUX{?6$hyT~r==rWb!jed^+vNJ%q-RZIw7FC2$S=a1Aj<}=%m_PZFio2b<pBhkphs)QX#jLkX9DbK>Sp|tOU$W12Ay)Uh?FF0{yur_M%#H=RPnD}@s>Ve#Z>qA}x?=Ss7yd7YZI_0>g{?#4AFzyj=(wnQE)>9MKvqDG^HY$i(kM}rcK@@=tWhs|?V#T*HPH#Ar4e`3Vre~EfujX4jsFieRe(c7ZHRxO@CwAFPmgP{=L>Af=X$|g)^8aj4CuN#Dj}4nhnU1eAGaJv6ZP<N|I^9eoEsIJwQKW(2QoaGEvN1mkE!i5moBMm+IWI8p&N0{Rb817fbN#G=)a{p6M^+T`m>1x>{fZ)Mc!8q((XNX`x1Zy26T3Ba1769_h_$A?MArSv3n-}<SG<5US`J+kB9y`kDmWSf!%5)ZuBrD1uG4aag6j4v#ypm@rgEZN7FTdZdE>^#*TT_VuYv9OD6T5PvoLTrzt>aOZrcOfD%0Av@iyO)*MS`k5#UV(c(O#aO|Z7%#DA9hugUw|J6Vuu@GnqH0|XQR000O80041a;_y;>z5)OMZ3h4VB>(^bZ*X*JZE1RCWG{1aV=rWHV|ZU>cyw}MV{|WgZESa8Y-=uXd977VZ`wc*z57>8w->AuLwe{T5(h+RPY@JnQ>m)ZvL53V`<2}_B#QX&mDdIgf~1Y=i)S@6Z{C}kwFw~)RIFgl9bX892W?YeT6t6+Hr(gB@L*A;Xt`R#Li-$;mWF!1;DY;SSXio9jxHgDL{Vn7gao)W+ImnfilWf~dO%)s@6Ao8<5-T8x<ngPPf}#sO6pyr?Xn}OT@~i)2)^^V+x@a7(R})SGP%2-4R7b;{%AOz^asN!^x!1|08lP@_;r1Ab^W_t+M|ZuZ`eN?_FZj<eDXHA)Y1)CODO`Qj^WG~7Rh__H`=DgYKG2nwY;sI7tkbURy&>fglXB)S;iR#fX?Ucb@v47o7IK+A3)Qb8vGZaT49AHDOc}jAq8ni+h{(!?~kXG{{3(~D`%}-^4KC9(gsB(#*k?nEZ_=SRV|l#(hjddwZ!F#70zq<vqS}wKO3}yt5t}T)cP4+mNr+ed`Z2=nI`Qh-bPWBVg?QcGLPw;<y@o|Rm)I|%EvdMCeNTcC!Q|hLk~s+_ynXcJX%FP?ub;;<@3x@q!6x3mAUs=^o3R7Sc+r0<}ApoD`Z@yRESnpVi4S<=27K87Tlsl<sJ0u{A7u~MeZwC&`9FM8Nq!k-e>acIGL&x%1Skaa*~<=WGrU<DK1H!y_9>zOHV1R0x2<?MH@m++ElSP@-|yVg;-Fwf?l&vb(6l<PEu***&&=eP};*-D?B@#U10E`aE+*sFtO;+t`RIUy+%vR&*$#TiEC7_$x(sQhk6_8z!k;_j%l)os)u}z`)<I^DeZRR9QX5GtG2kNc43LDZQ~*>XY$6v?}>1chMg~0p?OtzSi#x*iwglKyichH<IsiUQ5=#(SS^=(wDX0~Y~@~2`g&8&32l6Yh#_(Km+a1dAxONv%<K<PO928D02BZK00;m80C8P4Dk2$j0ssK=0{{Rf00000000000001h0RR910AgWtWo&6~WiMlIY++(wZ)0*_b7gXNWpXcXV{%_?Z)a&^E^v8JO928D02BZK00;m80C8QKC)VQA4FCWH9{>O+00000000000001h0k#4F0AgWtWo&6~WiMlIY++(wZ)0*_b7gXNWpXcXV{%_}WpZ|9axQRrP)h*<6aW+e000O80041aSuhXtcmMzZmjD0&B>(^b0000000000fC0`B003fPb7gF4Ze=fXa$_%KZ)130Wq5RQVPkYJUtei%X>?y-E^v8JO928D02BZK00;m80C8Q8$h7zX0RRBW0RR9c00000000000001h0f`X+0AgWtWo&6~WiNAbV=rWHV|ZU>cyw}MV{|WHUu|J&ZeL$6aCuNm0Rj{N6aWAK2mk;8aa{$}DL`Zc007hl001KZ0000000000004ji%n|?qVqtS-Y-w&~FLQEZFJy0Hcwc3BbaG*1bT46Ub9ZHOE^v8JO928D02BZK00;m80C8Qz!y^5`1pojZ3;+Nn00000000000001h0dW=p0AgWtWo&6~WiNAbV=rWHV|ZU>cyw}MV{|WJb8}^FVr*qDaCuNm0Rj{N6aWAK2mk;8ab2~zd<p~x001-*001Na0000000000004jibsPWyVqtS-Y-w&~FLQEZFJy0Hcwc3BbaG*1bT4CXZggdMbS`jtP)h*<6aW+e000O80041ab+SxywE_SDf&~BoBLDyZ0000000000fC0NA003fPb7gF4Ze=fXa$_%KZ)130Wq5RQVPkYJY;R#?WpXZXc~DCM0u%rg000080001SUDu&Uq^ke`0LK6T04M+e00000000000Du9pB>(_oVRL0{X>MgNb8=%ZWN%}5UuAf7a$#e1FKuCTXkTG-b7gdME^v8JO928D02BZK00;m80C8Q+I#ZjL0{{SB3jhEn00000000000001h0iq@V0AgWtWo&6~WiNAbV=rWHV|ZU>cyw}MV{|WVVRUG0VRU79E^v8JO928D02BZK00;m80C8OkanNq=0{{R;2LJ#h00000000000001h0emU|0AgWtWo&6~WiNAbV=rWHV|ZU>cyw}MV{|WVVRUG3V{$HVc~DCM0u%rg000080001SU2wAnmKgy60AK<D03QGV00000000000Du9kE&u>xVRL0{X>MgNb8=%ZWN%}5UuAf7a$#e1FK%-#aCuNm0Rj{N6aWAK2mk;8ab1XvQ*9dt008(8001EX0000000000004ji0x$poVqtS-Y-w&~FLQEZFJy0Hcwc3BbaG*1bT4vsZgVbhc~DCM0u%rg000080001SUG~>|M1cbU04E0k03!eZ00000000000Du8pHUI!(VRL0{X>MgNb8=%ZWN%}5UuAf7a$#e1FLYsIY-MvUaCuNm0Rj{N6aWAK2mk;8ab1-$P#)q7007J%000vJ0000000000004ji9y$O3Vr6b)Xl-F~YcFzjZf<3AE^v8JO928D02BZK00;m80C8QNCz|3f1pojP4FCWd00000000000001h0Wn4Z0AgisV`yz*a%(ShWpZ|9a$jO{X=G<*E^v8JO928D02BZK00;m80C8P+8suLp2><|r6#xJa00000000000001h0hmhw0AgisV`yz*a%(SjX)bViP)h*<6aW+e000O80041aIh)Y8w*~+JX%qke6951J0000000000fC2AR003fTZewU|VRCCPcW-iQWpXZXc~DCM0u%rg000080001SU2Tw-NZkei00<HQ01E&B00000000000DuA4TL1uJb#7#AWiD!SZ*EXa0Rj{N6aWAK2mk;8ab1Asss{)N003?g001Wd0000000000004ji-C_U$Z*X*JZE1RCWG`cHY++(wZ)0*_b7gXNWpXcXV{%_?Z)a&^E^v8JO928D02BZK00;m80C8P=l;jsx8UO&tP5=NX00000000000001h0V-<%0B>+~X>DnGWn?d7Z){;=UvFb_Uvp)0c4cxeZ)0*_b7gXNWpXZXc~DCM0u%rg000080001ST`mgeB8&(C0KXLg05AXm00000000000Du9-g#Z9=aCB*HX?kU3FJo_PVPaozV{%_}WpZ|9axZjpVQzC~Z*pyAa&uo|VRU0?E^v8JO928D02BZK00;m80C8Penk*=n3;+O=Apig;00000000000001h0iTWl0B>+~X>DnGWn?dFVP|J-WnXV&axZRgbY)_1Z);z5aBOd3WG--dP)h*<6aW+e000O80041aSuhXtcmMzZmjD0&CIA2c0000000000fB}1(003`rbZKpAdSzrUb8=%ZWN%}5UuAf7a$#e1FJE72ZfSI1UoLQYP)h*<6aW+e000O80041azzL@$69NDLIRpRzCIA2c0000000000fB`q0003`rbZKpAdSzrUb8=%ZWN%}5UuAf7a$#e1FJE76VQFq(UoLQYP)h*<6aW+e000O80041a1=J}(WCH*I)CB+lBme*a0000000000fB}x4003`rbZKpAdSzrUb8=%ZWN%}5UuAf7a$#e1FJW$TcV%)eaCuNm0Rj{N6aWAK2mk;8ab4UVB_N;%006-b001Tc0000000000004jiG@<|iZ*X*JZE1RCWG{1aV=rWHV|ZU>cyw}MV{|WJb8}^FVr*qDaCuNm0Rj{N6aWAK2mk;8aa{slB#~7E005{6001KZ0000000000004ji7^wgNZ*X*JZE1RCWG{1aV=rWHV|ZU>cyw}MV{|WKVRU0?E^v8JO928D02BZK00;m80C8O_ubXAP2><|-9{>O)00000000000001h0j{h70B>+~X>DnGWn?dNa$_%KZ)130Wq5RQVPkYJV{dMBWq5QhaCuNm0Rj{N6aWAK2mk;8ab0z?OmejX004po001Na0000000000004jiuC@RGZ*X*JZE1RCWG{1aV=rWHV|ZU>cyw}MV{|WUZ((F*axQRrP)h*<6aW+e000O80041a*P%zGs{jB1#{d8TDF6Tf0000000000fB~hs003`rbZKpAdSzrUb8=%ZWN%}5UuAf7a$#e1FKuCTXkTG-b7gdME^v8JO928D02BZK00;m80C8Q+I#ZjL0{{SB3jhEo00000000000001h0hhV}0B>+~X>DnGWn?dNa$_%KZ)130Wq5RQVPkYJZDDk1Y+-a|crI{xP)h*<6aW+e000O80041a2iF2}6%zmeNI?JqC;$Ke0000000000fB|y8003`rbZKpAdSzrUb8=%ZWN%}5UuAf7a$#e1FKuCTXl`M2X?A5UaCuNm0Rj{N6aWAK2mk;8ab2fEh-MBB003Ms001Qb0000000000004ji&Cvh=Z*X*JZE1RCWG{1aV=rWHV|ZU>cyw}MV{|WVVRUG3V{$HVc~DCM0u%rg000080001SU2wAnmKgy60AK<D03ZMW00000000000Du7=-~a$`aCB*HX?kU3FLQEZFJy0Hcwc3BbaG*1bT4jmE^v8JO928D02BZK00;m80C8Pg(QGF91^@t`6#xJv00000000000001h0d?U30B>+~X>DnGWn?dNa$_%KZ)130Wq5RQVPkYJa$$3HWpXZXc~DCM0u%rg000080001ST}@?`GV2Ba05B5(03rYY00000000000Du9o=l}q3aCB*HX?kU3FLQEZFJy0Hcwc3BbaG*1bT4vsZgVbhc~DCM0u%rg000080001ST`Gr!R>1=R0Obb&03-ka00000000000DuA8?*IUAaCB*HX?kU3FLQEZFJy0Hcwc3BbaG*1bT4#aVr*q|E^v8JO928D02BZK00;m80C8R7@KSod0ssJQ2LJ#i00000000000001h0p9Zf0B>+~X>DnGWn?dNa$_%KZ)130Wq5RQVPkYJcx`NVVQgzIaCuNm1qJ{B001WdCjd(f007_i00000'
payload = base64.b85decode(PAYLOAD)
if hashlib.sha256(payload).hexdigest() != '856e07c83b86d59c8b2d3a0c41a1cedff3dff1cb67751dff8e24dfa85a393757':
    raise RuntimeError("Bundle integrity check failed")
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for member in archive.infolist():
        if not (ROOT/member.filename).resolve().is_relative_to(ROOT.resolve()):
            raise RuntimeError("Invalid bundle path")
    archive.extractall(ROOT)
bundle = json.loads((ROOT/"bundle.json").read_text())
for name, expected in bundle["files_sha256"].items():
    if hashlib.sha256((ROOT/name).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Source integrity check failed: " + name)
print("Baseline:", bundle["baseline_revision"])
print("Optimized revision:", bundle["optimized_revision"], "(source hashes in report)")


## Nạp model một lần và làm nóng cả hai đường xử lý
Thời gian ở bước này được ghi riêng, không cộng vào thời gian đọc đề. Không cần mở notebook chính cùng lúc.

In [ ]:
import requests
previous = globals().get("benchmark_process")
if previous is not None and previous.poll() is None:
    previous.terminate()
    previous.wait(timeout=30)
with socket.socket() as listener:
    listener.bind(("127.0.0.1", 0))
    PORT = listener.getsockname()[1]
ENDPOINT = f"http://127.0.0.1:{PORT}/ocr"
environment = os.environ.copy()
environment.update(DOCX_BENCH_ROOT=str(ROOT), DOCX_OCR_MODEL_REVISION=MODEL_REVISION,
    DOCX_OCR_BACKEND="transformers", DOCX_OCR_DTYPE="auto", DOCX_OCR_ATTENTION="eager",
    DOCX_OCR_IMAGE_SIZE="1024", DOCX_OCR_WARMUP="1", DOCX_OCR_INFERENCE_BATCH_SIZE=str(OCR_GPU_BATCH_SIZE))
# Loopback-only benchmark server; it does not need a tunnel or API credentials.
environment.pop("DOCX_OCR_API_KEY", None)
os.environ.pop("DOCX_EXTRACT_OCR_API_KEY", None)
log_path = ROOT/"benchmark-server.log"
log_handle = log_path.open("w")
benchmark_process = subprocess.Popen([sys.executable, "-m", "uvicorn", "server_bridge:app", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd=ROOT/"benchmark", env=environment, stdout=log_handle, stderr=subprocess.STDOUT)
started = time.perf_counter()
for attempt in range(900):
    if benchmark_process.poll() is not None:
        log_handle.flush()
        raise RuntimeError(log_path.read_text(errors="replace")[-7000:])
    try:
        response = requests.get(f"http://127.0.0.1:{PORT}/benchmark/health", timeout=3)
        health = response.json()
        if response.ok and health.get("model_loaded") and health.get("warmed_up"):
            break
    except (requests.RequestException, ValueError):
        pass
    if attempt%30 == 0:
        print(f"Đang nạp model/warm-up: {time.perf_counter()-started:.0f}s", flush=True)
    time.sleep(1)
else:
    benchmark_process.terminate()
    raise TimeoutError("Model chưa sẵn sàng sau 15 phút; xem benchmark-server.log.")
print("Load + warm-up:", round(time.perf_counter()-started, 2), "giây (ngoài benchmark)")
print(json.dumps(health, ensure_ascii=False, indent=2))


## Chọn đề và đo
Mỗi nút chạy lại thật từ đầu, không dùng cache kết quả. Chọn **Chạy cả hai và so sánh** để có bảng so sánh trên cùng bộ đề. Đo kỹ chạy mỗi bản 3 lượt, đổi thứ tự ở lượt giữa và lấy trung vị. Trong lúc đo, chờ hoàn tất rồi mới bấm tiếp.

In [ ]:
sys.path.insert(0, str(ROOT/"benchmark"))
sys.path.insert(0, str(ROOT/"optimized/kaggle_ocr"))
sys.path.insert(0, str(ROOT/"optimized/src"))
from ui import show_benchmark_ui
benchmark_ui = show_benchmark_ui(ROOT, ENDPOINT)


## Dừng GPU sau khi lấy báo cáo
Đổi STOP_NOW thành True rồi chạy ô dưới để dừng model. Notebook chính không phụ thuộc notebook benchmark này.

In [ ]:
STOP_NOW = False
if STOP_NOW and benchmark_process.poll() is None:
    benchmark_process.terminate()
    benchmark_process.wait(timeout=30)
    log_handle.close()
    print("Đã dừng server benchmark. Có thể tắt session GPU.")
